In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:

import pandas as pd
import numpy as np
import os

file_path = "/content/drive/MyDrive/yield prediction /crop_production_124_with_weather_soil.csv"

if not os.path.exists(file_path):
    raise FileNotFoundError(
        f"File not found: {file_path}\n"
        "Please check the file's location in Google Drive."
    )

df = pd.read_csv(file_path)

print("Dataset loaded successfully!")
print("Shape:", df.shape)
display(df.head())


Dataset loaded successfully!
Shape: (246091, 11)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,temperature,humidity,rainfall_mm,soil_type
0,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Arecanut,1254.0,2000.0,27.618000,84.0040,1626.0,Laterite
1,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Other Kharif pulses,2.0,1.0,27.618000,84.0040,1626.0,Alluvial
2,Andaman and Nicobar Islands,NICOBARS,2000,Kharif,Rice,102.0,321.0,27.618000,84.0040,1626.0,Alluvial
3,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Banana,176.0,641.0,27.493333,81.7475,2520.3,Alluvial
4,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Cashewnut,720.0,165.0,27.493333,81.7475,2520.3,Laterite


In [ ]:

missing_report = pd.DataFrame({
    "Missing_Count": df.isnull().sum(),
    "Missing_Percentage": (
        df.isnull().sum() / len(df) * 100
    ).round(2)
})

display(missing_report)


,Missing_Count,Missing_Percentage
State_Name,0,0.00
District_Name,0,0.00
Crop_Year,0,0.00
Season,0,0.00
Crop,0,0.00
Area,0,0.00
Production,3730,1.52
temperature,0,0.00
humidity,0,0.00
rainfall_mm,0,0.00


In [ ]:

key_columns = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Crop"
]

print("Fully duplicated rows:", df.duplicated().sum())

print(
    "Duplicate crop-location-year-season keys:",
    df.duplicated(subset=key_columns).sum()
)


Fully duplicated rows: 0
Duplicate crop-location-year-season keys: 0


In [ ]:

print("Unique crops:", df["Crop"].nunique())
print("Unique states:", df["State_Name"].nunique())
print("Unique districts:", df["District_Name"].nunique())
print("Unique seasons:", df["Season"].nunique())

print("\nYear range:")
print("Minimum:", df["Crop_Year"].min())
print("Maximum:", df["Crop_Year"].max())

print("\nRecords per year:")
display(df["Crop_Year"].value_counts().sort_index())

print("\nRecords per crop:")
display(df["Crop"].value_counts().to_frame("Record_Count"))


Unique crops: 124
Unique states: 33
Unique districts: 646
Unique seasons: 6

Year range:
Minimum: 1997
Maximum: 2015

Records per year:


,count
Crop_Year,
1997,8899
1998,11533
1999,12515
2000,13658
2001,13361
2002,16671
2003,17287
2004,14117
2005,13799



Records per crop:


,Record_Count
Crop,
Rice,15104
Maize,13947
Moong(Green Gram),10318
Urad,9850
Sesamum,9046
...,...
Pear,6
Litchi,6
Peach,4


In [ ]:

numeric_columns = [
    "Area",
    "Production",
    "temperature",
    "humidity",
    "rainfall_mm"
]

display(df[numeric_columns].describe().T)

print("Area <= 0:", (df["Area"] <= 0).sum())
print("Missing production:", df["Production"].isna().sum())
print("Zero production:", (df["Production"] == 0).sum())
print("Negative production:", (df["Production"] < 0).sum())

print("\nSoil distribution:")
display(df["soil_type"].value_counts(dropna=False))


,count,mean,std,min,25%,50%,75%,max
Area,246091.0,12002.820864,5.052340e+04,0.0400,80.000,582.0000,4392.000,8.580100e+06
Production,242361.0,582503.442251,1.706581e+07,0.0000,88.000,729.0000,7023.000,1.250800e+09
temperature,246091.0,24.711056,4.682965e+00,-12.5340,22.096,25.6500,27.750,3.522200e+01
humidity,246091.0,62.581318,1.604321e+01,17.9925,50.600,65.6575,75.778,9.272800e+01
rainfall_mm,246091.0,680.196746,5.982700e+02,0.0000,156.000,591.3000,1009.300,4.787100e+03


Area <= 0: 0
Missing production: 3730
Zero production: 3523
Negative production: 0

Soil distribution:


,count
soil_type,
Alluvial,131128
Black,50346
Red,39319
Mountain (Forest),9999
Laterite,8499
Red and Yellow,4742
Arid (Desert),2042
Saline-Alkaline,16


In [ ]:

# STEP 2: YIELD TARGET VALIDATION
# Does not modify the original df.

yield_audit = df.copy()

valid_mask = (
    yield_audit["Area"].notna()
    & yield_audit["Production"].notna()
    & (yield_audit["Area"] > 0)
    & (yield_audit["Production"] >= 0)
)

yield_audit["calculated_yield"] = np.nan

yield_audit.loc[valid_mask, "calculated_yield"] = (
    yield_audit.loc[valid_mask, "Production"]
    / yield_audit.loc[valid_mask, "Area"]
)

print("Total rows:", len(yield_audit))
print("Rows with calculable yield:", valid_mask.sum())
print("Rows without calculable yield:", (~valid_mask).sum())

print("\nYield ratio summary:")
display(
    yield_audit["calculated_yield"].describe(
        percentiles=[0.01, 0.05, 0.50, 0.95, 0.99]
    ).to_frame("Calculated_Yield_Ratio")
)

print("\nHighest 20 yield ratios:")
display(
    yield_audit.nlargest(20, "calculated_yield")[
        [
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Crop", "Area", "Production",
            "calculated_yield"
        ]
    ]
)


Total rows: 246091
Rows with calculable yield: 242361
Rows without calculable yield: 3730

Yield ratio summary:


,Calculated_Yield_Ratio
count,242361.000000
mean,41.649059
std,817.572839
min,0.000000
1%,0.000000
5%,0.185049
50%,1.000000
95%,22.448276
99%,82.663712
max,88000.000000



Highest 20 yield ratios:


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,calculated_yield
163866,Punjab,PATIALA,2011,Whole Year,Sugarcane,3.0,264000.0,88000.000000
164316,Punjab,SANGRUR,2011,Whole Year,Sugarcane,2.0,163000.0,81500.000000
163003,Punjab,LUDHIANA,2011,Whole Year,Sugarcane,2.0,161000.0,80500.000000
161948,Punjab,FATEHGARH SAHIB,2011,Whole Year,Sugarcane,2.0,151000.0,75500.000000
164038,Punjab,RUPNAGAR,2011,Whole Year,Sugarcane,3.0,224000.0,74666.666667
162328,Punjab,GURDASPUR,2011,Whole Year,Sugarcane,20.0,1491000.0,74550.000000
163678,Punjab,PATHANKOT,2011,Whole Year,Sugarcane,4.0,298000.0,74500.000000
162835,Punjab,KAPURTHALA,2011,Whole Year,Sugarcane,3.0,221000.0,73666.666667
161967,Punjab,FAZILKA,2011,Whole Year,Sugarcane,1.0,73000.0,73000.000000
163657,Punjab,NAWANSHAHR,2011,Whole Year,Sugarcane,6.0,407000.0,67833.333333


In [ ]:

import pandas as pd
import numpy as np

# Recreate the audit dataframe
valid_mask = (
    df["Area"].notna()
    & df["Production"].notna()
    & (df["Area"] > 0)
    & (df["Production"] >= 0)
)

audit = df.loc[valid_mask].copy()

audit["calculated_yield"] = (
    audit["Production"] / audit["Area"]
)

print("Audit dataframe created:", audit.shape)

# 1. Count extreme yield ratios
print("\nYIELD RATIO THRESHOLD COUNTS")

for threshold in [10, 50, 100, 1000, 10000]:
    count = (audit["calculated_yield"] > threshold).sum()
    print(f"Yield ratio > {threshold:>5}: {count:,}")

# 2. Yield summary by crop
print("\nCROPS WITH HIGHEST MAXIMUM YIELD RATIOS")

crop_yield_summary = (
    audit.groupby("Crop")["calculated_yield"]
    .agg(
        Records="count",
        Median="median",
        Mean="mean",
        Maximum="max"
    )
    .sort_values("Maximum", ascending=False)
)

display(crop_yield_summary.head(20))

# 3. Inspect extreme sugarcane records
print("\nSUGARCANE RECORDS WITH YIELD RATIO > 100")

sugarcane = audit[
    audit["Crop"].str.lower().eq("sugarcane")
    & (audit["calculated_yield"] > 100)
]

display(
    sugarcane[
        [
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "Production", "calculated_yield"
        ]
    ].sort_values(
        "calculated_yield", ascending=False
    ).head(30)
)

# 4. Inspect area and production scales
print("\nAREA AND PRODUCTION QUANTILES")

display(
    audit[["Area", "Production"]]
    .quantile([0, 0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99, 1])
)


Audit dataframe created: (242361, 12)

YIELD RATIO THRESHOLD COUNTS
Yield ratio >    10: 23,738
Yield ratio >    50: 5,766
Yield ratio >   100: 1,628
Yield ratio >  1000: 969
Yield ratio > 10000: 287

CROPS WITH HIGHEST MAXIMUM YIELD RATIOS


,Records,Median,Mean,Maximum
Crop,,,,
Sugarcane,7827,53.241379,195.003397,88000.000000
Coconut,1958,43.120000,4040.403568,38800.000000
Cashewnut,1068,0.396104,9.675758,9801.000000
Onion,6984,10.297120,11.855243,2150.000000
Maize,13787,1.795918,2.632770,1494.000000
Cotton(lint),4382,1.367349,2.063290,300.000000
Sesamum,8821,0.333333,0.459043,235.000000
Rice,15082,1.938529,1.995014,223.727273
Potato,6914,10.833194,12.867025,213.846154



SUGARCANE RECORDS WITH YIELD RATIO > 100


,State_Name,District_Name,Crop_Year,Season,Area,Production,calculated_yield
163866,Punjab,PATIALA,2011,Whole Year,3.0,264000.0,88000.000000
164316,Punjab,SANGRUR,2011,Whole Year,2.0,163000.0,81500.000000
163003,Punjab,LUDHIANA,2011,Whole Year,2.0,161000.0,80500.000000
161948,Punjab,FATEHGARH SAHIB,2011,Whole Year,2.0,151000.0,75500.000000
164038,Punjab,RUPNAGAR,2011,Whole Year,3.0,224000.0,74666.666667
162328,Punjab,GURDASPUR,2011,Whole Year,20.0,1491000.0,74550.000000
163678,Punjab,PATHANKOT,2011,Whole Year,4.0,298000.0,74500.000000
162835,Punjab,KAPURTHALA,2011,Whole Year,3.0,221000.0,73666.666667
161967,Punjab,FAZILKA,2011,Whole Year,1.0,73000.0,73000.000000
163657,Punjab,NAWANSHAHR,2011,Whole Year,6.0,407000.0,67833.333333



AREA AND PRODUCTION QUANTILES


,Area,Production
0.00,0.1,0.000000e+00
0.01,1.0,0.000000e+00
0.05,5.0,3.000000e+00
0.25,87.0,8.800000e+01
0.50,603.0,7.290000e+02
0.75,4545.0,7.023000e+03
0.95,68526.0,1.820000e+05
0.99,190247.8,9.825800e+05
1.00,8580100.0,1.250800e+09


In [ ]:

# STEP 2B: CROP-SPECIFIC AND YEAR-SPECIFIC DIAGNOSTICS

# 1. Yield distribution by crop
crop_diagnostics = (
    audit.groupby("Crop")["calculated_yield"]
    .agg(
        Records="count",
        Median_Yield="median",
        P95_Yield=lambda x: x.quantile(0.95),
        P99_Yield=lambda x: x.quantile(0.99),
        Maximum_Yield="max"
    )
    .round(3)
    .sort_values("Maximum_Yield", ascending=False)
)

print("Crops with the highest maximum ratios:")
display(crop_diagnostics.head(25))

# 2. Sugarcane yield distribution by year
sugarcane_by_year = (
    audit[audit["Crop"].str.lower() == "sugarcane"]
    .groupby("Crop_Year")["calculated_yield"]
    .agg(
        Records="count",
        Median="median",
        P95=lambda x: x.quantile(0.95),
        Maximum="max"
    )
    .round(3)
)

print("\nSugarcane ratios by year:")
display(sugarcane_by_year)

# 3. Inspect all Punjab sugarcane records from 2011
print("\nPunjab sugarcane records for 2011:")

display(
    audit[
        (audit["State_Name"].str.strip().str.lower() == "punjab")
        & (audit["Crop"].str.strip().str.lower() == "sugarcane")
        & (audit["Crop_Year"] == 2011)
    ][
        [
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "Production", "calculated_yield"
        ]
    ].sort_values("District_Name")
)

# 4. Inspect records with ratios above 100
print("\nExtreme ratios by crop:")
display(
    audit[audit["calculated_yield"] > 100]
    ["Crop"]
    .value_counts()
    .head(20)
    .to_frame("Records_Above_100")
)


Crops with the highest maximum ratios:


,Records,Median_Yield,P95_Yield,P99_Yield,Maximum_Yield
Crop,,,,,
Sugarcane,7827,53.241,107.086,146.145,88000.000
Coconut,1958,43.120,13817.544,19909.530,38800.000
Cashewnut,1068,0.396,1.030,1.894,9801.000
Onion,6984,10.297,25.740,33.000,2150.000
Maize,13787,1.796,4.817,7.230,1494.000
Cotton(lint),4382,1.367,3.999,5.786,300.000
Sesamum,8821,0.333,0.882,1.102,235.000
Rice,15082,1.939,3.638,4.518,223.727
Potato,6914,10.833,25.998,31.350,213.846



Sugarcane ratios by year:


,Records,Median,P95,Maximum
Crop_Year,,,,
1997,330,55.366,1073.573,2247.533
1998,408,51.851,121.505,303.485
1999,444,54.189,111.969,162.500
2000,466,49.522,107.116,162.000
2001,461,51.933,100.173,158.573
2002,466,49.890,92.380,162.000
2003,461,49.284,100.000,163.333
2004,423,51.275,100.691,162.500
2005,469,53.788,99.201,147.258



Punjab sugarcane records for 2011:


,State_Name,District_Name,Crop_Year,Season,Area,Production,calculated_yield
161430,Punjab,AMRITSAR,2011,Whole Year,3.0,203000.0,67666.666667
161948,Punjab,FATEHGARH SAHIB,2011,Whole Year,2.0,151000.0,75500.000000
161967,Punjab,FAZILKA,2011,Whole Year,1.0,73000.0,73000.000000
162328,Punjab,GURDASPUR,2011,Whole Year,20.0,1491000.0,74550.000000
162523,Punjab,HOSHIARPUR,2011,Whole Year,20.0,1293000.0,64650.000000
162701,Punjab,JALANDHAR,2011,Whole Year,9.0,599000.0,66555.555556
162835,Punjab,KAPURTHALA,2011,Whole Year,3.0,221000.0,73666.666667
163003,Punjab,LUDHIANA,2011,Whole Year,2.0,161000.0,80500.000000
163657,Punjab,NAWANSHAHR,2011,Whole Year,6.0,407000.0,67833.333333
163678,Punjab,PATHANKOT,2011,Whole Year,4.0,298000.0,74500.000000



Extreme ratios by crop:


,Records_Above_100
Crop,
Coconut,958
Sugarcane,607
Papaya,37
Cotton(lint),7
Banana,6
Maize,6
Onion,3
Potato,1
Sesamum,1


In [ ]:

# STEP 2C: INVESTIGATE COCONUT AND SUGARCANE

crops_to_check = ["Sugarcane", "Coconut"]

for crop_name in crops_to_check:
    crop_df = audit[
        audit["Crop"].str.strip().str.lower()
        == crop_name.lower()
    ].copy()

    print(f"\n{'=' * 65}")
    print(f"CROP: {crop_name.upper()}")
    print(f"{'=' * 65}")

    # Overall distribution
    print("\nArea distribution:")
    display(crop_df["Area"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ).to_frame())

    print("\nProduction distribution:")
    display(crop_df["Production"].describe(
        percentiles=[0.01, 0.05, 0.25, 0.5, 0.75, 0.95, 0.99]
    ).to_frame())

    # Annual statistics
    print("\nAnnual yield-ratio statistics:")
    annual = crop_df.groupby("Crop_Year").agg(
        Records=("calculated_yield", "size"),
        Median_Yield=("calculated_yield", "median"),
        P95_Yield=("calculated_yield",
                   lambda x: x.quantile(0.95)),
        Maximum_Yield=("calculated_yield", "max"),
        Median_Area=("Area", "median"),
        Median_Production=("Production", "median")
    ).round(3)

    display(annual)

    # Highest ratios for this crop
    print("\nTop 10 highest ratios:")
    display(
        crop_df.nlargest(10, "calculated_yield")[
            [
                "State_Name", "District_Name", "Crop_Year",
                "Season", "Area", "Production",
                "calculated_yield"
            ]
        ]
    )

# Compare Punjab sugarcane in 2010–2012
print("\nPUNJAB SUGARCANE: 2010–2012")

display(
    audit[
        audit["State_Name"].str.strip().str.lower().eq("punjab")
        & audit["Crop"].str.strip().str.lower().eq("sugarcane")
        & audit["Crop_Year"].between(2010, 2012)
    ][
        [
            "District_Name", "Crop_Year", "Area",
            "Production", "calculated_yield"
        ]
    ].sort_values(["Crop_Year", "District_Name"])
)



CROP: SUGARCANE

Area distribution:


,Area
count,7827.000000
mean,9793.819184
std,26703.309773
min,0.200000
1%,1.000000
5%,7.000000
25%,129.000000
50%,818.000000
75%,6126.000000
95%,53474.500000



Production distribution:


,Production
count,7.827000e+03
mean,7.072546e+05
std,2.045078e+06
min,0.000000e+00
1%,9.260000e+00
5%,1.043000e+02
25%,4.720500e+03
50%,3.742200e+04
75%,3.953635e+05
95%,3.874741e+06



Annual yield-ratio statistics:


...

,Records,Median_Yield,P95_Yield,Maximum_Yield,Median_Area,Median_Production
Crop_Year,,,,,,
1997,57,5.607,19.280,20.975,1040.0,8766.0
1998,83,17.328,7705.035,9908.819,1174.0,75720.0
1999,86,17.768,10404.054,16500.000,1161.5,180378.0
2000,108,3752.314,10739.733,18000.000,1172.5,493405.0
2001,107,3583.784,11055.351,26270.833,1264.0,530400.0
2002,134,15.443,11255.634,14432.862,2105.5,158800.0
2003,134,15.635,11824.384,16906.298,2075.5,119740.0
2004,138,48.245,11789.000,28334.156,2177.5,354800.0
2005,114,5.016,8749.046,16500.000,4480.0,19252.0



Top 10 highest ratios:


,State_Name,District_Name,Crop_Year,Season,Area,Production,calculated_yield
238912,West Bengal,BIRBHUM,2006,Whole Year,509.0,19749200.0,38800.000000
9829,Andhra Pradesh,WEST GODAVARI,2014,Whole Year,21729.0,718991000.0,33089.005477
20171,Assam,KAMRUP,2004,Whole Year,2026.0,57405000.0,28334.155972
22705,Assam,LAKHIMPUR,2001,Whole Year,192.0,5044000.0,26270.833333
185975,Tamil Nadu,THANJAVUR,2011,Whole Year,33742.0,823800000.0,24414.676071
18503,Assam,GOLAGHAT,2004,Whole Year,658.0,16062000.0,24410.334347
7697,Andhra Pradesh,SRIKAKULAM,2014,Whole Year,13811.0,328149000.0,23759.973934
236732,West Bengal,24 PARAGANAS NORTH,2006,Whole Year,3338.0,75095000.0,22497.004194
18565,Assam,GOLAGHAT,2006,Whole Year,612.0,13555000.0,22148.692810
18598,Assam,GOLAGHAT,2007,Whole Year,639.0,14029000.0,21954.616588



PUNJAB SUGARCANE: 2010–2012


,District_Name,Crop_Year,Area,Production,calculated_yield
161430,AMRITSAR,2011,3.0,203000.0,67666.666667
161948,FATEHGARH SAHIB,2011,2.0,151000.0,75500.000000
161967,FAZILKA,2011,1.0,73000.0,73000.000000
162328,GURDASPUR,2011,20.0,1491000.0,74550.000000
162523,HOSHIARPUR,2011,20.0,1293000.0,64650.000000
162701,JALANDHAR,2011,9.0,599000.0,66555.555556
162835,KAPURTHALA,2011,3.0,221000.0,73666.666667
163003,LUDHIANA,2011,2.0,161000.0,80500.000000
163657,NAWANSHAHR,2011,6.0,407000.0,67833.333333
163678,PATHANKOT,2011,4.0,298000.0,74500.000000


In [ ]:

# Flag suspicious records; do not delete them yet.

audit["yield_review_flag"] = False

# Coconut uses an incompatible raw production unit
audit.loc[
    audit["Crop"].str.strip().str.lower().eq("coconut"),
    "yield_review_flag"
] = True

# Flag extreme ratios for investigation, not automatic deletion
audit.loc[
    audit["calculated_yield"] > 200,
    "yield_review_flag"
] = True

# Specifically flag Punjab sugarcane records from 2011
audit.loc[
    audit["State_Name"].str.strip().str.lower().eq("punjab")
    & audit["Crop"].str.strip().str.lower().eq("sugarcane")
    & (audit["Crop_Year"] == 2011),
    "yield_review_flag"
] = True

print("Records flagged for review:",
      audit["yield_review_flag"].sum())

print("\nFlagged records by crop:")
display(
    audit.loc[audit["yield_review_flag"], "Crop"]
    .value_counts()
    .to_frame("Flagged_Records")
)

print("\nFlagged record examples:")
display(
    audit.loc[audit["yield_review_flag"]].head(20)[
        [
            "State_Name", "District_Name", "Crop_Year",
            "Crop", "Area", "Production", "calculated_yield"
        ]
    ]
)


Records flagged for review: 2023

Flagged records by crop:


,Flagged_Records
Crop,
Coconut,1958
Sugarcane,50
Maize,6
Cotton(lint),3
Onion,2
Potato,1
Sesamum,1
Rice,1
Cashewnut,1



Flagged record examples:


,State_Name,District_Name,Crop_Year,Crop,Area,Production,calculated_yield
5,Andaman and Nicobar Islands,NICOBARS,2000,Coconut,18168.00,65100000.0,3583.223250
14,Andaman and Nicobar Islands,NICOBARS,2001,Coconut,18190.00,64430000.0,3542.056075
23,Andaman and Nicobar Islands,NICOBARS,2002,Coconut,18240.00,67490000.0,3700.109649
32,Andaman and Nicobar Islands,NICOBARS,2003,Coconut,18284.74,68580000.0,3750.668590
41,Andaman and Nicobar Islands,NICOBARS,2004,Coconut,18394.70,52380000.0,2847.559351
50,Andaman and Nicobar Islands,NICOBARS,2005,Coconut,13876.57,52030000.0,3749.485644
57,Andaman and Nicobar Islands,NICOBARS,2006,Coconut,14358.00,59370000.0,4134.977016
77,Andaman and Nicobar Islands,NICOBARS,2010,Coconut,14560.00,71300000.0,4896.978022
84,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,2000,Coconut,6992.00,23900000.0,3418.192220
94,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,2001,Coconut,7015.00,25250000.0,3599.429793


In [ ]:
# STEP: Review suspicious yield records by crop

flagged = audit[audit["yield_review_flag"]].copy()

print("Total flagged records:", len(flagged))

print("\n1. Flagged records by crop:")
display(
    flagged.groupby("Crop")
    .agg(
        records=("calculated_yield", "size"),
        min_yield=("calculated_yield", "min"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max")
    )
    .sort_values("records", ascending=False)
)

print("\n2. Sugarcane records from Punjab in 2011:")
display(
    flagged[
        flagged["State_Name"].str.strip().str.lower().eq("punjab")
        & flagged["Crop"].str.strip().str.lower().eq("sugarcane")
        & (flagged["Crop_Year"] == 2011)
    ][[
        "State_Name", "District_Name", "Crop_Year",
        "Crop", "Area", "Production", "calculated_yield"
    ]]
)

print("\n3. Number of flagged records by year:")
display(
    flagged.groupby("Crop_Year")
    .size()
    .sort_index()
    .to_frame("Flagged_Records")
)

Total flagged records: 2023

1. Flagged records by crop:


,records,min_yield,median_yield,max_yield
Crop,,,,
Coconut,1958,0.000000,43.120000,38800.000000
Sugarcane,50,220.617757,1118.465777,88000.000000
Maize,6,725.250000,1134.750000,1494.000000
Cotton(lint),3,234.375000,235.555556,300.000000
Onion,2,204.078947,1177.039474,2150.000000
Cashewnut,1,9801.000000,9801.000000,9801.000000
Potato,1,213.846154,213.846154,213.846154
Rice,1,223.727273,223.727273,223.727273
Sesamum,1,235.000000,235.000000,235.000000



2. Sugarcane records from Punjab in 2011:


,State_Name,District_Name,Crop_Year,Crop,Area,Production,calculated_yield
161430,Punjab,AMRITSAR,2011,Sugarcane,3.0,203000.0,67666.666667
161948,Punjab,FATEHGARH SAHIB,2011,Sugarcane,2.0,151000.0,75500.000000
161967,Punjab,FAZILKA,2011,Sugarcane,1.0,73000.0,73000.000000
162328,Punjab,GURDASPUR,2011,Sugarcane,20.0,1491000.0,74550.000000
162523,Punjab,HOSHIARPUR,2011,Sugarcane,20.0,1293000.0,64650.000000
162701,Punjab,JALANDHAR,2011,Sugarcane,9.0,599000.0,66555.555556
162835,Punjab,KAPURTHALA,2011,Sugarcane,3.0,221000.0,73666.666667
163003,Punjab,LUDHIANA,2011,Sugarcane,2.0,161000.0,80500.000000
163657,Punjab,NAWANSHAHR,2011,Sugarcane,6.0,407000.0,67833.333333
163678,Punjab,PATHANKOT,2011,Sugarcane,4.0,298000.0,74500.000000



3. Number of flagged records by year:


,Flagged_Records
Crop_Year,
1997,96
1998,86
1999,86
2000,108
2001,107
2002,135
2003,134
2004,138
2005,114


In [ ]:
# STEP: Inspect suspicious yield records excluding Coconut

non_coconut_flags = audit[
    audit["yield_review_flag"]
    & ~audit["Crop"].str.strip().str.lower().eq("coconut")
].copy()

print("Suspicious non-coconut records:", len(non_coconut_flags))

display(
    non_coconut_flags[
        [
            "State_Name",
            "District_Name",
            "Crop_Year",
            "Season",
            "Crop",
            "Area",
            "Production",
            "calculated_yield"
        ]
    ].sort_values(
        "calculated_yield",
        ascending=False
    ).reset_index(drop=True)
)

Suspicious non-coconut records: 65


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,calculated_yield
0,Punjab,PATIALA,2011,Whole Year,Sugarcane,3.0,264000.0,88000.000000
1,Punjab,SANGRUR,2011,Whole Year,Sugarcane,2.0,163000.0,81500.000000
2,Punjab,LUDHIANA,2011,Whole Year,Sugarcane,2.0,161000.0,80500.000000
3,Punjab,FATEHGARH SAHIB,2011,Whole Year,Sugarcane,2.0,151000.0,75500.000000
4,Punjab,RUPNAGAR,2011,Whole Year,Sugarcane,3.0,224000.0,74666.666667
...,...,...,...,...,...,...,...,...
60,Tamil Nadu,THENI,1998,Whole Year,Sugarcane,16133.0,3617820.0,224.249675
61,Maharashtra,KOLHAPUR,1997,Kharif,Rice,1100.0,246100.0,223.727273
62,West Bengal,MEDINIPUR WEST,2013,Whole Year,Sugarcane,3210.0,708183.0,220.617757
63,Haryana,PANIPAT,2002,Whole Year,Potato,195.0,41700.0,213.846154


In [ ]:
# STEP: Compare suspicious records with nearby years
# This cell only inspects data; it does not delete or modify records.

# 1. Punjab sugarcane: compare 2010, 2011 and 2012
print("PUNJAB SUGARCANE: 2010–2012")

punjab_sugarcane = audit[
    audit["State_Name"].str.strip().str.lower().eq("punjab")
    & audit["Crop"].str.strip().str.lower().eq("sugarcane")
    & audit["Crop_Year"].between(2010, 2012)
].copy()

display(
    punjab_sugarcane[
        [
            "State_Name", "District_Name", "Crop_Year",
            "Area", "Production", "calculated_yield"
        ]
    ].sort_values(["District_Name", "Crop_Year"])
)

# 2. Compare the other suspicious crops with nearby years
print("\nOTHER SUSPICIOUS RECORDS: NEARBY-YEAR COMPARISON")

other_flags = non_coconut_flags[
    ~non_coconut_flags["Crop"].str.strip().str.lower().eq("sugarcane")
]

comparison_rows = []

for _, flagged_row in other_flags.iterrows():
    same_location_crop = audit[
        audit["State_Name"].eq(flagged_row["State_Name"])
        & audit["District_Name"].eq(flagged_row["District_Name"])
        & audit["Crop"].eq(flagged_row["Crop"])
        & audit["Crop_Year"].between(
            flagged_row["Crop_Year"] - 2,
            flagged_row["Crop_Year"] + 2
        )
    ].copy()

    same_location_crop["Flagged_Year"] = flagged_row["Crop_Year"]
    comparison_rows.append(same_location_crop)

if comparison_rows:
    comparison = pd.concat(comparison_rows, ignore_index=True)
    display(
        comparison[
            [
                "State_Name", "District_Name", "Crop",
                "Crop_Year", "Flagged_Year", "Season",
                "Area", "Production", "calculated_yield"
            ]
        ].sort_values(
            ["Crop", "State_Name", "District_Name", "Crop_Year"]
        ).reset_index(drop=True)
    )
else:
    print("No comparison records found.")

PUNJAB SUGARCANE: 2010–2012


,State_Name,District_Name,Crop_Year,Area,Production,calculated_yield
161430,Punjab,AMRITSAR,2011,3.0,203000.0,67666.666667
161440,Punjab,AMRITSAR,2012,2000.0,141000.0,70.500000
161948,Punjab,FATEHGARH SAHIB,2011,2.0,151000.0,75500.000000
161955,Punjab,FATEHGARH SAHIB,2012,3000.0,258000.0,86.000000
161967,Punjab,FAZILKA,2011,1.0,73000.0,73000.000000
161976,Punjab,FAZILKA,2012,1000.0,82000.0,82.000000
162168,Punjab,FIROZEPUR,2012,1000.0,67000.0,67.000000
162328,Punjab,GURDASPUR,2011,20.0,1491000.0,74550.000000
162336,Punjab,GURDASPUR,2012,22000.0,1594000.0,72.454545
162523,Punjab,HOSHIARPUR,2011,20.0,1293000.0,64650.000000



OTHER SUSPICIOUS RECORDS: NEARBY-YEAR COMPARISON


,State_Name,District_Name,Crop,Crop_Year,Flagged_Year,Season,Area,Production,calculated_yield
0,Tamil Nadu,PERAMBALUR,Cashewnut,2006,2008,Whole Year,35522.0,7114.0,0.200270
1,Tamil Nadu,PERAMBALUR,Cashewnut,2007,2008,Whole Year,33231.0,9130.0,0.274743
2,Tamil Nadu,PERAMBALUR,Cashewnut,2008,2008,Whole Year,1.0,9801.0,9801.000000
3,Tamil Nadu,PERAMBALUR,Cashewnut,2009,2008,Whole Year,2.0,1.0,0.500000
4,Maharashtra,PUNE,Cotton(lint),1997,1997,Whole Year,2.0,600.0,300.000000
...,...,...,...,...,...,...,...,...,...
86,Maharashtra,KOLHAPUR,Rice,1999,1997,Kharif,103200.0,261900.0,2.537791
87,Maharashtra,KOLHAPUR,Rice,1999,1997,Summer,300.0,700.0,2.333333
88,Gujarat,NARMADA,Sesamum,2011,2012,Kharif,100.0,100.0,1.000000
89,Gujarat,NARMADA,Sesamum,2012,2012,Kharif,200.0,100.0,0.500000


In [ ]:
# STEP: Review all suspicious non-coconut records

review_columns = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Crop",
    "Area",
    "Production",
    "calculated_yield"
]

print("Total suspicious non-coconut records:", len(non_coconut_flags))

review_df = (
    non_coconut_flags[review_columns]
    .sort_values(
        ["Crop", "State_Name", "District_Name", "Crop_Year", "Season"]
    )
    .reset_index(drop=True)
)

display(review_df)

Total suspicious non-coconut records: 65


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,calculated_yield
0,Tamil Nadu,PERAMBALUR,2008,Whole Year,Cashewnut,1.0,9801.0,9801.000000
1,Maharashtra,PUNE,1997,Whole Year,Cotton(lint),2.0,600.0,300.000000
2,Maharashtra,SANGLI,1997,Whole Year,Cotton(lint),32.0,7500.0,234.375000
3,Maharashtra,SATARA,1997,Whole Year,Cotton(lint),45.0,10600.0,235.555556
4,Maharashtra,AHMEDNAGAR,1997,Autumn,Maize,1.0,1113.0,1113.000000
...,...,...,...,...,...,...,...,...
60,Tamil Nadu,TUTICORIN,1997,Whole Year,Sugarcane,90.0,97000.0,1077.777778
61,Tamil Nadu,VELLORE,1997,Whole Year,Sugarcane,29072.0,22069000.0,759.115300
62,Tamil Nadu,VILLUPURAM,1997,Whole Year,Sugarcane,33175.0,37158000.0,1120.060286
63,Tamil Nadu,VIRUDHUNAGAR,1997,Whole Year,Sugarcane,2639.0,2833000.0,1073.512694


In [ ]:
# STEP: Audit records with very small reported area

small_area = audit[audit["Area"] <= 20].copy()

print("Total records with Area <= 20:", len(small_area))

print("\nRecords by crop:")
display(
    small_area["Crop"]
    .value_counts()
    .head(30)
    .to_frame("Record_Count")
)

print("\nRecords by year:")
display(
    small_area["Crop_Year"]
    .value_counts()
    .sort_index()
    .to_frame("Record_Count")
)

print("\nSmall-area records with high calculated yield:")
display(
    small_area[small_area["calculated_yield"] > 200][[
        "State_Name",
        "District_Name",
        "Crop_Year",
        "Season",
        "Crop",
        "Area",
        "Production",
        "calculated_yield"
    ]].sort_values(
        "calculated_yield",
        ascending=False
    ).head(50)
)

Total records with Area <= 20: 31142

Records by crop:


,Record_Count
Crop,
Turmeric,1482
Maize,1329
Moong(Green Gram),1280
Sweet potato,1181
Sunflower,1130
Urad,1017
Tobacco,951
Garlic,945
Castor seed,878



Records by year:


,Record_Count
Crop_Year,
1997,747
1998,1187
1999,1335
2000,1539
2001,1638
2002,2412
2003,2577
2004,1746
2005,1705



Small-area records with high calculated yield:


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,calculated_yield
163866,Punjab,PATIALA,2011,Whole Year,Sugarcane,3.0,264000.0,88000.000000
164316,Punjab,SANGRUR,2011,Whole Year,Sugarcane,2.0,163000.0,81500.000000
163003,Punjab,LUDHIANA,2011,Whole Year,Sugarcane,2.0,161000.0,80500.000000
161948,Punjab,FATEHGARH SAHIB,2011,Whole Year,Sugarcane,2.0,151000.0,75500.000000
164038,Punjab,RUPNAGAR,2011,Whole Year,Sugarcane,3.0,224000.0,74666.666667
162328,Punjab,GURDASPUR,2011,Whole Year,Sugarcane,20.0,1491000.0,74550.000000
163678,Punjab,PATHANKOT,2011,Whole Year,Sugarcane,4.0,298000.0,74500.000000
162835,Punjab,KAPURTHALA,2011,Whole Year,Sugarcane,3.0,221000.0,73666.666667
161967,Punjab,FAZILKA,2011,Whole Year,Sugarcane,1.0,73000.0,73000.000000
163657,Punjab,NAWANSHAHR,2011,Whole Year,Sugarcane,6.0,407000.0,67833.333333


In [ ]:
# STEP: Create a review list for known suspicious records

review = audit.copy()

review["review_reason"] = ""

# Coconut: production units need verification
coconut_mask = (
    review["Crop"].str.strip().str.lower() == "coconut"
)

review.loc[coconut_mask, "review_reason"] = (
    "Verify production unit before yield modelling"
)

# Punjab sugarcane, 2011: suspected area-scale/data-entry issue
punjab_sugarcane_mask = (
    review["State_Name"].str.strip().str.lower().eq("punjab")
    & review["Crop"].str.strip().str.lower().eq("sugarcane")
    & review["Crop_Year"].eq(2011)
)

review.loc[punjab_sugarcane_mask, "review_reason"] = (
    "Verify suspicious area and production values"
)

# Cashewnut anomaly: Perambalur, 2008
cashew_mask = (
    review["State_Name"].str.strip().str.lower().eq("tamil nadu")
    & review["District_Name"].str.strip().str.upper().eq("PERAMBALUR")
    & review["Crop"].str.strip().str.lower().eq("cashewnut")
    & review["Crop_Year"].eq(2008)
)

review.loc[cashew_mask, "review_reason"] = (
    "Verify unusually small area against original source"
)

review_list = review[review["review_reason"] != ""]

print("Records marked for review:", len(review_list))

display(
    review_list[
        [
            "State_Name",
            "District_Name",
            "Crop_Year",
            "Season",
            "Crop",
            "Area",
            "Production",
            "calculated_yield",
            "review_reason"
        ]
    ].head(30)
)

Records marked for review: 1974


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,calculated_yield,review_reason
5,Andaman and Nicobar Islands,NICOBARS,2000,Whole Year,Coconut,18168.00,65100000.0,3583.223250,Verify production unit before yield modelling
14,Andaman and Nicobar Islands,NICOBARS,2001,Whole Year,Coconut,18190.00,64430000.0,3542.056075,Verify production unit before yield modelling
23,Andaman and Nicobar Islands,NICOBARS,2002,Whole Year,Coconut,18240.00,67490000.0,3700.109649,Verify production unit before yield modelling
32,Andaman and Nicobar Islands,NICOBARS,2003,Whole Year,Coconut,18284.74,68580000.0,3750.668590,Verify production unit before yield modelling
41,Andaman and Nicobar Islands,NICOBARS,2004,Whole Year,Coconut,18394.70,52380000.0,2847.559351,Verify production unit before yield modelling
50,Andaman and Nicobar Islands,NICOBARS,2005,Whole Year,Coconut,13876.57,52030000.0,3749.485644,Verify production unit before yield modelling
57,Andaman and Nicobar Islands,NICOBARS,2006,Whole Year,Coconut,14358.00,59370000.0,4134.977016,Verify production unit before yield modelling
77,Andaman and Nicobar Islands,NICOBARS,2010,Whole Year,Coconut,14560.00,71300000.0,4896.978022,Verify production unit before yield modelling
84,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,2000,Whole Year,Coconut,6992.00,23900000.0,3418.192220,Verify production unit before yield modelling
94,Andaman and Nicobar Islands,NORTH AND MIDDLE ANDAMAN,2001,Whole Year,Coconut,7015.00,25250000.0,3599.429793,Verify production unit before yield modelling


In [ ]:
# STEP: Verify review categories

print("Records by review reason:")
display(
    review_list["review_reason"]
    .value_counts()
    .to_frame("Record_Count")
)

print("\nAll non-coconut records marked for review:")

display(
    review_list[
        review_list["Crop"].str.strip().str.lower().ne("coconut")
    ][
        [
            "State_Name",
            "District_Name",
            "Crop_Year",
            "Crop",
            "Area",
            "Production",
            "calculated_yield",
            "review_reason"
        ]
    ].reset_index(drop=True)
)

Records by review reason:


,Record_Count
review_reason,
Verify production unit before yield modelling,1958
Verify suspicious area and production values,15
Verify unusually small area against original source,1



All non-coconut records marked for review:


,State_Name,District_Name,Crop_Year,Crop,Area,Production,calculated_yield,review_reason
0,Punjab,AMRITSAR,2011,Sugarcane,3.0,203000.0,67666.666667,Verify suspicious area and production values
1,Punjab,FATEHGARH SAHIB,2011,Sugarcane,2.0,151000.0,75500.000000,Verify suspicious area and production values
2,Punjab,FAZILKA,2011,Sugarcane,1.0,73000.0,73000.000000,Verify suspicious area and production values
3,Punjab,GURDASPUR,2011,Sugarcane,20.0,1491000.0,74550.000000,Verify suspicious area and production values
4,Punjab,HOSHIARPUR,2011,Sugarcane,20.0,1293000.0,64650.000000,Verify suspicious area and production values
5,Punjab,JALANDHAR,2011,Sugarcane,9.0,599000.0,66555.555556,Verify suspicious area and production values
6,Punjab,KAPURTHALA,2011,Sugarcane,3.0,221000.0,73666.666667,Verify suspicious area and production values
7,Punjab,LUDHIANA,2011,Sugarcane,2.0,161000.0,80500.000000,Verify suspicious area and production values
8,Punjab,NAWANSHAHR,2011,Sugarcane,6.0,407000.0,67833.333333,Verify suspicious area and production values
9,Punjab,PATHANKOT,2011,Sugarcane,4.0,298000.0,74500.000000,Verify suspicious area and production values


In [ ]:
# STEP: Create a candidate dataset for yield modelling
# The original df and audit remain unchanged.

candidate = audit.copy()

# Exclude Coconut until production units are verified
coconut_mask = (
    candidate["Crop"].str.strip().str.lower().eq("coconut")
)

# Exclude suspicious Punjab sugarcane records from 2011
punjab_sugarcane_mask = (
    candidate["State_Name"].str.strip().str.lower().eq("punjab")
    & candidate["Crop"].str.strip().str.lower().eq("sugarcane")
    & candidate["Crop_Year"].eq(2011)
)

# Exclude the suspicious Cashewnut record
cashew_mask = (
    candidate["State_Name"].str.strip().str.lower().eq("tamil nadu")
    & candidate["District_Name"].str.strip().str.upper().eq("PERAMBALUR")
    & candidate["Crop"].str.strip().str.lower().eq("cashewnut")
    & candidate["Crop_Year"].eq(2008)
)

exclude_mask = (
    coconut_mask
    | punjab_sugarcane_mask
    | cashew_mask
)

candidate = candidate.loc[~exclude_mask].copy()

print("Original dataset rows:", len(df))
print("Valid area/production rows before exclusions:", len(audit))
print("Coconut records excluded:", int(coconut_mask.sum()))
print("Punjab sugarcane records excluded:", int(punjab_sugarcane_mask.sum()))
print("Cashewnut record excluded:", int(cashew_mask.sum()))
print("Candidate dataset rows:", len(candidate))
print("Remaining crop classes:", candidate["Crop"].nunique())

print("\nYield summary:")
display(candidate["calculated_yield"].describe().to_frame("Value"))

Original dataset rows: 246091
Valid area/production rows before exclusions: 242361
Coconut records excluded: 1958
Punjab sugarcane records excluded: 15
Cashewnut record excluded: 1
Candidate dataset rows: 240387
Remaining crop classes: 123

Yield summary:


,Value
count,240387.000000
mean,4.599282
std,18.990898
min,0.000000
25%,0.510204
50%,1.000000
75%,2.290162
max,2247.533300


In [ ]:
# STEP: Inspect remaining extreme yield values
# This is an inspection only; no records are removed.

print("Top 30 highest calculated yields:")

display(
    candidate[
        [
            "State_Name",
            "District_Name",
            "Crop_Year",
            "Season",
            "Crop",
            "Area",
            "Production",
            "calculated_yield"
        ]
    ]
    .sort_values("calculated_yield", ascending=False)
    .head(30)
    .reset_index(drop=True)
)

print("\nCrop-level yield summary:")

crop_yield_summary = (
    candidate.groupby("Crop")["calculated_yield"]
    .agg(
        records="count",
        median_yield="median",
        p95_yield=lambda x: x.quantile(0.95),
        max_yield="max"
    )
    .sort_values("max_yield", ascending=False)
)

display(crop_yield_summary.head(25))

Top 30 highest calculated yields:


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,calculated_yield
0,Tamil Nadu,MADURAI,1997,Whole Year,Sugarcane,8108.0,18223000.0,2247.533300
1,Haryana,KARNAL,2008,Whole Year,Onion,2.0,4300.0,2150.000000
2,Tamil Nadu,THIRUVALLUR,1997,Whole Year,Sugarcane,5964.0,11811000.0,1980.382294
3,Maharashtra,NASHIK,1997,Autumn,Maize,1.0,1494.0,1494.000000
4,Tamil Nadu,KANCHIPURAM,1997,Whole Year,Sugarcane,6735.0,9896000.0,1469.339272
5,Tamil Nadu,NAMAKKAL,1997,Whole Year,Sugarcane,9415.0,13608000.0,1445.353160
6,Tamil Nadu,TIRUCHIRAPPALLI,1997,Whole Year,Sugarcane,4315.0,5806000.0,1345.538818
7,Maharashtra,JALGAON,1997,Autumn,Maize,3.0,3980.0,1326.666667
8,Tamil Nadu,CUDDALORE,1997,Whole Year,Sugarcane,23826.0,28322000.0,1188.701419
9,Tamil Nadu,KARUR,1997,Whole Year,Sugarcane,5961.0,7081000.0,1187.887938



Crop-level yield summary:


,records,median_yield,p95_yield,max_yield
Crop,,,,
Sugarcane,7812,53.170909,106.838291,2247.533300
Onion,6984,10.297120,25.739948,2150.000000
Maize,13787,1.795918,4.816606,1494.000000
Cotton(lint),4382,1.367349,3.999432,300.000000
Sesamum,8821,0.333333,0.882353,235.000000
Rice,15082,1.938529,3.638213,223.727273
Potato,6914,10.833194,25.997861,213.846154
Banana,3133,21.804276,63.566667,138.421053
Papaya,483,15.500000,108.934491,134.142636


In [ ]:
# STEP: Investigate extreme yields by year and district

# 1. Sugarcane yield summary by year
sugarcane = candidate[
    candidate["Crop"].str.strip().str.lower().eq("sugarcane")
].copy()

print("SUGARCANE: YEAR-WISE YIELD SUMMARY")

display(
    sugarcane.groupby("Crop_Year")["calculated_yield"]
    .agg(
        records="count",
        median_yield="median",
        p95_yield=lambda x: x.quantile(0.95),
        max_yield="max"
    )
    .sort_index()
)

# 2. Tamil Nadu sugarcane records for 1997
print("\nTAMIL NADU SUGARCANE: 1997")

display(
    sugarcane[
        sugarcane["State_Name"].str.strip().str.lower().eq("tamil nadu")
        & sugarcane["Crop_Year"].eq(1997)
    ][
        [
            "District_Name", "Area", "Production",
            "calculated_yield"
        ]
    ].sort_values("calculated_yield", ascending=False)
)

# 3. Onion records for Karnal around 2008
print("\nONION: KARNAL, 2006–2010")

display(
    candidate[
        candidate["District_Name"].str.strip().str.upper().eq("KARNAL")
        & candidate["Crop"].str.strip().str.lower().eq("onion")
        & candidate["Crop_Year"].between(2006, 2010)
    ][
        [
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "Production", "calculated_yield"
        ]
    ].sort_values(["Crop_Year", "Season"])
)

SUGARCANE: YEAR-WISE YIELD SUMMARY


,records,median_yield,p95_yield,max_yield
Crop_Year,,,,
1997,330,55.366046,1073.572929,2247.533300
1998,408,51.851274,121.504860,303.484773
1999,444,54.189472,111.969394,162.500000
2000,466,49.522222,107.116143,162.000000
2001,461,51.933015,100.172612,158.572753
2002,466,49.889998,92.379624,162.000000
2003,461,49.284046,100.000000,163.333333
2004,423,51.274725,100.691208,162.500000
2005,469,53.787998,99.200822,147.258056



TAMIL NADU SUGARCANE: 1997


,District_Name,Area,Production,calculated_yield
182001,MADURAI,8108.0,18223000.0,2247.533300
186870,THIRUVALLUR,5964.0,11811000.0,1980.382294
180534,KANCHIPURAM,6735.0,9896000.0,1469.339272
182819,NAMAKKAL,9415.0,13608000.0,1445.353160
187574,TIRUCHIRAPPALLI,4315.0,5806000.0,1345.538818
178362,CUDDALORE,23826.0,28322000.0,1188.701419
181195,KARUR,5961.0,7081000.0,1187.887938
184641,SALEM,13759.0,15866000.0,1153.136129
177823,COIMBATORE,10756.0,12272000.0,1140.944589
190249,VILLUPURAM,33175.0,37158000.0,1120.060286



ONION: KARNAL, 2006–2010


,State_Name,District_Name,Crop_Year,Season,Area,Production,calculated_yield
68458,Haryana,KARNAL,2007,Whole Year,5.0,100.0,20.000000
68479,Haryana,KARNAL,2008,Whole Year,2.0,4300.0,2150.000000
68524,Haryana,KARNAL,2010,Whole Year,22.0,400.0,18.181818


In [ ]:
# STEP: Locate extreme sugarcane yields by state and year
# Inspection only — no records are deleted.

sugarcane_extremes = sugarcane[
    sugarcane["calculated_yield"] > 200
].copy()

print("Sugarcane records with calculated yield > 200:",
      len(sugarcane_extremes))

print("\nExtreme sugarcane records by year:")
display(
    sugarcane_extremes.groupby("Crop_Year")
    .agg(
        flagged_records=("calculated_yield", "size"),
        affected_states=("State_Name", "nunique"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max")
    )
    .sort_index()
)

print("\nExtreme sugarcane records by state:")
display(
    sugarcane_extremes.groupby("State_Name")
    .agg(
        flagged_records=("calculated_yield", "size"),
        affected_years=("Crop_Year", "nunique"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max")
    )
    .sort_values("flagged_records", ascending=False)
)

print("\nTamil Nadu sugarcane records from 1997:")
display(
    sugarcane[
        sugarcane["State_Name"].str.strip().str.lower().eq("tamil nadu")
        & sugarcane["Crop_Year"].eq(1997)
    ][
        [
            "District_Name", "Area", "Production",
            "calculated_yield"
        ]
    ].sort_values("calculated_yield", ascending=False)
)

Sugarcane records with calculated yield > 200: 35

Extreme sugarcane records by year:


,flagged_records,affected_states,median_yield,max_yield
Crop_Year,,,,
1997,28,2,1075.768639,2247.533300
1998,3,1,242.657872,303.484773
2008,3,2,258.352766,314.736149
2013,1,1,220.617757,220.617757



Extreme sugarcane records by state:


,flagged_records,affected_years,median_yield,max_yield
State_Name,,,,
Tamil Nadu,32,3,1073.650924,2247.533300
Bihar,1,1,500.491525,500.491525
Kerala,1,1,229.285714,229.285714
West Bengal,1,1,220.617757,220.617757



Tamil Nadu sugarcane records from 1997:


,District_Name,Area,Production,calculated_yield
182001,MADURAI,8108.0,18223000.0,2247.533300
186870,THIRUVALLUR,5964.0,11811000.0,1980.382294
180534,KANCHIPURAM,6735.0,9896000.0,1469.339272
182819,NAMAKKAL,9415.0,13608000.0,1445.353160
187574,TIRUCHIRAPPALLI,4315.0,5806000.0,1345.538818
178362,CUDDALORE,23826.0,28322000.0,1188.701419
181195,KARUR,5961.0,7081000.0,1187.887938
184641,SALEM,13759.0,15866000.0,1153.136129
177823,COIMBATORE,10756.0,12272000.0,1140.944589
190249,VILLUPURAM,33175.0,37158000.0,1120.060286


In [ ]:
# STEP: Review all 35 extreme sugarcane records
# Inspection only — no records are modified or deleted.

sugarcane_review = sugarcane[
    sugarcane["calculated_yield"] > 200
].copy()

print("Total records requiring review:", len(sugarcane_review))

display(
    sugarcane_review[
        [
            "State_Name",
            "District_Name",
            "Crop_Year",
            "Area",
            "Production",
            "calculated_yield"
        ]
    ]
    .sort_values(
        ["Crop_Year", "State_Name", "calculated_yield"],
        ascending=[True, True, False]
    )
    .reset_index(drop=True)
)

Total records requiring review: 35


,State_Name,District_Name,Crop_Year,Area,Production,calculated_yield
0,Bihar,BUXAR,1997,59.0,29529.0,500.491525
1,Tamil Nadu,MADURAI,1997,8108.0,18223000.0,2247.533300
2,Tamil Nadu,THIRUVALLUR,1997,5964.0,11811000.0,1980.382294
3,Tamil Nadu,KANCHIPURAM,1997,6735.0,9896000.0,1469.339272
4,Tamil Nadu,NAMAKKAL,1997,9415.0,13608000.0,1445.353160
5,Tamil Nadu,TIRUCHIRAPPALLI,1997,4315.0,5806000.0,1345.538818
6,Tamil Nadu,CUDDALORE,1997,23826.0,28322000.0,1188.701419
7,Tamil Nadu,KARUR,1997,5961.0,7081000.0,1187.887938
8,Tamil Nadu,SALEM,1997,13759.0,15866000.0,1153.136129
9,Tamil Nadu,COIMBATORE,1997,10756.0,12272000.0,1140.944589


In [ ]:
# STEP: Save extreme sugarcane records for source verification

sugarcane_review = candidate[
    candidate["Crop"].str.strip().str.lower().eq("sugarcane")
    & (candidate["calculated_yield"] > 200)
].copy()

print("Sugarcane records requiring verification:",
      len(sugarcane_review))

# Save a review copy; the candidate dataset remains unchanged
sugarcane_review.to_csv(
    "sugarcane_extreme_yield_review.csv",
    index=False
)

print("Saved: sugarcane_extreme_yield_review.csv")

Sugarcane records requiring verification: 35
Saved: sugarcane_extreme_yield_review.csv


In [ ]:
import os

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"
os.makedirs(DRIVE_FOLDER, exist_ok=True)

print("Google Drive folder:", DRIVE_FOLDER)
print("Files in folder:", os.listdir(DRIVE_FOLDER))

Google Drive folder: /content/drive/MyDrive/yield prediction
Files in folder: []


In [ ]:
import os

output_path = os.path.join(
    DRIVE_FOLDER,
    "sugarcane_extreme_yield_review.csv"
)

sugarcane_review.to_csv(output_path, index=False)

print("Saved successfully to Google Drive!")
print("File path:", output_path)
print("Rows saved:", len(sugarcane_review))

Saved successfully to Google Drive!
File path: /content/drive/MyDrive/yield prediction/sugarcane_extreme_yield_review.csv
Rows saved: 35


In [ ]:
import os

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

output_path = os.path.join(
    DRIVE_FOLDER,
    "sugarcane_extreme_yield_review.csv"
)

sugarcane_review.to_csv(output_path, index=False)

print("Saved successfully to Google Drive!")
print("File path:", output_path)
print("Rows saved:", len(sugarcane_review))

Saved successfully to Google Drive!
File path: /content/drive/MyDrive/yield prediction/sugarcane_extreme_yield_review.csv
Rows saved: 35


In [ ]:
import pandas as pd
import os

review_path = os.path.join(
    "/content/drive/MyDrive/yield prediction",
    "sugarcane_extreme_yield_review.csv"
)

sugarcane_review = pd.read_csv(review_path)

display(
    sugarcane_review[
        [
            "State_Name",
            "District_Name",
            "Crop_Year",
            "Season",
            "Area",
            "Production",
            "calculated_yield"
        ]
    ].sort_values("calculated_yield", ascending=False)
)

,State_Name,District_Name,Crop_Year,Season,Area,Production,calculated_yield
11,Tamil Nadu,MADURAI,1997,Whole Year,8108.0,18223000.0,2247.533300
25,Tamil Nadu,THIRUVALLUR,1997,Whole Year,5964.0,11811000.0,1980.382294
8,Tamil Nadu,KANCHIPURAM,1997,Whole Year,6735.0,9896000.0,1469.339272
14,Tamil Nadu,NAMAKKAL,1997,Whole Year,9415.0,13608000.0,1445.353160
27,Tamil Nadu,TIRUCHIRAPPALLI,1997,Whole Year,4315.0,5806000.0,1345.538818
4,Tamil Nadu,CUDDALORE,1997,Whole Year,23826.0,28322000.0,1188.701419
10,Tamil Nadu,KARUR,1997,Whole Year,5961.0,7081000.0,1187.887938
19,Tamil Nadu,SALEM,1997,Whole Year,13759.0,15866000.0,1153.136129
2,Tamil Nadu,COIMBATORE,1997,Whole Year,10756.0,12272000.0,1140.944589
32,Tamil Nadu,VILLUPURAM,1997,Whole Year,33175.0,37158000.0,1120.060286


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

# Districts from the flagged Tamil Nadu sugarcane records
tn_districts = sugarcane_review.loc[
    sugarcane_review["State_Name"].str.strip().str.lower().eq("tamil nadu"),
    "District_Name"
].dropna().unique()

# Use the original dataset for comparison
comparison = df[
    df["State_Name"].str.strip().str.lower().eq("tamil nadu")
    & df["Crop"].str.strip().str.lower().eq("sugarcane")
    & df["District_Name"].isin(tn_districts)
    & df["Crop_Year"].between(1997, 2000)
].copy()

comparison["calculated_yield"] = (
    comparison["Production"] / comparison["Area"]
)

columns = [
    "State_Name",
    "District_Name",
    "Crop_Year",
    "Season",
    "Area",
    "Production",
    "calculated_yield"
]

comparison = comparison[columns].sort_values(
    ["District_Name", "Crop_Year", "Season"]
)

display(comparison)

output_path = os.path.join(
    DRIVE_FOLDER,
    "sugarcane_nearby_years_comparison.csv"
)

comparison.to_csv(output_path, index=False)

print("Saved comparison to:", output_path)
print("Records compared:", len(comparison))

,State_Name,District_Name,Crop_Year,Season,Area,Production,calculated_yield
177823,Tamil Nadu,COIMBATORE,1997,Whole Year,10756.0,12272000.0,1140.944589
177856,Tamil Nadu,COIMBATORE,1998,Whole Year,13193.0,1467460.0,111.230198
177884,Tamil Nadu,COIMBATORE,1999,Whole Year,11939.0,1195037.0,100.095234
177915,Tamil Nadu,COIMBATORE,2000,Whole Year,11167.0,1117764.0,100.095281
178362,Tamil Nadu,CUDDALORE,1997,Whole Year,23826.0,28322000.0,1188.701419
...,...,...,...,...,...,...,...
190338,Tamil Nadu,VILLUPURAM,2000,Whole Year,40233.0,4380107.0,108.868516
190745,Tamil Nadu,VIRUDHUNAGAR,1997,Whole Year,2639.0,2833000.0,1073.512694
190774,Tamil Nadu,VIRUDHUNAGAR,1998,Whole Year,3707.0,454970.0,122.732668
190798,Tamil Nadu,VIRUDHUNAGAR,1999,Whole Year,3052.0,317444.0,104.011796


Saved comparison to: /content/drive/MyDrive/yield prediction/sugarcane_nearby_years_comparison.csv
Records compared: 108


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

comparison_path = os.path.join(
    DRIVE_FOLDER,
    "sugarcane_nearby_years_comparison.csv"
)

comparison = pd.read_csv(comparison_path)

year_summary = (
    comparison.groupby("Crop_Year")
    .agg(
        records=("calculated_yield", "size"),
        median_yield=("calculated_yield", "median"),
        min_yield=("calculated_yield", "min"),
        max_yield=("calculated_yield", "max"),
        total_area=("Area", "sum"),
        total_production=("Production", "sum")
    )
    .round(2)
    .reset_index()
)

display(year_summary)

summary_path = os.path.join(
    DRIVE_FOLDER,
    "sugarcane_yearwise_quality_summary.csv"
)

year_summary.to_csv(summary_path, index=False)

print("Saved summary to:", summary_path)

,Crop_Year,records,median_yield,min_yield,max_yield,total_area,total_production
0,1997,27,1077.78,623.33,2247.53,282845.0,307019000.0
1,1998,27,122.73,79.58,303.48,347910.0,46672800.0
2,1999,27,107.18,87.14,133.61,332332.0,35501817.0
3,2000,27,107.14,87.14,133.61,317473.0,34062855.0


Saved summary to: /content/drive/MyDrive/yield prediction/sugarcane_yearwise_quality_summary.csv


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

# Select Tamil Nadu sugarcane records from 1997
tn_sugarcane_1997 = comparison[
    comparison["State_Name"].str.strip().str.lower().eq("tamil nadu")
    & comparison["Crop_Year"].eq(1997)
].copy()

tn_sugarcane_1997["review_status"] = "Needs official-source verification"

output_path = os.path.join(
    DRIVE_FOLDER,
    "tamil_nadu_sugarcane_1997_review.csv"
)

tn_sugarcane_1997.to_csv(output_path, index=False)

print("Records saved:", len(tn_sugarcane_1997))
print("Saved to:", output_path)

display(tn_sugarcane_1997.sort_values(
    "calculated_yield", ascending=False
))

Records saved: 27
Saved to: /content/drive/MyDrive/yield prediction/tamil_nadu_sugarcane_1997_review.csv


,State_Name,District_Name,Crop_Year,Season,Area,Production,calculated_yield,review_status
28,Tamil Nadu,MADURAI,1997,Whole Year,8108.0,18223000.0,2247.533300,Needs official-source verification
72,Tamil Nadu,THIRUVALLUR,1997,Whole Year,5964.0,11811000.0,1980.382294,Needs official-source verification
20,Tamil Nadu,KANCHIPURAM,1997,Whole Year,6735.0,9896000.0,1469.339272,Needs official-source verification
36,Tamil Nadu,NAMAKKAL,1997,Whole Year,9415.0,13608000.0,1445.353160,Needs official-source verification
80,Tamil Nadu,TIRUCHIRAPPALLI,1997,Whole Year,4315.0,5806000.0,1345.538818,Needs official-source verification
4,Tamil Nadu,CUDDALORE,1997,Whole Year,23826.0,28322000.0,1188.701419,Needs official-source verification
24,Tamil Nadu,KARUR,1997,Whole Year,5961.0,7081000.0,1187.887938,Needs official-source verification
52,Tamil Nadu,SALEM,1997,Whole Year,13759.0,15866000.0,1153.136129,Needs official-source verification
0,Tamil Nadu,COIMBATORE,1997,Whole Year,10756.0,12272000.0,1140.944589,Needs official-source verification
100,Tamil Nadu,VILLUPURAM,1997,Whole Year,33175.0,37158000.0,1120.060286,Needs official-source verification


In [ ]:
import requests
from getpass import getpass

API_KEY = getpass("Enter your NEW API key: ")

url = "https://data.gov.in/api/datastore/resource.json"

params = {
    "resource_id": "35be999b-0208-4354-b557-f6ca9a5355de",
    "api-key": API_KEY,
    "limit": 10
}

try:
    response = requests.get(url, params=params, timeout=30)
    print("HTTP status:", response.status_code)
    print("Response:", response.text[:2000])

except requests.RequestException as e:
    print("Connection error:", type(e).__name__)
    print("Details:", str(e))

Enter your NEW API key: ··········
Connection error: ConnectionError
Details: HTTPSConnectionPool(host='data.gov.in', port=443): Max retries exceeded with url: /api/datastore/resource.json?resource_id=35be999b-0208-4354-b557-f6ca9a5355de&api-key=579b464db66ec23bdd0000010d2adffafe254bfb7bf207dac74ed6f3&limit=10 (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7d4d49d03750>: Failed to establish a new connection: [Errno 111] Connection refused'))


In [ ]:
import requests

for url in [
    "https://api.data.gov.in/",
    "https://www.data.gov.in/"
]:
    try:
        response = requests.get(url, timeout=15)
        print(url, "HTTP", response.status_code)
    except requests.RequestException as e:
        print(url, type(e).__name__, str(e))

https://api.data.gov.in/ ConnectionError HTTPSConnectionPool(host='api.data.gov.in', port=443): Max retries exceeded with url: / (Caused by NewConnectionError('<urllib3.connection.HTTPSConnection object at 0x7d4d49d03c50>: Failed to establish a new connection: [Errno 111] Connection refused'))
https://www.data.gov.in/ HTTP 503


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

# Load the original dataset
input_path = os.path.join(
    DRIVE_FOLDER,
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(input_path)

print("Original shape:", df.shape)
print("Original crop count:", df["Crop"].nunique())

# Remove Coconut and Sugarcane
crops_to_remove = ["coconut", "sugarcane"]

cleaned_df = df[
    ~df["Crop"].str.strip().str.lower().isin(crops_to_remove)
].copy()

print("\nCleaned shape:", cleaned_df.shape)
print("Remaining crop count:", cleaned_df["Crop"].nunique())

print("\nVerify removed crops:")
print(
    cleaned_df[
        cleaned_df["Crop"].str.strip().str.lower().isin(crops_to_remove)
    ].shape[0]
)

# Save as a new file; do not overwrite the original
output_path = os.path.join(
    DRIVE_FOLDER,
    "crop_production_without_coconut_sugarcane.csv"
)

cleaned_df.to_csv(output_path, index=False)

print("\nSaved cleaned dataset to:", output_path)

Original shape: (246091, 11)
Original crop count: 124

Cleaned shape: (236185, 11)
Remaining crop count: 122

Verify removed crops:
0

Saved cleaned dataset to: /content/drive/MyDrive/yield prediction/crop_production_without_coconut_sugarcane.csv


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

INPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "crop_production_124_with_weather_soil.csv"
)

OUTPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_dataset_without_coconut.csv"
)

# Load original dataset
df = pd.read_csv(INPUT_PATH)

print("Original rows:", len(df))
print("Original crop count:", df["Crop"].nunique())

# Remove only Coconut
cleaned_df = df[
    ~df["Crop"].astype(str).str.strip().str.lower().eq("coconut")
].copy()

# Save the cleaned dataset to Google Drive
cleaned_df.to_csv(OUTPUT_PATH, index=False)

print("\nCleaning completed!")
print("Rows removed:", len(df) - len(cleaned_df))
print("Remaining rows:", len(cleaned_df))
print("Remaining crop count:", cleaned_df["Crop"].nunique())
print("Sugarcane retained:",
      cleaned_df["Crop"].astype(str).str.strip().str.lower().eq("sugarcane").sum())
print("Coconut retained:",
      cleaned_df["Crop"].astype(str).str.strip().str.lower().eq("coconut").sum())
print("\nSaved to:", OUTPUT_PATH)

Original rows: 246091
Original crop count: 124

Cleaning completed!
Rows removed: 1985
Remaining rows: 244106
Remaining crop count: 123
Sugarcane retained: 7921
Coconut retained: 0

Saved to: /content/drive/MyDrive/yield prediction/yield_prediction_dataset_without_coconut.csv


In [ ]:
import os
import pandas as pd

DATASET_PATH = (
    "/content/drive/MyDrive/yield prediction/"
    "yield_prediction_dataset_without_coconut.csv"
)

df_clean = pd.read_csv(DATASET_PATH)

print("Dataset shape:", df_clean.shape)
print("Unique crops:", df_clean["Crop"].nunique())
print("Missing values:", df_clean.isnull().sum().sum())
print("Duplicate rows:", df_clean.duplicated().sum())

print("\nRemaining Coconut records:",
      df_clean["Crop"].astype(str).str.strip().str.lower().eq("coconut").sum())

print("\nSugarcane records:",
      df_clean["Crop"].astype(str).str.strip().str.lower().eq("sugarcane").sum())

print("\nRecords by year:")
display(df_clean["Crop_Year"].value_counts().sort_index())


Dataset shape: (244106, 11)
Unique crops: 123
Missing values: 3703
Duplicate rows: 0

Remaining Coconut records: 0

Sugarcane records: 7921

Records by year:


,count
Crop_Year,
1997,8842
1998,11450
1999,12429
2000,13550
2001,13254
2002,16537
2003,17153
2004,13979
2005,13684


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

# Count missing values in each column
missing_report = pd.DataFrame({
    "Missing_Count": df_clean.isnull().sum(),
    "Missing_Percentage": (
        df_clean.isnull().mean() * 100
    ).round(2)
}).sort_values("Missing_Count", ascending=False)

display(missing_report)

# Save report to Drive
report_path = os.path.join(
    DRIVE_FOLDER,
    "yield_missing_values_audit.csv"
)

missing_report.to_csv(report_path)

print("Saved report:", report_path)

,Missing_Count,Missing_Percentage
Production,3703,1.52
District_Name,0,0.00
State_Name,0,0.00
Crop_Year,0,0.00
Season,0,0.00
Crop,0,0.00
Area,0,0.00
temperature,0,0.00
humidity,0,0.00
rainfall_mm,0,0.00


Saved report: /content/drive/MyDrive/yield prediction/yield_missing_values_audit.csv


In [ ]:
# Compare the last three years
for year in [2013, 2014, 2015]:
    year_df = df_clean[df_clean["Crop_Year"] == year]

    print(f"\nYear: {year}")
    print("Total records:", len(year_df))
    print("Unique crops:", year_df["Crop"].nunique())
    print("Unique districts:", year_df["District_Name"].nunique())
    print("Missing Production:", year_df["Production"].isna().sum())


Year: 2013
Total records: 13524
Unique crops: 77
Unique districts: 506
Missing Production: 171

Year: 2014
Total records: 10904
Unique crops: 75
Unique districts: 427
Missing Production: 153

Year: 2015
Total records: 562
Unique crops: 17
Unique districts: 34
Missing Production: 1


In [ ]:
import os
import pandas as pd
import numpy as np

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

INPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_dataset_without_coconut.csv"
)

OUTPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_dataset_ready.csv"
)

# Load cleaned dataset
df_clean = pd.read_csv(INPUT_PATH)

print("Input rows:", len(df_clean))

# Keep rows where production and area are valid
valid_mask = (
    df_clean["Production"].notna()
    & df_clean["Area"].notna()
    & (df_clean["Area"] > 0)
    & (df_clean["Production"] >= 0)
)

yield_df = df_clean.loc[valid_mask].copy()

# Create yield target
yield_df["yield_tonnes_per_hectare"] = (
    yield_df["Production"] / yield_df["Area"]
)

# Save training-ready dataset
yield_df.to_csv(OUTPUT_PATH, index=False)

print("\nYield dataset created!")
print("Rows retained:", len(yield_df))
print("Rows excluded:", len(df_clean) - len(yield_df))
print("Crop classes:", yield_df["Crop"].nunique())
print("Missing yield targets:",
      yield_df["yield_tonnes_per_hectare"].isna().sum())
print("Saved to:", OUTPUT_PATH)

print("\nYield summary:")
print(yield_df["yield_tonnes_per_hectare"].describe())

Input rows: 244106

Yield dataset created!
Rows retained: 240403
Rows excluded: 3703
Crop classes: 123
Missing yield targets: 0
Saved to: /content/drive/MyDrive/yield prediction/yield_prediction_dataset_ready.csv

Yield summary:
count    240403.000000
mean          9.080575
std         568.836296
min           0.000000
25%           0.510204
50%           1.000000
75%           2.291430
max       88000.000000
Name: yield_tonnes_per_hectare, dtype: float64


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

INPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_dataset_ready.csv"
)

df_yield = pd.read_csv(INPUT_PATH)

# Summarize yield distribution by crop
crop_summary = (
    df_yield.groupby("Crop")["yield_tonnes_per_hectare"]
    .agg(
        Records="count",
        Median_Yield="median",
        P95_Yield=lambda x: x.quantile(0.95),
        P99_Yield=lambda x: x.quantile(0.99),
        Maximum_Yield="max"
    )
    .round(3)
    .sort_values("Maximum_Yield", ascending=False)
)

display(crop_summary.head(20))

# Save the report to Google Drive
output_path = os.path.join(
    DRIVE_FOLDER,
    "yield_by_crop_quality_report.csv"
)

crop_summary.to_csv(output_path)

print("Saved report:", output_path)


,Records,Median_Yield,P95_Yield,P99_Yield,Maximum_Yield
Crop,,,,,
Sugarcane,7827,53.241,107.086,146.145,88000.000
Cashewnut,1068,0.396,1.030,1.894,9801.000
Onion,6984,10.297,25.740,33.000,2150.000
Maize,13787,1.796,4.817,7.230,1494.000
Cotton(lint),4382,1.367,3.999,5.786,300.000
Sesamum,8821,0.333,0.882,1.102,235.000
Rice,15082,1.939,3.638,4.518,223.727
Potato,6914,10.833,25.998,31.350,213.846
Banana,3133,21.804,63.567,80.259,138.421


Saved report: /content/drive/MyDrive/yield prediction/yield_by_crop_quality_report.csv


In [ ]:
import os
import numpy as np
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

INPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_dataset_ready.csv"
)

MODEL_DATA_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_model_data_v1.csv"
)

REVIEW_PATH = os.path.join(
    DRIVE_FOLDER,
    "crop_specific_extreme_yield_review.csv"
)

df_yield = pd.read_csv(INPUT_PATH)

target = "yield_tonnes_per_hectare"

# Create a log-transformed target.
# log1p safely handles zero yields.
df_yield["log_yield_target"] = np.log1p(df_yield[target])

# Flag observations above each crop's 99th percentile.
crop_p99 = df_yield.groupby("Crop")[target].transform(
    lambda values: values.quantile(0.99)
)

extreme_mask = df_yield[target] > crop_p99

review_df = df_yield.loc[extreme_mask].copy()

# Save the review file separately.
review_df.to_csv(REVIEW_PATH, index=False)

# Save all rows for future model preparation.
df_yield.to_csv(MODEL_DATA_PATH, index=False)

print("Rows retained:", len(df_yield))
print("Crop classes:", df_yield["Crop"].nunique())
print("Extreme-yield records flagged:", int(extreme_mask.sum()))
print("Missing log targets:", df_yield["log_yield_target"].isna().sum())

print("\nModel dataset:", MODEL_DATA_PATH)
print("Review dataset:", REVIEW_PATH)

Rows retained: 240403
Crop classes: 123
Extreme-yield records flagged: 2419
Missing log targets: 0

Model dataset: /content/drive/MyDrive/yield prediction/yield_prediction_model_data_v1.csv
Review dataset: /content/drive/MyDrive/yield prediction/crop_specific_extreme_yield_review.csv


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

INPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_model_data_v1.csv"
)

df_model = pd.read_csv(INPUT_PATH)

train_df = df_model[df_model["Crop_Year"].between(1997, 2012)].copy()
val_df = df_model[df_model["Crop_Year"] == 2013].copy()
test_df = df_model[df_model["Crop_Year"] == 2014].copy()
holdout_2015 = df_model[df_model["Crop_Year"] == 2015].copy()

for name, data in [
    ("Train", train_df),
    ("Validation", val_df),
    ("Test", test_df),
    ("2015 Holdout", holdout_2015)
]:
    print(
        f"{name}: rows={len(data):,}, "
        f"crops={data['Crop'].nunique()}"
    )

print("\nCrop classes in train but missing from test:")
print(sorted(set(train_df["Crop"]) - set(test_df["Crop"])))

print("\nCrop classes in test but missing from train:")
print(sorted(set(test_df["Crop"]) - set(train_df["Crop"])))

Train: rows=215,738, crops=116
Validation: rows=13,353, crops=76
Test: rows=10,751, crops=74
2015 Holdout: rows=561, crops=17

Crop classes in train but missing from test:
['Apple', 'Arcanut (Processed)', 'Ash Gourd', 'Atcanut (Raw)', 'Beans & Mutter(Vegetable)', 'Beet Root', 'Ber', 'Bhindi', 'Bitter Gourd', 'Bottle Gourd', 'Carrot', 'Cashewnut Processed', 'Cashewnut Raw', 'Cauliflower', 'Citrus Fruit', 'Coffee', 'Cond-spcs other', 'Cucumber', 'Jack Fruit', 'Jute & mesta', 'Kapas', 'Korra', 'Lab-Lab', 'Litchi', 'Other Citrus Fruit', 'Other Dry Fruit', 'Peach', 'Pear', 'Peas  (vegetable)', 'Plums', 'Pome Fruit', 'Pump Kin', 'Redish', 'Ribed Guard', 'Rubber', 'Samai', 'Snak Guard', 'Tea', 'Turnip', 'Varagu', 'Water Melon', 'Yam', 'other fibres', 'other misc. pulses']

Crop classes in test but missing from train:
['Bean', 'Lentil']


In [ ]:
import os
import pandas as pd

DRIVE_FOLDER = "/content/drive/MyDrive/yield prediction"

INPUT_PATH = os.path.join(
    DRIVE_FOLDER,
    "yield_prediction_model_data_v1.csv"
)

df_model = pd.read_csv(INPUT_PATH)

train_df = df_model[df_model["Crop_Year"].between(1997, 2012)].copy()
val_df = df_model[df_model["Crop_Year"] == 2013].copy()
test_df = df_model[df_model["Crop_Year"] == 2014].copy()

# Use crops present in all three sets
common_crops = (
    set(train_df["Crop"])
    & set(val_df["Crop"])
    & set(test_df["Crop"])
)

train_common = train_df[train_df["Crop"].isin(common_crops)].copy()
val_common = val_df[val_df["Crop"].isin(common_crops)].copy()
test_common = test_df[test_df["Crop"].isin(common_crops)].copy()

print("Common crop classes:", len(common_crops))
print("Training records:", len(train_common))
print("Validation records:", len(val_common))
print("Test records:", len(test_common))

print("\nCrops included:")
print(sorted(common_crops))

Common crop classes: 69
Training records: 212081
Validation records: 13277
Test records: 10615

Crops included:
['Arecanut', 'Arhar/Tur', 'Bajra', 'Banana', 'Barley', 'Black pepper', 'Blackgram', 'Brinjal', 'Cabbage', 'Cardamom', 'Cashewnut', 'Castor seed', 'Coriander', 'Cotton(lint)', 'Cowpea(Lobia)', 'Drum Stick', 'Dry chillies', 'Dry ginger', 'Garlic', 'Ginger', 'Gram', 'Grapes', 'Groundnut', 'Guar seed', 'Horse-gram', 'Jowar', 'Jute', 'Khesari', 'Lemon', 'Linseed', 'Maize', 'Mango', 'Masoor', 'Mesta', 'Moong(Green Gram)', 'Moth', 'Niger seed', 'Oilseeds total', 'Onion', 'Orange', 'Other  Rabi pulses', 'Other Fresh Fruits', 'Other Kharif pulses', 'Other Vegetables', 'Paddy', 'Papaya', 'Peas & beans (Pulses)', 'Pineapple', 'Pome Granet', 'Potato', 'Ragi', 'Rapeseed &Mustard', 'Rice', 'Safflower', 'Sannhamp', 'Sapota', 'Sesamum', 'Small millets', 'Soyabean', 'Sugarcane', 'Sunflower', 'Sweet potato', 'Tapioca', 'Tobacco', 'Tomato', 'Turmeric', 'Urad', 'Wheat', 'other oilseeds']


# Yield prediction model training using random forest

In [ ]:

# ============================================================
# CROPSENSE AI - YIELD PREDICTION MODEL TRAINING
# Self-contained Google Colab cell
# ============================================================

import os
import json
import warnings
import numpy as np
import pandas as pd
import joblib
import sklearn

from google.colab import drive
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    median_absolute_error,
    r2_score
)

warnings.filterwarnings("ignore")

# ------------------------------------------------------------
# 1. GOOGLE DRIVE
# ------------------------------------------------------------
drive.mount("/content/drive")

FOLDER = "/content/drive/MyDrive/yield prediction"
os.makedirs(FOLDER, exist_ok=True)

INPUT_FILE = os.path.join(
    FOLDER, "crop_production_124_with_weather_soil.csv"
)

if not os.path.exists(INPUT_FILE):
    raise FileNotFoundError(
        f"Dataset not found: {INPUT_FILE}\n"
        "Upload the original CSV to this Google Drive folder."
    )

df = pd.read_csv(INPUT_FILE)

required = [
    "State_Name", "District_Name", "Crop_Year", "Season",
    "Crop", "Area", "Production", "temperature",
    "humidity", "rainfall_mm", "soil_type"
]

missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing required columns: {missing}")

print("Original dataset shape:", df.shape)

# ------------------------------------------------------------
# 2. CLEAN TARGET DATA
# ------------------------------------------------------------
# Remove Coconut because its production units may be inconsistent.
# Keep Sugarcane records, but investigate extreme values separately.

df = df[
    ~df["Crop"].astype(str).str.strip().str.casefold().eq("coconut")
].copy()

for col in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].notna()
    & df["Production"].ge(0)
].copy()

# Historical target
df["yield_target"] = df["Production"] / df["Area"]

# Train on log-transformed yield
df["log_yield_target"] = np.log1p(df["yield_target"])

df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["log_yield_target"].notna()].copy()

print("Rows after target cleaning:", len(df))
print("Crop classes after cleaning:", df["Crop"].nunique())

# ------------------------------------------------------------
# 3. TIME-BASED SPLIT
# ------------------------------------------------------------
train_df = df[df["Crop_Year"].between(1997, 2012)].copy()
val_df = df[df["Crop_Year"].eq(2013)].copy()
test_df = df[df["Crop_Year"].eq(2014)].copy()

common_crops = (
    set(train_df["Crop"].dropna().unique())
    & set(val_df["Crop"].dropna().unique())
    & set(test_df["Crop"].dropna().unique())
)

train_df = train_df[train_df["Crop"].isin(common_crops)].copy()
val_df = val_df[val_df["Crop"].isin(common_crops)].copy()
test_df = test_df[test_df["Crop"].isin(common_crops)].copy()

if min(len(train_df), len(val_df), len(test_df)) == 0:
    raise ValueError(
        "A split is empty. Check the year coverage in your dataset."
    )

print("\nTIME-BASED SPLIT")
print("Train (1997-2012):", len(train_df))
print("Validation (2013):", len(val_df))
print("Test (2014):", len(test_df))
print("Common crops:", len(common_crops))

# ------------------------------------------------------------
# 4. MODEL INPUT FEATURES
# ------------------------------------------------------------
# Crop_Year is intentionally excluded in this first version.
# A Random Forest cannot reliably extrapolate a year trend to 2026.
# We will test it separately if needed.

categorical_features = [
    "State_Name",
    "District_Name",
    "Crop",
    "Season",
    "soil_type"
]

numerical_features = [
    "temperature",
    "humidity",
    "rainfall_mm"
]

features = categorical_features + numerical_features
target = "log_yield_target"

X_train = train_df[features].copy()
y_train = train_df[target].copy()

X_val = val_df[features].copy()
y_val = val_df[target].copy()

X_test = test_df[features].copy()
y_test = test_df[target].copy()

# ------------------------------------------------------------
# 5. PREPROCESSING
# ------------------------------------------------------------
numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(
        handle_unknown="ignore",
        min_frequency=10
    ))
])

preprocessor = ColumnTransformer([
    ("categorical", categorical_pipeline, categorical_features),
    ("numerical", numeric_pipeline, numerical_features)
])

# ------------------------------------------------------------
# 6. TRAIN RANDOM FOREST
# ------------------------------------------------------------
model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", RandomForestRegressor(
        n_estimators=200,
        min_samples_leaf=2,
        max_features=0.8,
        random_state=42,
        n_jobs=-1
    ))
])

print("\nTraining Random Forest...")
model.fit(X_train, y_train)
print("Training completed!")

# ------------------------------------------------------------
# 7. EVALUATION
# ------------------------------------------------------------
def evaluate(name, X, y):
    pred_log = model.predict(X)

    actual = np.expm1(y.to_numpy())
    predicted = np.maximum(0, np.expm1(pred_log))

    return {
        "Dataset": name,
        "Log_MAE": mean_absolute_error(y, pred_log),
        "Log_RMSE": np.sqrt(mean_squared_error(y, pred_log)),
        "Yield_MAE": mean_absolute_error(actual, predicted),
        "Yield_RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "Yield_Median_AE": median_absolute_error(actual, predicted),
        "Yield_R2": r2_score(actual, predicted)
    }, actual, predicted

val_metrics, _, _ = evaluate(
    "Validation_2013", X_val, y_val
)
test_metrics, actual_test, predicted_test = evaluate(
    "Test_2014", X_test, y_test
)

metrics = pd.DataFrame([val_metrics, test_metrics])

print("\nMODEL RESULTS")
print(metrics.round(4).to_string(index=False))

# ------------------------------------------------------------
# 8. SAVE MODEL, METRICS AND TEST PREDICTIONS
# ------------------------------------------------------------
MODEL_FILE = os.path.join(
    FOLDER, "cropsense_yield_random_forest.joblib"
)
METRICS_FILE = os.path.join(
    FOLDER, "cropsense_yield_metrics.csv"
)
PREDICTIONS_FILE = os.path.join(
    FOLDER, "cropsense_yield_test_predictions.csv"
)
INFO_FILE = os.path.join(
    FOLDER, "cropsense_yield_model_info.json"
)

joblib.dump(model, MODEL_FILE)
metrics.to_csv(METRICS_FILE, index=False)

predictions = test_df[
    ["State_Name", "District_Name", "Crop_Year", "Season", "Crop"]
].copy()

predictions["actual_yield_ratio"] = actual_test
predictions["predicted_yield_ratio"] = predicted_test
predictions.to_csv(PREDICTIONS_FILE, index=False)

info = {
    "model": "RandomForestRegressor",
    "features": features,
    "target": "log_yield_target",
    "training_years": "1997-2012",
    "validation_year": 2013,
    "test_year": 2014,
    "common_crop_count": int(len(common_crops)),
    "training_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "area_used_as_feature": False,
    "production_used_as_feature": False,
    "sklearn_version": sklearn.__version__
}

with open(INFO_FILE, "w") as f:
    json.dump(info, f, indent=2)

print("\nFILES SAVED TO GOOGLE DRIVE")
print(MODEL_FILE)
print(METRICS_FILE)
print(PREDICTIONS_FILE)
print(INFO_FILE)
print("\nTraining pipeline finished.")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Original dataset shape: (246091, 11)
Rows after target cleaning: 240403
Crop classes after cleaning: 123

TIME-BASED SPLIT
Train (1997-2012): 212081
Validation (2013): 13277
Test (2014): 10615
Common crops: 69

Training Random Forest...
Training completed!

MODEL RESULTS
        Dataset  Log_MAE  Log_RMSE  Yield_MAE  Yield_RMSE  Yield_Median_AE    Yield_R2
Validation_2013   0.1687    0.3243    56.0021   1811.3792           0.2535 -19620.6028
      Test_2014   0.1652    0.3007    26.6709   1043.4311           0.2503  -6476.5573

FILES SAVED TO GOOGLE DRIVE
/content/drive/MyDrive/yield prediction/cropsense_yield_random_forest.joblib
/content/drive/MyDrive/yield prediction/cropsense_yield_metrics.csv
/content/drive/MyDrive/yield prediction/cropsense_yield_test_predictions.csv
/content/drive/MyDrive/yield prediction/cropsense_yield_model_info.json

Training pipel

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import r2_score, mean_absolute_error

# Actual and predicted yields on the original scale
actual = test_df["yield_target"].to_numpy()
predicted = np.maximum(
    0,
    np.expm1(model.predict(test_df[features]))
)

print("=" * 60)
print("TEST DATA: YIELD DISTRIBUTION")
print("=" * 60)

print("\nActual yield percentiles:")
print(pd.Series(actual).describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
))

print("\nPredicted yield percentiles:")
print(pd.Series(predicted).describe(
    percentiles=[0.50, 0.90, 0.95, 0.99, 0.999]
))

print("\nLog-scale R²:", round(
    r2_score(
        test_df["log_yield_target"],
        model.predict(test_df[features])
    ), 4
))

# Identify the records with the largest actual yields
review = test_df[
    [
        "State_Name", "District_Name", "Crop_Year",
        "Season", "Crop", "Area", "Production",
        "yield_target"
    ]
].copy()

review["predicted_yield"] = predicted
review["absolute_error"] = np.abs(
    review["yield_target"] - review["predicted_yield"]
)

print("\nTOP 20 LARGEST ACTUAL YIELDS")
print(
    review.sort_values("yield_target", ascending=False)
    .head(20)
    .to_string(index=False)
)

print("\nTOP 20 LARGEST PREDICTION ERRORS")
print(
    review.sort_values("absolute_error", ascending=False)
    .head(20)
    .to_string(index=False)
)

# Save diagnostic records to Google Drive
output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "cropsense_yield_extreme_records_review.csv"
)

review.sort_values(
    "absolute_error", ascending=False
).to_csv(output_path, index=False)

print("\nDiagnostic review saved to:", output_path)


TEST DATA: YIELD DISTRIBUTION

Actual yield percentiles:
count    10615.000000
mean         5.088229
std         12.965181
min          0.000000
50%          1.125000
90%         12.301949
95%         21.859996
99%         73.192964
99.9%      113.007665
max        179.866515
dtype: float64

Predicted yield percentiles:
count    10615.000000
mean        30.029545
std       1044.847410
min          0.008046
50%          1.001998
90%         10.521366
95%         19.868796
99%         66.978421
99.9%      119.123111
max      58828.863047
dtype: float64

Log-scale R²: 0.8966

TOP 20 LARGEST ACTUAL YIELDS
    State_Name  District_Name  Crop_Year      Season      Crop     Area  Production  yield_target  predicted_yield  absolute_error
   West Bengal MEDINIPUR WEST       2014 Whole Year  Sugarcane   4390.0    789614.0    179.866515        85.930774       93.935741
   West Bengal        HOOGHLY       2014 Whole Year  Sugarcane    830.0    149289.0    179.866265        72.444819      107.42144

In [ ]:

import numpy as np
import pandas as pd

# Inspect raw model predictions before converting out of log space
raw_log_predictions = model.predict(test_df[features])

check = test_df[
    [
        "State_Name", "District_Name", "Crop",
        "Season", "Area", "Production",
        "yield_target", "log_yield_target"
    ]
].copy()

check["predicted_log_yield"] = raw_log_predictions
check["predicted_yield"] = np.expm1(raw_log_predictions)

# Show the highest predicted values
print("TOP 20 PREDICTED YIELDS")
print(
    check.sort_values("predicted_yield", ascending=False)
    .head(20)
    .to_string(index=False)
)

print("\nPUNJAB SUGARCANE RECORDS")
print(
    check[
        check["State_Name"].astype(str).str.contains(
            "Punjab", case=False, na=False
        )
        & check["Crop"].astype(str).str.contains(
            "Sugarcane", case=False, na=False
        )
    ][
        [
            "District_Name", "Area", "Production",
            "yield_target", "log_yield_target",
            "predicted_log_yield", "predicted_yield"
        ]
    ].to_string(index=False)
)

print("\nLargest predicted log value:",
      check["predicted_log_yield"].max())
print("Largest actual log target:",
      check["log_yield_target"].max())


TOP 20 PREDICTED YIELDS
    State_Name  District_Name      Crop      Season    Area  Production  yield_target  log_yield_target  predicted_log_yield  predicted_yield
        Punjab     HOSHIARPUR Sugarcane Whole Year  22000.0   1515000.0     68.863636          4.246545            10.982405     58828.863047
        Punjab      JALANDHAR Sugarcane Whole Year  11000.0    825000.0     75.000000          4.330733            10.981714     58788.214810
        Punjab    S.A.S NAGAR Sugarcane Whole Year   1000.0     64000.0     64.000000          4.174387            10.786930     48383.257760
        Punjab      GURDASPUR Sugarcane Whole Year  24000.0   1840000.0     76.666667          4.352426            10.183757     26468.735070
        Punjab       AMRITSAR Sugarcane Whole Year   4000.0    301000.0     75.250000          4.334017            10.146798     25508.286240
        Punjab     NAWANSHAHR Sugarcane Whole Year   6000.0    395000.0     65.833333          4.202202             9.965735

In [ ]:

import numpy as np
import pandas as pd

# Inspect the fitted Random Forest directly
rf = model.named_steps["regressor"]
preprocessor = model.named_steps["preprocessor"]

print("Number of trees:", len(rf.estimators_))
print("Training target log range:",
      float(y_train.min()), "to", float(y_train.max()))

print("\nMaximum predicted log yield by Punjab Sugarcane district:")

raw = model.predict(test_df[features])
check = test_df[
    ["State_Name", "District_Name", "Crop", "Season", "yield_target"]
].copy()
check["predicted_log"] = raw

print(
    check[
        check["State_Name"].astype(str).str.casefold().eq("punjab")
        & check["Crop"].astype(str).str.casefold().eq("sugarcane")
    ].sort_values("predicted_log", ascending=False).to_string(index=False)
)

# Check predictions from individual trees for the most extreme row
idx = np.argmax(raw)
X_extreme = test_df.iloc[[idx]][features]
X_encoded = preprocessor.transform(X_extreme)

tree_predictions = np.array([
    tree.predict(X_encoded)[0] for tree in rf.estimators_
])

print("\nMost extreme test record:")
print(test_df.iloc[idx][
    ["State_Name", "District_Name", "Crop", "Season", "yield_target"]
])

print("Model predicted log yield:", raw[idx])
print("Individual tree prediction percentiles:",
      np.percentile(tree_predictions, [0, 25, 50, 75, 100]))



Number of trees: 200
Training target log range: 0.0 to 11.385103457032141

Maximum predicted log yield by Punjab Sugarcane district:
State_Name   District_Name      Crop      Season  yield_target  predicted_log
    Punjab      HOSHIARPUR Sugarcane Whole Year      68.863636      10.982405
    Punjab       JALANDHAR Sugarcane Whole Year      75.000000      10.981714
    Punjab     S.A.S NAGAR Sugarcane Whole Year      64.000000      10.786930
    Punjab       GURDASPUR Sugarcane Whole Year      76.666667      10.183757
    Punjab        AMRITSAR Sugarcane Whole Year      75.250000      10.146798
    Punjab      NAWANSHAHR Sugarcane Whole Year      65.833333       9.965735
    Punjab        RUPNAGAR Sugarcane Whole Year      79.333333       9.943042
    Punjab       PATHANKOT Sugarcane Whole Year      63.250000       9.208525
    Punjab      KAPURTHALA Sugarcane Whole Year      78.000000       6.760861
    Punjab         PATIALA Sugarcane Whole Year      90.666667       4.850944
    Punja

In [ ]:

# Investigate extreme yield ratios in the training data

import numpy as np
import pandas as pd

# Show the largest training targets
columns_to_show = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production",
    "yield_target", "log_yield_target"
]

print("=" * 70)
print("TOP 30 EXTREME TRAINING YIELDS")
print("=" * 70)

print(
    train_df.sort_values("yield_target", ascending=False)[columns_to_show]
    .head(30)
    .to_string(index=False)
)

# Count records above selected thresholds
print("\n" + "=" * 70)
print("EXTREME TARGET COUNTS")
print("=" * 70)

for threshold in [100, 500, 1000, 5000, 10000]:
    print(
        f"Yield ratio > {threshold}:",
        int((train_df["yield_target"] > threshold).sum())
    )

# Check whether Punjab Sugarcane exists in the training data
print("\n" + "=" * 70)
print("PUNJAB SUGARCANE IN TRAINING")
print("=" * 70)

mask = (
    train_df["State_Name"].astype(str).str.strip().str.casefold().eq("punjab")
    & train_df["Crop"].astype(str).str.strip().str.casefold().eq("sugarcane")
)

punjab_sugarcane = train_df.loc[mask, columns_to_show]

print("Matching records:", len(punjab_sugarcane))

if len(punjab_sugarcane):
    print(punjab_sugarcane.to_string(index=False))

# Save a review copy without changing the training data
output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "extreme_yield_training_review.csv"
)

train_df.sort_values(
    "yield_target", ascending=False
).head(500).to_csv(output_path, index=False)

print("\nSaved review file:", output_path)


TOP 30 EXTREME TRAINING YIELDS
 State_Name   District_Name  Crop_Year      Season      Crop    Area  Production  yield_target  log_yield_target
     Punjab         PATIALA       2011 Whole Year  Sugarcane     3.0    264000.0  88000.000000         11.385103
     Punjab         SANGRUR       2011 Whole Year  Sugarcane     2.0    163000.0  81500.000000         11.308371
     Punjab        LUDHIANA       2011 Whole Year  Sugarcane     2.0    161000.0  80500.000000         11.296025
     Punjab FATEHGARH SAHIB       2011 Whole Year  Sugarcane     2.0    151000.0  75500.000000         11.231901
     Punjab        RUPNAGAR       2011 Whole Year  Sugarcane     3.0    224000.0  74666.666667         11.220802
     Punjab       GURDASPUR       2011 Whole Year  Sugarcane    20.0   1491000.0  74550.000000         11.219239
     Punjab       PATHANKOT       2011 Whole Year  Sugarcane     4.0    298000.0  74500.000000         11.218568
     Punjab      KAPURTHALA       2011 Whole Year  Sugarcane     

In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

raw_df = pd.read_csv(file_path)

# Select Punjab sugarcane records
mask = (
    raw_df["State_Name"].astype(str).str.strip().str.casefold().eq("punjab")
    & raw_df["Crop"].astype(str).str.strip().str.casefold().eq("sugarcane")
)

cols = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production"
]

punjab_sc = raw_df.loc[mask, cols].copy()

punjab_sc["Area"] = pd.to_numeric(punjab_sc["Area"], errors="coerce")
punjab_sc["Production"] = pd.to_numeric(
    punjab_sc["Production"], errors="coerce"
)

punjab_sc["yield_ratio"] = (
    punjab_sc["Production"] / punjab_sc["Area"].replace(0, np.nan)
)

print("Punjab sugarcane records by year:")
print(
    punjab_sc.groupby("Crop_Year").agg(
        records=("Crop", "size"),
        min_area=("Area", "min"),
        median_area=("Area", "median"),
        max_area=("Area", "max"),
        median_production=("Production", "median"),
        max_production=("Production", "max"),
        max_yield_ratio=("yield_ratio", "max")
    ).to_string()
)

print("\n2011 Punjab sugarcane records:")
print(
    punjab_sc[punjab_sc["Crop_Year"] == 2011]
    .sort_values("Area")[cols + ["yield_ratio"]]
    .to_string(index=False)
)

print("\nRecords with Area below 100:")
print(
    punjab_sc[punjab_sc["Area"] < 100]
    .sort_values(["Crop_Year", "Area"])
    .to_string(index=False)
)


Punjab sugarcane records by year:
           records  min_area  median_area  max_area  median_production  max_production  max_yield_ratio
Crop_Year                                                                                              
1997            16    1000.0       7500.0   19000.0           460000.0        980000.0        66.250000
1998            15    1000.0       5000.0   15000.0           330000.0        840000.0        66.000000
1999            15    1000.0       6000.0   16000.0           420000.0       1100000.0        71.250000
2000            16    1000.0       6000.0   18000.0           400000.0       1220000.0        73.333333
2001            17    1000.0       6000.0   23000.0           380000.0       1570000.0        87.500000
2002            17    1000.0       8000.0   25000.0           460000.0       1700000.0        70.000000
2003            16    1000.0       5500.0   22000.0           275000.0       1320000.0        66.666667
2004            15    1000.0  

In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path)

df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")

# Examine all 2011 records with suspiciously small area
suspect = df[
    (df["Crop_Year"] == 2011)
    & (df["Area"] > 0)
    & (df["Area"] < 100)
    & (df["Production"] > 0)
].copy()

suspect["yield_ratio"] = suspect["Production"] / suspect["Area"]

print("Total suspicious 2011 records:", len(suspect))

print("\nCounts by state and crop:")
print(
    suspect.groupby(["State_Name", "Crop"])
    .size()
    .sort_values(ascending=False)
    .to_string()
)

print("\nTop 50 suspicious records:")
cols = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production", "yield_ratio"
]

print(
    suspect.sort_values("yield_ratio", ascending=False)[cols]
    .head(50)
    .to_string(index=False)
)

output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "suspicious_area_records_2011.csv"
)
suspect.to_csv(output_path, index=False)
print("\nSaved:", output_path)


Total suspicious 2011 records: 3754

Counts by state and crop:
State_Name              Crop                   
Uttar Pradesh           Maize                      70
                        Onion                      70
                        Moong(Green Gram)          64
                        Guar seed                  59
                        Coriander                  49
                        Sweet potato               44
                        Dry chillies               40
Odisha                  Maize                      38
Uttar Pradesh           Garlic                     37
Karnataka               Peas & beans (Pulses)      36
Uttar Pradesh           Urad                       33
Karnataka               Dry chillies               32
Madhya Pradesh          Sannhamp                   32
Odisha                  Sesamum                    31
                        Urad                       30
Madhya Pradesh          Turmeric                   30
Tamil Nadu              R

In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path)

df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")
df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")

df = df[
    (df["Area"] > 0)
    & (df["Production"] > 0)
].copy()

df["yield_ratio"] = df["Production"] / df["Area"]

# 1. Count extreme ratios across the full dataset
print("=" * 70)
print("EXTREME YIELD RATIOS ACROSS ALL YEARS")
print("=" * 70)

for threshold in [100, 500, 1000, 5000, 10000]:
    count = (df["yield_ratio"] > threshold).sum()
    print(f"Yield ratio > {threshold:>5}: {count:,}")

# 2. Crops contributing most to extreme ratios
extreme = df[df["yield_ratio"] > 500].copy()

print("\n" + "=" * 70)
print("CROPS WITH MOST RATIOS ABOVE 500")
print("=" * 70)

print(
    extreme.groupby("Crop").agg(
        extreme_records=("yield_ratio", "size"),
        max_ratio=("yield_ratio", "max"),
        median_ratio=("yield_ratio", "median"),
        first_year=("Crop_Year", "min"),
        last_year=("Crop_Year", "max")
    )
    .sort_values("extreme_records", ascending=False)
    .head(30)
    .to_string()
)

# 3. Check year-wise patterns for the most extreme crops
print("\n" + "=" * 70)
print("YEAR-WISE DETAILS FOR EXTREME RECORDS")
print("=" * 70)

print(
    extreme.groupby(["Crop", "Crop_Year"]).agg(
        records=("yield_ratio", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        max_ratio=("yield_ratio", "max")
    )
    .sort_values("max_ratio", ascending=False)
    .head(50)
    .to_string()
)

# Save the complete extreme-record review
output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "extreme_yield_ratios_full_dataset.csv"
)

extreme.sort_values(
    "yield_ratio", ascending=False
).to_csv(output_path, index=False)

print("\nSaved:", output_path)


EXTREME YIELD RATIOS ACROSS ALL YEARS
Yield ratio >   100: 1,628
Yield ratio >   500: 983
Yield ratio >  1000: 969
Yield ratio >  5000: 761
Yield ratio > 10000: 287

CROPS WITH MOST RATIOS ABOVE 500
           extreme_records  max_ratio  median_ratio  first_year  last_year
Crop                                                                      
Coconut                932    38800.0   7701.895623        1998       2014
Sugarcane               43    88000.0   1187.887938        1997       2011
Maize                    6     1494.0   1134.750000        1997       1997
Cashewnut                1     9801.0   9801.000000        2008       2008
Onion                    1     2150.0   2150.000000        2008       2008

YEAR-WISE DETAILS FOR EXTREME RECORDS
                     records  median_area  median_production     max_ratio
Crop      Crop_Year                                                       
Sugarcane 2011            15        3.000           221000.0  88000.000000
Coconut   20

In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path)

df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")

df = df[
    (df["Area"] > 0)
    & (df["Production"] > 0)
].copy()

df["yield_ratio"] = df["Production"] / df["Area"]

# Exclude coconut only from this review, not by modifying the source file
review = df[
    (df["yield_ratio"] > 500)
    & (~df["Crop"].astype(str).str.strip().str.casefold().eq("coconut"))
].copy()

cols = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production", "yield_ratio"
]

print("Extreme non-coconut records:", len(review))
print("\nCounts by crop:")
print(review.groupby("Crop").size().sort_values(ascending=False))

print("\nAll extreme sugarcane records:")
print(
    review[review["Crop"].astype(str).str.strip().str.casefold().eq("sugarcane")]
    .sort_values("yield_ratio", ascending=False)[cols]
    .to_string(index=False)
)

output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "extreme_non_coconut_yield_review.csv"
)

review.sort_values("yield_ratio", ascending=False).to_csv(
    output_path, index=False
)

print("\nSaved review file:", output_path)


Extreme non-coconut records: 51

Counts by crop:
Crop
Sugarcane    43
Maize         6
Cashewnut     1
Onion         1
dtype: int64

All extreme sugarcane records:
State_Name   District_Name  Crop_Year      Season      Crop    Area  Production  yield_ratio
    Punjab         PATIALA       2011 Whole Year  Sugarcane     3.0    264000.0 88000.000000
    Punjab         SANGRUR       2011 Whole Year  Sugarcane     2.0    163000.0 81500.000000
    Punjab        LUDHIANA       2011 Whole Year  Sugarcane     2.0    161000.0 80500.000000
    Punjab FATEHGARH SAHIB       2011 Whole Year  Sugarcane     2.0    151000.0 75500.000000
    Punjab        RUPNAGAR       2011 Whole Year  Sugarcane     3.0    224000.0 74666.666667
    Punjab       GURDASPUR       2011 Whole Year  Sugarcane    20.0   1491000.0 74550.000000
    Punjab       PATHANKOT       2011 Whole Year  Sugarcane     4.0    298000.0 74500.000000
    Punjab      KAPURTHALA       2011 Whole Year  Sugarcane     3.0    221000.0 73666.666667


In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path)

df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")
df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")

# Filter Tamil Nadu sugarcane
mask = (
    df["State_Name"].astype(str).str.strip().str.casefold().eq("tamil nadu")
    & df["Crop"].astype(str).str.strip().str.casefold().eq("sugarcane")
)

sc = df.loc[mask].copy()
sc = sc[(sc["Area"] > 0) & (sc["Production"] > 0)]
sc["yield_ratio"] = sc["Production"] / sc["Area"]

print("=" * 70)
print("TAMIL NADU SUGARCANE: YEAR-WISE COMPARISON")
print("=" * 70)

print(
    sc.groupby("Crop_Year").agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield_ratio=("yield_ratio", "median"),
        maximum_yield_ratio=("yield_ratio", "max")
    ).round(3).to_string()
)

print("\n" + "=" * 70)
print("TAMIL NADU SUGARCANE: 1997 RECORDS")
print("=" * 70)

cols = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production", "yield_ratio"
]

print(
    sc[sc["Crop_Year"] == 1997]
    .sort_values("yield_ratio", ascending=False)[cols]
    .to_string(index=False)
)

print("\n" + "=" * 70)
print("TAMIL NADU SUGARCANE: 1998 RECORDS")
print("=" * 70)

print(
    sc[sc["Crop_Year"] == 1998]
    .sort_values("District_Name")[cols]
    .to_string(index=False)
)

# Save this comparison for source verification
output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "tamil_nadu_sugarcane_year_comparison.csv"
)

sc[cols].sort_values(
    ["Crop_Year", "District_Name"]
).to_csv(output_path, index=False)

print("\nSaved:", output_path)


TAMIL NADU SUGARCANE: YEAR-WISE COMPARISON
           records  median_area  median_production  median_yield_ratio  maximum_yield_ratio
Crop_Year                                                                                  
1997            27       6826.0          9900000.0            1077.778             2247.533
1998            27      10081.0          1432090.0             122.732              303.485
1999            27       8926.0          1069224.0             107.179              133.613
2000            27       7163.0           860112.0             107.143              133.613
2001            27       7437.0           805700.0             100.187              158.573
2002            27       5993.0           488160.0              91.838              135.687
2003            26       4787.5           454119.0             101.678              125.810
2004            28       4327.5           444558.0             110.074              137.919
2005            28       7337.0      

In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path)

for col in ["Area", "Production", "Crop_Year"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

mask = (
    df["State_Name"].astype(str).str.strip().str.casefold().eq("tamil nadu")
    & df["Crop"].astype(str).str.strip().str.casefold().eq("sugarcane")
    & df["Season"].astype(str).str.strip().str.casefold().eq("whole year")
    & df["Crop_Year"].isin([1997, 1998])
)

sc = df.loc[mask].copy()
sc = sc[(sc["Area"] > 0) & (sc["Production"] > 0)]
sc["yield_ratio"] = sc["Production"] / sc["Area"]

wide = sc.pivot_table(
    index="District_Name",
    columns="Crop_Year",
    values=["Area", "Production", "yield_ratio"],
    aggfunc="first"
)

wide.columns = [f"{metric}_{year}" for metric, year in wide.columns]
wide = wide.reset_index()

# Factor is diagnostic, not an approved correction
wide["yield_ratio_factor_1997_vs_1998"] = (
    wide["yield_ratio_1997"] / wide["yield_ratio_1998"]
)

wide["production_1997_if_1998_ratio_applied"] = (
    wide["Area_1997"] * wide["yield_ratio_1998"]
)

print(
    wide.sort_values(
        "yield_ratio_factor_1997_vs_1998", ascending=False
    ).round(3).to_string(index=False)
)

print("\nMedian 1997/1998 yield-ratio factor:")
print(wide["yield_ratio_factor_1997_vs_1998"].median())

output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "tn_sugarcane_1997_1998_district_comparison.csv"
)

wide.to_csv(output_path, index=False)
print("\nSaved:", output_path)


  District_Name  Area_1997  Area_1998  Production_1997  Production_1998  yield_ratio_1997  yield_ratio_1998  yield_ratio_factor_1997_vs_1998  production_1997_if_1998_ratio_applied
    THIRUVALLUR     5964.0     8284.0       11811000.0        1432090.0          1980.382           172.874                           11.456                            1031021.820
     COIMBATORE    10756.0    13193.0       12272000.0        1467460.0          1140.945           111.230                           10.258                            1196392.008
     VILLUPURAM    33175.0    44737.0       37158000.0        4995600.0          1120.060           111.666                           10.030                            3704518.184
          KARUR     5961.0    10193.0        7081000.0        1251010.0          1187.888           122.732                            9.679                             731607.045
     DHARMAPURI    18938.0    27836.0       18100000.0        2772300.0           955.750           

In [ ]:
import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path, low_memory=False)

# Convert numeric columns safely
for col in ["Area", "Production", "Crop_Year"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

print("=" * 55)
print("DATASET AUDIT — STEP 1")
print("=" * 55)

print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())

# Check missing values
print("\nMissing values:")
print(df[["Area", "Production", "Crop_Year"]].isna().sum())

# Valid records for yield calculation
valid = df[
    df["Area"].gt(0) &
    df["Production"].gt(0)
].copy()

valid["calculated_yield"] = (
    valid["Production"] / valid["Area"]
)

print("\nValid area and production records:", len(valid))
print("Records with zero or negative area:",
      (df["Area"] <= 0).sum())
print("Records with zero or negative production:",
      (df["Production"] <= 0).sum())

print("\nCalculated yield distribution:")
print(
    valid["calculated_yield"]
    .describe(percentiles=[0.50, 0.90, 0.95, 0.99])
    .round(3)
)

print("\nExtreme yield ratios:")
for threshold in [100, 500, 1000, 5000, 10000]:
    count = (valid["calculated_yield"] > threshold).sum()
    print(f"Yield > {threshold:>5}: {count:,}")

print("\nTop 20 highest yield ratios:")
cols = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production",
    "calculated_yield"
]

print(
    valid.nlargest(20, "calculated_yield")[cols]
    .to_string(index=False)
)

DATASET AUDIT — STEP 1
Dataset shape: (246091, 11)
Columns: ['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production', 'temperature', 'humidity', 'rainfall_mm', 'soil_type']

Missing values:
Area             0
Production    3730
Crop_Year        0
dtype: int64

Valid area and production records: 238838
Records with zero or negative area: 0
Records with zero or negative production: 3523

Calculated yield distribution:
count    238838.000
mean         42.263
std         823.565
min           0.000
50%           1.000
90%          10.000
95%          22.856
99%          83.545
max       88000.000
Name: calculated_yield, dtype: float64

Extreme yield ratios:
Yield >   100: 1,628
Yield >   500: 983
Yield >  1000: 969
Yield >  5000: 761
Yield > 10000: 287

Top 20 highest yield ratios:
    State_Name   District_Name  Crop_Year      Season      Crop    Area  Production  calculated_yield
        Punjab         PATIALA       2011 Whole Year  Sugarcane     3.0    264000.

In [ ]:
import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path, low_memory=False)

df["Crop_Year"] = pd.to_numeric(
    df["Crop_Year"], errors="coerce"
)
df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(
    df["Production"], errors="coerce"
)

mask = (
    df["State_Name"].astype(str).str.strip().str.casefold().eq("punjab")
    & df["Crop"].astype(str).str.strip().str.casefold().eq("sugarcane")
    & df["Season"].astype(str).str.strip().str.casefold().eq("whole year")
    & df["Crop_Year"].between(2009, 2013)
)

check = df.loc[mask].copy()

check["calculated_yield"] = np.where(
    (check["Area"] > 0) & (check["Production"] > 0),
    check["Production"] / check["Area"],
    np.nan
)

print("Records by year:")
print(check.groupby("Crop_Year").size())

print("\nYear-wise summary:")
print(
    check.groupby("Crop_Year").agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max")
    ).round(3).to_string()
)

print("\nDistrict-level records:")
print(
    check[
        [
            "Crop_Year", "District_Name", "Area",
            "Production", "calculated_yield"
        ]
    ]
    .sort_values(["Crop_Year", "District_Name"])
    .to_string(index=False)
)

Records by year:
Crop_Year
2009    13
2011    15
2012    16
2013    16
dtype: int64

Year-wise summary:
           records  median_area  median_production  median_yield  max_yield
Crop_Year                                                                  
2009            13        200.0            15000.0        60.000     80.000
2011            15          3.0           221000.0     73666.667  88000.000
2012            16       2500.0           205500.0        70.750     92.000
2013            16       3000.0           253500.0        78.300     93.667

District-level records:
 Crop_Year   District_Name    Area  Production  calculated_yield
      2009        AMRITSAR   300.0     18000.0         60.000000
      2009 FATEHGARH SAHIB   200.0     15000.0         75.000000
      2009       FIROZEPUR   100.0      6000.0         60.000000
      2009       GURDASPUR  1900.0    124000.0         65.263158
      2009      HOSHIARPUR  1400.0     76000.0         54.285714
      2009       JALANDHA

In [ ]:
import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path, low_memory=False)

for col in ["Crop_Year", "Area", "Production"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Punjab records from 2011
punjab_2011 = df[
    df["State_Name"].astype(str).str.strip().str.casefold().eq("punjab")
    & df["Crop_Year"].eq(2011)
].copy()

punjab_2011["calculated_yield"] = np.where(
    (punjab_2011["Area"] > 0)
    & (punjab_2011["Production"] > 0),
    punjab_2011["Production"] / punjab_2011["Area"],
    np.nan
)

print("Punjab records in 2011:", len(punjab_2011))

print("\nSummary by crop:")
summary = (
    punjab_2011.groupby("Crop")
    .agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max")
    )
    .sort_values("median_yield", ascending=False)
)

print(summary.round(3).to_string())

print("\nAll Punjab 2011 records with yield ratio > 500:")
print(
    punjab_2011[punjab_2011["calculated_yield"] > 500][
        [
            "District_Name", "Crop", "Season",
            "Area", "Production", "calculated_yield"
        ]
    ].sort_values("calculated_yield", ascending=False).to_string(index=False)
)

Punjab records in 2011: 162

Summary by crop:
                       records  median_area  median_production  median_yield  max_yield
Crop                                                                                   
Sugarcane                   15          3.0           221000.0     73666.667  88000.000
Wheat                       20     173500.0           884000.0         5.015      5.495
Barley                       8       1000.0             4500.0         4.000      5.000
Maize                       11       6000.0            24000.0         4.000      6.000
Rice                        20     115500.0           446000.0         3.789      4.371
Cotton(lint)                11      11000.0            44000.0         3.000      4.000
Groundnut                    2       8500.0            14500.0         1.375      1.750
Peas & beans (Pulses)        8       2500.0             3000.0         1.143      1.333
Arhar/Tur                   10       2000.0             2000.0         1.0

In [ ]:
import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path, low_memory=False)

for col in ["Crop_Year", "Area", "Production"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

coconut = df[
    df["Crop"].astype(str).str.strip().str.casefold().eq("coconut")
].copy()

coconut["calculated_yield"] = np.where(
    (coconut["Area"] > 0) & (coconut["Production"] > 0),
    coconut["Production"] / coconut["Area"],
    np.nan
)

print("Total coconut records:", len(coconut))

print("\nCoconut records by state:")
print(
    coconut.groupby("State_Name").agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max")
    ).sort_values("max_yield", ascending=False)
    .round(3)
    .to_string()
)

print("\nCoconut records with calculated yield > 500:")
print(
    coconut[coconut["calculated_yield"] > 500][
        [
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "Production", "calculated_yield"
        ]
    ]
    .sort_values("calculated_yield", ascending=False)
    .head(30)
    .to_string(index=False)
)

Total coconut records: 1985

Coconut records by state:
                             records  median_area  median_production  median_yield  max_yield
State_Name                                                                                   
West Bengal                      216       858.50       1.844370e+04        14.225  38800.000
Andhra Pradesh                   169      1829.00       1.720927e+07     10282.390  33089.005
Assam                            400       556.00       2.059500e+06      5257.042  28334.156
Tamil Nadu                       295      6825.00       7.405000e+04        10.293  24414.676
Puducherry                        57       318.00       4.040000e+06     11336.127  19792.982
Telangana                         52        12.50       1.276440e+05     11105.057  14606.000
Kerala                           252     58475.00       3.778770e+08      6306.291  11051.716
Goa                               12     12738.00       5.553500e+07      4930.285   4995.104
Andam

In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path, low_memory=False)

for col in ["Crop_Year", "Area", "Production"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

coconut = df[
    df["Crop"].astype(str).str.strip().str.casefold().eq("coconut")
].copy()

coconut = coconut[
    (coconut["Area"] > 0) &
    (coconut["Production"] > 0)
].copy()

coconut["calculated_yield"] = (
    coconut["Production"] / coconut["Area"]
)

# Year-wise summary
print("===== COCONUT: YEAR-WISE SUMMARY =====")

print(
    coconut.groupby("Crop_Year").agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max"),
        records_over_500=(
            "calculated_yield",
            lambda s: (s > 500).sum()
        )
    ).round(2).to_string()
)

# State and year combinations with suspiciously high ratios
print("\n===== STATE-YEAR COMBINATIONS =====")

summary = (
    coconut.groupby(["State_Name", "Crop_Year"])
    .agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max"),
        records_over_500=(
            "calculated_yield",
            lambda s: (s > 500).sum()
        )
    )
    .reset_index()
)

print(
    summary[summary["max_yield"] > 500]
    .sort_values("max_yield", ascending=False)
    .head(40)
    .round(2)
    .to_string(index=False)
)


===== COCONUT: YEAR-WISE SUMMARY =====
           records  median_area  median_production  median_yield  max_yield  records_over_500
Crop_Year                                                                                    
1997            57       1040.0             8766.0          5.61      20.98                 0
1998            83       1174.0            75720.0         17.33    9908.82                37
1999            86       1161.5           180378.0         17.77   16500.00                41
2000           108       1172.5           493405.0       3752.31   18000.00                62
2001           107       1264.0           530400.0       3583.78   26270.83                61
2002           134       2105.5           158800.0         15.44   14432.86                61
2003           133       2101.0           129580.0         16.10   16906.30                60
2004           138       2177.5           354800.0         48.24   28334.16                63
2005           114   

In [ ]:

# Inspect a small sample of coconut records from the original dataset.
# This does not modify or delete anything.

cols = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Area", "Production"
]

sample = coconut[cols + ["calculated_yield"]].copy()

print("===== COCONUT RECORDS: 1997 =====")
print(
    sample[sample["Crop_Year"] == 1997]
    .head(15)
    .to_string(index=False)
)

print("\n===== COCONUT RECORDS: 2014 =====")
print(
    sample[sample["Crop_Year"] == 2014]
    .head(15)
    .to_string(index=False)
)

print("\n===== COCONUT PRODUCTION MAGNITUDES =====")
print(
    sample.groupby("Crop_Year")["Production"]
    .agg(["count", "median", "min", "max"])
    .round(2)
    .to_string()
)


===== COCONUT RECORDS: 1997 =====
State_Name District_Name  Crop_Year      Season   Area  Production  calculated_yield
     Assam       BARPETA       1997 Whole Year  1122.0      8093.0          7.213012
     Assam    BONGAIGAON       1997 Whole Year   468.0      1156.0          2.470085
     Assam        CACHAR       1997 Whole Year  1400.0      2386.0          1.704286
     Assam       DARRANG       1997 Whole Year   864.0      8766.0         10.145833
     Assam       DHEMAJI       1997 Whole Year   114.0       267.0          2.342105
     Assam        DHUBRI       1997 Whole Year   675.0      1325.0          1.962963
     Assam     DIBRUGARH       1997 Whole Year    65.0       261.0          4.015385
     Assam    DIMA HASAO       1997 Whole Year    40.0       259.0          6.475000
     Assam      GOALPARA       1997 Whole Year   589.0      2480.0          4.210526
     Assam      GOLAGHAT       1997 Whole Year   545.0      3056.0          5.607339
     Assam    HAILAKANDI       

In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

df = pd.read_csv(file_path, low_memory=False)

for col in ["Crop_Year", "Area", "Production"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Keep only records where a ratio can be calculated.
valid = df[
    (df["Area"] > 0) &
    (df["Production"] > 0)
].copy()

valid["calculated_yield"] = (
    valid["Production"] / valid["Area"]
)

# Audit flags only — these are NOT automatic exclusions.
valid["yield_over_100"] = valid["calculated_yield"] > 100
valid["yield_over_500"] = valid["calculated_yield"] > 500

summary = (
    valid.groupby("Crop")
    .agg(
        records=("calculated_yield", "size"),
        median_yield=("calculated_yield", "median"),
        p95_yield=(
            "calculated_yield",
            lambda s: s.quantile(0.95)
        ),
        max_yield=("calculated_yield", "max"),
        records_over_100=("yield_over_100", "sum"),
        records_over_500=("yield_over_500", "sum")
    )
    .sort_values(
        ["records_over_500", "max_yield"],
        ascending=False
    )
)

print("Total valid yield records:", len(valid))
print("\n===== TOP 30 CROPS TO INVESTIGATE =====")
print(summary.head(30).round(3).to_string())

print("\n===== TOTAL EXTREME-RATIO COUNTS =====")
print("Yield > 100:", int((valid["calculated_yield"] > 100).sum()))
print("Yield > 500:", int((valid["calculated_yield"] > 500).sum()))
print("Yield > 1000:", int((valid["calculated_yield"] > 1000).sum()))


Total valid yield records: 238838

===== TOP 30 CROPS TO INVESTIGATE =====
                    records  median_yield  p95_yield  max_yield  records_over_100  records_over_500
Crop                                                                                               
Coconut                1957        43.120  13817.583  38800.000               958               932
Sugarcane              7814        53.287    107.106  88000.000               607                43
Maize                 13723         1.800      4.819   1494.000                 6                 6
Cashewnut              1065         0.397      1.030   9801.000                 1                 1
Onion                  6979        10.300     25.751   2150.000                 3                 1
Cotton(lint)           4350         1.383      4.000    300.000                 7                 0
Sesamum                8724         0.333      0.885    235.000                 1                 0
Rice                  150

In [ ]:

# Continue the audit — this cell does not change the original dataset.

# Reuse df and valid from your previous cell.
# If you restarted Colab, rerun the previous audit cell first.

print("===== SUGARCANE: HIGHEST YIELD RATIOS =====")

sugarcane = valid[
    valid["Crop"].astype(str).str.strip().str.casefold()
    .eq("sugarcane")
].copy()

print(
    sugarcane.sort_values("calculated_yield", ascending=False)[
        [
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "Production", "calculated_yield"
        ]
    ].head(25).to_string(index=False)
)

print("\n===== OTHER CROPS: RECORDS ABOVE 500 =====")

other_crops = valid[
    (valid["calculated_yield"] > 500) &
    (~valid["Crop"].astype(str).str.strip().str.casefold()
     .isin(["coconut", "sugarcane"]))
].copy()

print(
    other_crops[
        [
            "Crop", "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "Production", "calculated_yield"
        ]
    ].sort_values("calculated_yield", ascending=False)
    .to_string(index=False)
)

print("\n===== SUGARCANE: YEAR-WISE SUMMARY =====")

print(
    sugarcane.groupby("Crop_Year").agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield=("calculated_yield", "median"),
        max_yield=("calculated_yield", "max"),
        records_over_500=(
            "calculated_yield",
            lambda s: (s > 500).sum()
        )
    ).round(2).to_string()
)


===== SUGARCANE: HIGHEST YIELD RATIOS =====
State_Name   District_Name  Crop_Year      Season    Area  Production  calculated_yield
    Punjab         PATIALA       2011 Whole Year      3.0    264000.0      88000.000000
    Punjab         SANGRUR       2011 Whole Year      2.0    163000.0      81500.000000
    Punjab        LUDHIANA       2011 Whole Year      2.0    161000.0      80500.000000
    Punjab FATEHGARH SAHIB       2011 Whole Year      2.0    151000.0      75500.000000
    Punjab        RUPNAGAR       2011 Whole Year      3.0    224000.0      74666.666667
    Punjab       GURDASPUR       2011 Whole Year     20.0   1491000.0      74550.000000
    Punjab       PATHANKOT       2011 Whole Year      4.0    298000.0      74500.000000
    Punjab      KAPURTHALA       2011 Whole Year      3.0    221000.0      73666.666667
    Punjab         FAZILKA       2011 Whole Year      1.0     73000.0      73000.000000
    Punjab      NAWANSHAHR       2011 Whole Year      6.0    407000.0      6

In [ ]:

# Create a review file only. The original dataset remains unchanged.

import os

review = valid.copy()

review["audit_reason"] = ""

review.loc[
    review["calculated_yield"] > 500,
    "audit_reason"
] += "Yield ratio above 500; verify source and units. "

review.loc[
    (review["Area"] <= 5) &
    (review["calculated_yield"] > 100),
    "audit_reason"
] += "Very small reported area with high ratio. "

review.loc[
    (
        review["Crop"].astype(str).str.strip().str.casefold()
        .eq("coconut")
    ),
    "audit_reason"
] += "Coconut production unit needs verification. "

review.loc[
    (
        review["Crop"].astype(str).str.strip().str.casefold()
        .eq("sugarcane")
    ) &
    (
        ((review["State_Name"] == "Punjab") &
         (review["Crop_Year"] == 2011)) |
        ((review["State_Name"] == "Tamil Nadu") &
         (review["Crop_Year"] == 1997))
    ),
    "audit_reason"
] += "Known sugarcane anomaly group; verify source. "

review = review[review["audit_reason"] != ""].copy()

output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "yield_anomaly_review.csv"
)

review.to_csv(output_path, index=False)

print("Review records saved:", len(review))
print("File:", output_path)
print("\nFlagged records by crop:")
print(
    review.groupby("Crop").size()
    .sort_values(ascending=False)
    .head(20)
    .to_string()
)


Review records saved: 2033
File: /content/drive/MyDrive/yield prediction/yield_anomaly_review.csv

Flagged records by crop:
Crop
Coconut         1957
Sugarcane         66
Maize              6
Cotton(lint)       1
Cashewnut          1
Onion              1
Papaya             1


In [ ]:

import pandas as pd
import numpy as np
import os

input_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_cleaned.csv"
)

df = pd.read_csv(input_path, low_memory=False)

# Convert columns used to calculate yield.
for col in ["Area", "Production"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Calculate yield only where Area and Production are positive.
valid = (
    (df["Area"] > 0) &
    (df["Production"] > 0)
)

yield_values = pd.Series(np.nan, index=df.index, dtype=float)
yield_values.loc[valid] = (
    df.loc[valid, "Production"] / df.loc[valid, "Area"]
)

# Remove only the four individually flagged records.
# Match crop, state, district, year, season, area and production
# using the review file to identify the exact records.
review_path = (
    "/content/drive/MyDrive/yield prediction/"
    "yield_anomaly_review.csv"
)

review_df = pd.read_csv(review_path, low_memory=False)

target_crops = [
    "Cotton(lint)", "Cashewnut", "Onion", "Papaya"
]

targets = review_df[
    review_df["Crop"].isin(target_crops)
].copy()

# Keep only records flagged for a high ratio or very small area.
# The user requested the four individual records, not all records
# from these crop classes.
targets = targets[
    targets["audit_reason"].str.contains(
        "Yield ratio above 500|Very small reported area",
        case=False,
        na=False,
        regex=True
    )
]

key_cols = [
    "Crop", "State_Name", "District_Name",
    "Crop_Year", "Season", "Area", "Production"
]

# Normalize text for reliable matching.
def normalize_text(series):
    return series.astype("string").str.strip().str.casefold()

remove_mask = pd.Series(False, index=df.index)

for _, target in targets.iterrows():
    match = pd.Series(True, index=df.index)

    for col in key_cols:
        if col in ["Area", "Production", "Crop_Year"]:
            source_values = pd.to_numeric(df[col], errors="coerce")
            target_value = pd.to_numeric(
                pd.Series([target[col]]), errors="coerce"
            ).iloc[0]
            match &= source_values.eq(target_value)
        else:
            match &= normalize_text(df[col]).eq(
                str(target[col]).strip().casefold()
            )

    remove_mask |= match

cleaned_df = df.loc[~remove_mask].copy()

print("Original rows:", len(df))
print("Rows removed:", int(remove_mask.sum()))
print("Cleaned rows:", len(cleaned_df))

if int(remove_mask.sum()) != len(targets):
    print(
        "WARNING: The number of matched rows differs from the "
        "number of target records. Review matches before training."
    )
else:
    cleaned_df.to_csv(output_path, index=False)
    print("Cleaned dataset saved:", output_path)


Original rows: 246091
Rows removed: 4
Cleaned rows: 246087
Cleaned dataset saved: /content/drive/MyDrive/yield prediction/crop_production_124_cleaned.csv


In [ ]:

import pandas as pd
import numpy as np
import os

# ============================================
# 1. FILE PATHS
# ============================================

input_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_with_weather_soil.csv"
)

output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_clean.csv"
)

review_output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "remaining_yield_records_to_review.csv"
)

# ============================================
# 2. LOAD ORIGINAL DATA
# ============================================

df = pd.read_csv(input_path, low_memory=False)

print("Original dataset shape:", df.shape)

# Normalize crop, state and district text for matching.
crop_norm = df["Crop"].astype("string").str.strip().str.casefold()
state_norm = df["State_Name"].astype("string").str.strip().str.casefold()

year = pd.to_numeric(df["Crop_Year"], errors="coerce")
area = pd.to_numeric(df["Area"], errors="coerce")
production = pd.to_numeric(df["Production"], errors="coerce")

# ============================================
# 3. CALCULATE YIELD FOR AUDIT PURPOSES
# ============================================

valid_yield = (area > 0) & (production > 0)

calculated_yield = pd.Series(
    np.nan, index=df.index, dtype=float
)

calculated_yield.loc[valid_yield] = (
    production.loc[valid_yield] / area.loc[valid_yield]
)

# ============================================
# 4. EXCLUDE COCONUT
# ============================================

coconut_mask = crop_norm.eq("coconut").fillna(False)

# ============================================
# 5. EXCLUDE THE 15 PUNJAB SUGARCANE RECORDS
#    FROM 2011
# ============================================

sugarcane_mask = crop_norm.eq("sugarcane").fillna(False)

punjab_mask = state_norm.eq("punjab").fillna(False)

punjab_sugarcane_2011 = (
    sugarcane_mask &
    punjab_mask &
    year.eq(2011)
)

# ============================================
# 6. BUILD TRAINING DATASET
# ============================================

exclude_mask = coconut_mask | punjab_sugarcane_2011

training_df = df.loc[~exclude_mask].copy()

# ============================================
# 7. FLAG OTHER EXTREME RECORDS FOR REVIEW
# ============================================

remaining_indices = training_df.index

remaining_yield = calculated_yield.loc[remaining_indices]

remaining_crop = (
    training_df["Crop"]
    .astype("string")
    .str.strip()
    .str.casefold()
)

# Flag high ratios in the remaining dataset.
maize_extreme = (
    remaining_crop.eq("maize").fillna(False) &
    remaining_yield.gt(500)
)

sugarcane_tamil_nadu_1997 = (
    remaining_crop.eq("sugarcane").fillna(False) &
    training_df["State_Name"].astype("string")
    .str.strip().str.casefold().eq("tamil nadu").fillna(False) &
    pd.to_numeric(
        training_df["Crop_Year"], errors="coerce"
    ).eq(1997)
)

other_extreme = (
    remaining_yield.gt(500) |
    maize_extreme |
    sugarcane_tamil_nadu_1997
)

review_df = training_df.loc[other_extreme].copy()

review_df["calculated_yield"] = remaining_yield.loc[
    review_df.index
]

review_df["review_reason"] = np.where(
    review_df["Crop"].astype("string").str.strip()
    .str.casefold().eq("maize"),
    "Extreme maize yield ratio: verify source and units",
    np.where(
        review_df["State_Name"].astype("string").str.strip()
        .str.casefold().eq("tamil nadu") &
        pd.to_numeric(
            review_df["Crop_Year"], errors="coerce"
        ).eq(1997),
        "Tamil Nadu sugarcane 1997: verify source and units",
        "Extreme yield ratio: verify source and units"
    )
)

# ============================================
# 8. SAVE SEPARATE FILES
# ============================================

training_df.to_csv(output_path, index=False)
review_df.to_csv(review_output_path, index=False)

# ============================================
# 9. REPORT RESULTS
# ============================================

print("\n========== CLEANING SUMMARY ==========")
print("Original rows:", len(df))
print("Coconut rows excluded:", int(coconut_mask.sum()))
print(
    "Punjab Sugarcane 2011 rows excluded:",
    int(punjab_sugarcane_2011.sum())
)
print("Total rows excluded:", int(exclude_mask.sum()))
print("Training dataset rows:", len(training_df))
print("Records flagged for further review:", len(review_df))

print("\nTraining dataset shape:", training_df.shape)
print("\nFiles saved:")
print(output_path)
print(review_output_path)

print("\nRemaining crops:", training_df["Crop"].nunique())
print("Missing Production:", training_df["Production"].isna().sum())


Original dataset shape: (246091, 11)

========== CLEANING SUMMARY ==========
Original rows: 246091
Coconut rows excluded: 1985
Punjab Sugarcane 2011 rows excluded: 15
Total rows excluded: 2000
Training dataset rows: 244091
Records flagged for further review: 36

Training dataset shape: (244091, 11)

Files saved:
/content/drive/MyDrive/yield prediction/crop_production_124_training_clean.csv
/content/drive/MyDrive/yield prediction/remaining_yield_records_to_review.csv

Remaining crops: 123
Missing Production: 3703


In [ ]:

import pandas as pd
import numpy as np

input_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_clean.csv"
)

train_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

missing_path = (
    "/content/drive/MyDrive/yield prediction/"
    "records_missing_production.csv"
)

df = pd.read_csv(input_path, low_memory=False)

df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(
    df["Production"], errors="coerce"
)

# Keep rows that have a usable target and positive area.
usable = (
    df["Production"].notna() &
    df["Area"].notna() &
    (df["Area"] > 0) &
    (df["Production"] > 0)
)

training = df.loc[usable].copy()
missing = df.loc[df["Production"].isna()].copy()

# Derive the target; do not use it as a feature.
training["yield_per_area"] = (
    training["Production"] / training["Area"]
)

training.to_csv(train_path, index=False)
missing.to_csv(missing_path, index=False)

print("Original rows:", len(df))
print("Rows ready for supervised training:", len(training))
print("Rows missing production:", int(df["Production"].isna().sum()))
print("Rows excluded for missing/invalid target or area:",
      len(df) - len(training))
print("Training file:", train_path)
print("Missing-production file:", missing_path)
print("\nYield summary:")
print(training["yield_per_area"].describe().round(3))


Original rows: 244091
Rows ready for supervised training: 236866
Rows missing production: 3703
Rows excluded for missing/invalid target or area: 7225
Training file: /content/drive/MyDrive/yield prediction/crop_production_124_training_ready.csv
Missing-production file: /content/drive/MyDrive/yield prediction/records_missing_production.csv

Yield summary:
count    236866.000
mean          4.709
std          27.764
min           0.000
25%           0.529
50%           1.000
75%           2.333
max        9801.000
Name: yield_per_area, dtype: float64


In [ ]:

import pandas as pd
import numpy as np

input_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_clean.csv"
)

train_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

missing_path = (
    "/content/drive/MyDrive/yield prediction/"
    "records_missing_production.csv"
)

df = pd.read_csv(input_path, low_memory=False)

df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(
    df["Production"], errors="coerce"
)

# Keep rows that have a usable target and positive area.
usable = (
    df["Production"].notna() &
    df["Area"].notna() &
    (df["Area"] > 0) &
    (df["Production"] > 0)
)

training = df.loc[usable].copy()
missing = df.loc[df["Production"].isna()].copy()

# Derive the target; do not use it as a feature.
training["yield_per_area"] = (
    training["Production"] / training["Area"]
)

training.to_csv(train_path, index=False)
missing.to_csv(missing_path, index=False)

print("Original rows:", len(df))
print("Rows ready for supervised training:", len(training))
print("Rows missing production:", int(df["Production"].isna().sum()))
print("Rows excluded for missing/invalid target or area:",
      len(df) - len(training))
print("Training file:", train_path)
print("Missing-production file:", missing_path)
print("\nYield summary:")
print(training["yield_per_area"].describe().round(3))


Original rows: 244091
Rows ready for supervised training: 236866
Rows missing production: 3703
Rows excluded for missing/invalid target or area: 7225
Training file: /content/drive/MyDrive/yield prediction/crop_production_124_training_ready.csv
Missing-production file: /content/drive/MyDrive/yield prediction/records_missing_production.csv

Yield summary:
count    236866.000
mean          4.709
std          27.764
min           0.000
25%           0.529
50%           1.000
75%           2.333
max        9801.000
Name: yield_per_area, dtype: float64


In [ ]:

import pandas as pd
import numpy as np

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

df = pd.read_csv(file_path, low_memory=False)

df["yield_per_area"] = pd.to_numeric(
    df["yield_per_area"], errors="coerce"
)

print("===== YIELD DISTRIBUTION =====")
print(
    df["yield_per_area"]
    .quantile([0.50, 0.90, 0.95, 0.99, 0.995, 0.999, 1.0])
    .to_string()
)

print("\n===== TOP 30 HIGHEST YIELD RECORDS =====")
cols = [
    "Crop", "State_Name", "District_Name",
    "Crop_Year", "Season", "Area",
    "Production", "yield_per_area"
]

print(
    df.sort_values("yield_per_area", ascending=False)[cols]
    .head(30)
    .to_string(index=False)
)

print("\n===== COUNTS ABOVE THRESHOLDS =====")
for threshold in [100, 500, 1000]:
    print(
        f"Yield > {threshold}:",
        int((df["yield_per_area"] > threshold).sum())
    )

print("\n===== CROPS WITH HIGHEST MAXIMUM YIELD =====")
print(
    df.groupby("Crop")["yield_per_area"]
    .agg(["count", "median", "max"])
    .sort_values("max", ascending=False)
    .head(20)
    .round(3)
    .to_string()
)


===== YIELD DISTRIBUTION =====
0.500       1.000000
0.900       9.400000
0.950      20.888889
0.990      68.480678
0.995      87.681139
0.999     119.369935
1.000    9801.000000

===== TOP 30 HIGHEST YIELD RECORDS =====
     Crop  State_Name   District_Name  Crop_Year      Season    Area  Production  yield_per_area
Cashewnut  Tamil Nadu      PERAMBALUR       2008 Whole Year      1.0      9801.0     9801.000000
Sugarcane  Tamil Nadu         MADURAI       1997 Whole Year   8108.0  18223000.0     2247.533300
    Onion     Haryana          KARNAL       2008 Whole Year      2.0      4300.0     2150.000000
Sugarcane  Tamil Nadu     THIRUVALLUR       1997 Whole Year   5964.0  11811000.0     1980.382294
    Maize Maharashtra          NASHIK       1997 Autumn          1.0      1494.0     1494.000000
Sugarcane  Tamil Nadu     KANCHIPURAM       1997 Whole Year   6735.0   9896000.0     1469.339272
Sugarcane  Tamil Nadu        NAMAKKAL       1997 Whole Year   9415.0  13608000.0     1445.353160
Suga

In [ ]:

import pandas as pd
import numpy as np

input_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

output_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready_v2.csv"
)

df = pd.read_csv(input_path, low_memory=False)

df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(
    df["Production"], errors="coerce"
)

crop = df["Crop"].astype(str).str.strip().str.casefold()

yield_value = df["Production"] / df["Area"].where(df["Area"] > 0)

# Remove extreme records for these crops only.
# Do not remove entire crop classes.
selected_crops = crop.isin(["maize", "cashewnut", "onion"])
extreme_mask = selected_crops & (yield_value > 500)

print("Maize/Cashewnut/Onion extreme records to remove:",
      int(extreme_mask.sum()))

# Create a separate file of the removed records for traceability.
removed = df.loc[extreme_mask].copy()
removed["calculated_yield"] = yield_value.loc[extreme_mask]

removed_path = (
    "/content/drive/MyDrive/yield prediction/"
    "excluded_extreme_maize_cashew_onion.csv"
)
removed.to_csv(removed_path, index=False)

revised = df.loc[~extreme_mask].copy()
revised.to_csv(output_path, index=False)

print("Rows before:", len(df))
print("Rows removed:", int(extreme_mask.sum()))
print("Rows after:", len(revised))
print("Revised dataset:", output_path)
print("Removed records saved separately:", removed_path)

print("\nTop 10 remaining yield values:")
print(
    revised.nlargest(10, "yield_per_area")[
        ["Crop", "State_Name", "District_Name",
         "Crop_Year", "Area", "Production", "yield_per_area"]
    ].to_string(index=False)
)


Maize/Cashewnut/Onion extreme records to remove: 8
Rows before: 236866
Rows removed: 8
Rows after: 236858
Revised dataset: /content/drive/MyDrive/yield prediction/crop_production_124_training_ready_v2.csv
Removed records saved separately: /content/drive/MyDrive/yield prediction/excluded_extreme_maize_cashew_onion.csv

Top 10 remaining yield values:
     Crop State_Name   District_Name  Crop_Year    Area  Production  yield_per_area
Sugarcane Tamil Nadu         MADURAI       1997  8108.0  18223000.0     2247.533300
Sugarcane Tamil Nadu     THIRUVALLUR       1997  5964.0  11811000.0     1980.382294
Sugarcane Tamil Nadu     KANCHIPURAM       1997  6735.0   9896000.0     1469.339272
Sugarcane Tamil Nadu        NAMAKKAL       1997  9415.0  13608000.0     1445.353160
Sugarcane Tamil Nadu TIRUCHIRAPPALLI       1997  4315.0   5806000.0     1345.538818
Sugarcane Tamil Nadu       CUDDALORE       1997 23826.0  28322000.0     1188.701419
Sugarcane Tamil Nadu           KARUR       1997  5961.0   708

In [ ]:

import pandas as pd

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready_v2.csv"
)

df = pd.read_csv(file_path, low_memory=False)

df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")
df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")

crop = df["Crop"].astype(str).str.strip().str.casefold()
state = df["State_Name"].astype(str).str.strip().str.casefold()

sugarcane_tn = df[
    crop.eq("sugarcane") &
    state.eq("tamil nadu") &
    df["Crop_Year"].between(1996, 1999)
].copy()

sugarcane_tn["yield_per_area"] = (
    sugarcane_tn["Production"] / sugarcane_tn["Area"]
)

print("Records:", len(sugarcane_tn))

print(
    sugarcane_tn[
        ["State_Name", "District_Name", "Crop_Year",
         "Area", "Production", "yield_per_area"]
    ]
    .sort_values(["District_Name", "Crop_Year"])
    .to_string(index=False)
)

print("\n===== YEAR-WISE SUMMARY =====")
print(
    sugarcane_tn.groupby("Crop_Year").agg(
        records=("Crop", "size"),
        median_area=("Area", "median"),
        median_production=("Production", "median"),
        median_yield=("yield_per_area", "median"),
        max_yield=("yield_per_area", "max")
    ).round(2).to_string()
)


Records: 81
State_Name   District_Name  Crop_Year    Area  Production  yield_per_area
Tamil Nadu      COIMBATORE       1997 10756.0  12272000.0     1140.944589
Tamil Nadu      COIMBATORE       1998 13193.0   1467460.0      111.230198
Tamil Nadu      COIMBATORE       1999 11939.0   1195037.0      100.095234
Tamil Nadu       CUDDALORE       1997 23826.0  28322000.0     1188.701419
Tamil Nadu       CUDDALORE       1998 32654.0   4401890.0      134.804006
Tamil Nadu       CUDDALORE       1999 38500.0   4196384.0      108.996987
Tamil Nadu      DHARMAPURI       1997 18938.0  18100000.0      955.750343
Tamil Nadu      DHARMAPURI       1998 27836.0   2772300.0       99.594051
Tamil Nadu      DHARMAPURI       1999 25826.0   2311813.0       89.514946
Tamil Nadu        DINDIGUL       1997  6826.0   5749000.0      842.220920
Tamil Nadu        DINDIGUL       1998  7051.0    886010.0      125.657354
Tamil Nadu        DINDIGUL       1999  6655.0    799113.0      120.077085
Tamil Nadu           ERODE

In [ ]:

import pandas as pd

input_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready_v2.csv"
)

df = pd.read_csv(input_path, low_memory=False)

# Normalize text for reliable filtering
crop = df["Crop"].astype(str).str.strip().str.casefold()
state = df["State_Name"].astype(str).str.strip().str.casefold()
year = pd.to_numeric(df["Crop_Year"], errors="coerce")

# Isolate the suspicious records for review
review_mask = (
    crop.eq("sugarcane")
    & state.eq("tamil nadu")
    & year.eq(1997)
)

review_records = df.loc[review_mask].copy()
remaining_records = df.loc[~review_mask].copy()

# Save both outputs separately; preserve the current dataset
review_path = (
    "/content/drive/MyDrive/yield prediction/"
    "review_tamil_nadu_sugarcane_1997.csv"
)

review_dataset_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready_review_v1.csv"
)

review_records.to_csv(review_path, index=False)
remaining_records.to_csv(review_dataset_path, index=False)

print("Records flagged:", len(review_records))
print("Original training-ready rows:", len(df))
print("Rows in review dataset:", len(remaining_records))
print("Review records saved to:", review_path)
print("Separate review dataset saved to:", review_dataset_path)


Records flagged: 27
Original training-ready rows: 236858
Rows in review dataset: 236831
Review records saved to: /content/drive/MyDrive/yield prediction/review_tamil_nadu_sugarcane_1997.csv
Separate review dataset saved to: /content/drive/MyDrive/yield prediction/crop_production_124_training_ready_review_v1.csv


# XGBoost model training

In [ ]:

# ============================================================
# CropSense AI: Complete Yield Prediction Pipeline
# Preprocessing + Train/Validation/Test + XGBoost
# ============================================================

!pip -q install xgboost scikit-learn joblib

import os
import numpy as np
import pandas as pd
import joblib

from google.colab import drive
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)
from xgboost import XGBRegressor

# ------------------------------------------------------------
# 1. Mount Google Drive and load data
# ------------------------------------------------------------

drive.mount("/content/drive")

file_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

df = pd.read_csv(file_path, low_memory=False)
df.columns = df.columns.str.strip()

print("=" * 60)
print("DATASET LOADED")
print("=" * 60)
print("Original shape:", df.shape)
print("Columns:", df.columns.tolist())

# ------------------------------------------------------------
# 2. Verify required columns
# ------------------------------------------------------------

required_columns = [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production",
    "temperature", "humidity", "rainfall_mm", "soil_type"
]

missing_columns = [
    c for c in required_columns if c not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Missing columns: {missing_columns}. "
        "Check your dataset's actual column names."
    )

# ------------------------------------------------------------
# 3. Data cleaning and target calculation
# ------------------------------------------------------------

numeric_columns = [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]

for col in numeric_columns:
    df[col] = pd.to_numeric(df[col], errors="coerce")

# Remove rows that cannot provide a valid supervised target
df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

# Target: production per unit area
df["yield_per_area"] = df["Production"] / df["Area"]

df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

print("\nRows after target validation:", len(df))
print("Unique crops:", df["Crop"].nunique())
print(
    "Year range:",
    int(df["Crop_Year"].min()),
    "to",
    int(df["Crop_Year"].max())
)

# ------------------------------------------------------------
# 4. Define features and target
# ------------------------------------------------------------

categorical_features = [
    "State_Name",
    "District_Name",
    "Season",
    "Crop",
    "soil_type"
]

numeric_features = [
    "Crop_Year",
    "temperature",
    "humidity",
    "rainfall_mm"
]

feature_columns = categorical_features + numeric_features

# IMPORTANT:
# Area and Production are NOT input features.
X = df[feature_columns].copy()
y = df["yield_per_area"].copy()

# Normalize categorical data; remaining missing values are
# handled inside the preprocessing pipeline.
for col in categorical_features:
    X[col] = (
        X[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

# ------------------------------------------------------------
# 5. Chronological train/validation/test split
# ------------------------------------------------------------
# Train: 1997–2011
# Validation: 2012–2013
# Test: 2014–2015
#
# This simulates predicting future years from historical data.

train_mask = df["Crop_Year"] <= 2011
val_mask = df["Crop_Year"].isin([2012, 2013])
test_mask = df["Crop_Year"].isin([2014, 2015])

if not train_mask.any() or not val_mask.any() or not test_mask.any():
    raise ValueError(
        "One or more splits are empty. Check the year range."
    )

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("\n" + "=" * 60)
print("TRAIN / VALIDATION / TEST SPLIT")
print("=" * 60)

for name, mask, target in [
    ("TRAIN", train_mask, y_train),
    ("VALIDATION", val_mask, y_val),
    ("TEST", test_mask, y_test)
]:
    years = df.loc[mask, "Crop_Year"]
    print(
        f"{name:12} | Rows: {len(target):,} | "
        f"Years: {int(years.min())}-{int(years.max())}"
    )

# ------------------------------------------------------------
# 6. Build preprocessing pipeline
# ------------------------------------------------------------

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features)
])

# ------------------------------------------------------------
# 7. Build XGBoost regression model
# ------------------------------------------------------------

xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", xgb)
])

# ------------------------------------------------------------
# 8. Train
# ------------------------------------------------------------

print("\nTraining XGBoost model...")
model.fit(X_train, y_train)
print("Training completed!")

# ------------------------------------------------------------
# 9. Evaluate validation and test sets
# ------------------------------------------------------------

def evaluate_model(name, actual, predicted):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    r2 = r2_score(actual, predicted)
    median_ae = np.median(
        np.abs(np.asarray(actual) - np.asarray(predicted))
    )

    print(f"\n{name}")
    print("-" * 40)
    print(f"MAE:        {mae:.4f}")
    print(f"RMSE:       {rmse:.4f}")
    print(f"R²:         {r2:.4f}")
    print(f"Median AE:  {median_ae:.4f}")

    return {
        "Dataset": name,
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Median_AE": median_ae
    }

val_predictions = model.predict(X_val)
test_predictions = model.predict(X_test)

val_metrics = evaluate_model(
    "VALIDATION RESULTS (2012–2013)",
    y_val,
    val_predictions
)

test_metrics = evaluate_model(
    "FINAL HELD-OUT TEST RESULTS (2014–2015)",
    y_test,
    test_predictions
)

metrics_df = pd.DataFrame([val_metrics, test_metrics])

# ------------------------------------------------------------
# 10. Save model, metrics and test predictions
# ------------------------------------------------------------

output_dir = "/content/drive/MyDrive/yield prediction"
os.makedirs(output_dir, exist_ok=True)

model_path = os.path.join(
    output_dir,
    "cropsense_xgboost_yield_pipeline.joblib"
)

metrics_path = os.path.join(
    output_dir,
    "cropsense_yield_evaluation_metrics.csv"
)

predictions_path = os.path.join(
    output_dir,
    "cropsense_yield_test_predictions.csv"
)

joblib.dump(model, model_path)
metrics_df.to_csv(metrics_path, index=False)

test_results = df.loc[test_mask, [
    "State_Name", "District_Name", "Crop_Year",
    "Season", "Crop", "Area", "Production",
    "yield_per_area"
]].copy()

test_results["predicted_yield"] = test_predictions
test_results["predicted_production"] = (
    test_results["predicted_yield"] * test_results["Area"]
)

test_results.to_csv(predictions_path, index=False)

print("\n" + "=" * 60)
print("SAVED OUTPUTS")
print("=" * 60)
print("Model:", model_path)
print("Metrics:", metrics_path)
print("Test predictions:", predictions_path)

print("\nSample test predictions:")
print(
    test_results[[
        "Crop", "Crop_Year", "Area",
        "yield_per_area", "predicted_yield",
        "Production", "predicted_production"
    ]].head(10).round(3).to_string(index=False)
)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DATASET LOADED
Original shape: (236866, 12)
Columns: ['State_Name', 'District_Name', 'Crop_Year', 'Season', 'Crop', 'Area', 'Production', 'temperature', 'humidity', 'rainfall_mm', 'soil_type', 'yield_per_area']

Rows after target validation: 236866
Unique crops: 104
Year range: 1997 to 2015

TRAIN / VALIDATION / TEST SPLIT
TRAIN        | Rows: 199,107 | Years: 1997-2011
VALIDATION   | Rows: 26,448 | Years: 2012-2013
TEST         | Rows: 11,311 | Years: 2014-2015

Training XGBoost model...
Training completed!

VALIDATION RESULTS (2012–2013)
----------------------------------------
MAE:        1.5811
RMSE:       5.0798
R²:         0.8473
Median AE:  0.5053

FINAL HELD-OUT TEST RESULTS (2014–2015)
----------------------------------------
MAE:        1.4705
RMSE:       4.5478
R²:         0.8756
Median AE:  0.4899

SAVED OUTPUTS
Model: /content/drive/MyDrive/yield

In [ ]:

import numpy as np
import pandas as pd
from sklearn.metrics import (
    mean_absolute_error,
    mean_squared_error,
    r2_score
)

path = (
    "/content/drive/MyDrive/yield prediction/"
    "cropsense_yield_test_predictions.csv"
)

results = pd.read_csv(path)

actual = results["yield_per_area"]
predicted = results["predicted_yield"]

# 1. Overall prediction checks
print("=" * 60)
print("TEST PREDICTION QUALITY AUDIT")
print("=" * 60)

print("Test records:", len(results))
print("Negative predicted yields:", int((predicted < 0).sum()))
print(
    "Negative prediction percentage:",
    round((predicted < 0).mean() * 100, 3), "%"
)
print("Actual yields <= 0:", int((actual <= 0).sum()))

# 2. Largest absolute prediction errors
results["absolute_error"] = np.abs(actual - predicted)

print("\n===== 20 LARGEST ABSOLUTE ERRORS =====")
print(
    results.sort_values("absolute_error", ascending=False)[[
        "State_Name", "District_Name", "Crop_Year",
        "Season", "Crop", "Area",
        "yield_per_area", "predicted_yield",
        "absolute_error"
    ]].head(20).round(4).to_string(index=False)
)

# 3. Crop-wise evaluation
crop_metrics = results.groupby("Crop").apply(
    lambda group: pd.Series({
        "records": len(group),
        "actual_median_yield": group["yield_per_area"].median(),
        "predicted_median_yield": group["predicted_yield"].median(),
        "MAE": mean_absolute_error(
            group["yield_per_area"], group["predicted_yield"]
        ),
        "RMSE": np.sqrt(mean_squared_error(
            group["yield_per_area"], group["predicted_yield"]
        )),
        "negative_predictions": int(
            (group["predicted_yield"] < 0).sum()
        )
    }),
    include_groups=False
).reset_index()

print("\n===== 20 CROPS WITH HIGHEST MAE =====")
print(
    crop_metrics.sort_values("MAE", ascending=False)
    .head(20).round(4).to_string(index=False)
)

# Save the audit report
out_path = (
    "/content/drive/MyDrive/yield prediction/"
    "cropsense_yield_cropwise_audit.csv"
)
crop_metrics.to_csv(out_path, index=False)

print("\nCrop-wise audit saved to:", out_path)


TEST PREDICTION QUALITY AUDIT
Test records: 11311
Negative predicted yields: 39
Negative prediction percentage: 0.345 %
Actual yields <= 0: 0

===== 20 LARGEST ABSOLUTE ERRORS =====
    State_Name  District_Name  Crop_Year      Season      Crop    Area  yield_per_area  predicted_yield  absolute_error
    Telangana      RANGAREDDI       2014 Kharif          Onion   24.00        114.1667          12.1718        101.9948
   West Bengal        HOOGHLY       2014 Whole Year  Sugarcane  830.00        179.8663          81.8572         98.0091
        Kerala         KANNUR       2014 Whole Year  Sugarcane    3.43         13.4111         109.9885         96.5774
   West Bengal MEDINIPUR WEST       2014 Whole Year  Sugarcane 4390.00        179.8665          97.7659         82.1006
        Kerala       PALAKKAD       2014 Whole Year  Sugarcane  371.81          8.4506          81.9844         73.5338
        Kerala         IDUKKI       2014 Whole Year  Sugarcane 1068.90         10.7700          82

# XGBoost Model

In [ ]:

import os
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# Load the same dataset used for the original baseline
path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)
df = pd.read_csv(path, low_memory=False)
df.columns = df.columns.str.strip()

for col in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

categorical = [
    "State_Name", "District_Name", "Season", "Crop", "soil_type"
]
numeric = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = categorical + numeric

X = df[features].copy()
y = df["yield_per_area"].copy()

for col in categorical:
    X[col] = (
        X[col].fillna("Unknown").astype(str).str.strip()
        .replace("", "Unknown")
    )

train_mask = df["Crop_Year"] <= 2011
val_mask = df["Crop_Year"].isin([2012, 2013])
test_mask = df["Crop_Year"].isin([2014, 2015])

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), numeric),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), categorical)
])

# Train on log-transformed target.
# Keep the pipeline so preprocessing is fitted on training data only.
log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=800,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
])

print("Training log-target XGBoost...")
log_model.fit(X_train, np.log1p(y_train))
print("Training completed!")

# Convert predictions back to original yield units
val_pred = np.expm1(log_model.predict(X_val))
test_pred_raw = np.expm1(log_model.predict(X_test))

# Enforce physically valid nonnegative yield estimates
val_pred = np.maximum(val_pred, 0)
test_pred = np.maximum(test_pred_raw, 0)

def report(name, actual, predicted):
    print(f"\n{name}")
    print("MAE:", round(mean_absolute_error(actual, predicted), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(actual, predicted)), 4))
    print("R²:", round(r2_score(actual, predicted), 4))
    print("Negative predictions:", int((predicted < 0).sum()))

report("LOG MODEL — VALIDATION", y_val, val_pred)
report("LOG MODEL — TEST", y_test, test_pred)

# Crop-level test comparison
test_comparison = df.loc[test_mask, [
    "Crop", "yield_per_area"
]].copy()
test_comparison["predicted_yield"] = test_pred
test_comparison["absolute_error"] = np.abs(
    test_comparison["yield_per_area"]
    - test_comparison["predicted_yield"]
)

crop_report = test_comparison.groupby("Crop").agg(
    records=("absolute_error", "size"),
    actual_median=("yield_per_area", "median"),
    predicted_median=("predicted_yield", "median"),
    MAE=("absolute_error", "mean")
).sort_values("MAE", ascending=False)

print("\nTOP 15 CROPS BY MAE — LOG MODEL")
print(crop_report.head(15).round(4).to_string())

out_dir = "/content/drive/MyDrive/yield prediction"
joblib.dump(
    log_model,
    os.path.join(out_dir, "cropsense_xgboost_log_yield_pipeline.joblib")
)

test_comparison.to_csv(
    os.path.join(out_dir, "cropsense_log_model_test_predictions.csv"),
    index=False
)

print("\nLog-target model and test predictions saved.")


Training log-target XGBoost...
Training completed!

LOG MODEL — VALIDATION
MAE: 1.4189
RMSE: 5.2798
R²: 0.835
Negative predictions: 0

LOG MODEL — TEST
MAE: 1.3732
RMSE: 4.832
R²: 0.8596
Negative predictions: 0

TOP 15 CROPS BY MAE — LOG MODEL
                    records  actual_median  predicted_median      MAE
Crop                                                                 
Grapes                    4        23.0000          6.300000  18.0596
Papaya                   44        24.2875         18.402201  17.2655
Cabbage                  19        12.3210         28.121599  17.0668
Sugarcane               375        60.0000         52.695301  13.7080
Tomato                   26        14.3766          4.102100  13.1284
Pome Granet               9        10.3636          1.164600   9.0964
Brinjal                  29        14.9286          8.026400   7.9519
other oilseeds           24         1.6851          1.440900   7.9291
Other Fresh Fruits        2        10.6448          2.83

In [ ]:

import pandas as pd
import numpy as np

data_path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

pred_path = (
    "/content/drive/MyDrive/yield prediction/"
    "cropsense_log_model_test_predictions.csv"
)

# Load original data and predictions
df = pd.read_csv(data_path, low_memory=False)
results = pd.read_csv(pred_path)

# Recreate the same filtering used during training
df.columns = df.columns.str.strip()

for col in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

# Recreate the chronological test split
test_mask = df["Crop_Year"].isin([2014, 2015])
test_df = df.loc[test_mask].copy().reset_index(drop=True)

# Confirm row counts before attaching predictions
print("Test rows in source dataset:", len(test_df))
print("Rows in prediction file:", len(results))

if len(test_df) != len(results):
    raise ValueError(
        "Row counts do not match. Do not merge by row position; "
        "inspect the prediction file and training filtering."
    )

# The saved predictions were generated from this same ordered test split.
results = results.reset_index(drop=True)

test_df["predicted_yield"] = results["predicted_yield"].to_numpy()
test_df["absolute_error"] = np.abs(
    test_df["yield_per_area"] - test_df["predicted_yield"]
)

# 1. Target distribution
print("\n===== TEST TARGET DISTRIBUTION =====")
print(
    test_df["yield_per_area"]
    .quantile([0.50, 0.90, 0.95, 0.99, 0.995, 0.999])
    .round(4).to_string()
)

# 2. Largest actual yields
print("\n===== 25 LARGEST ACTUAL YIELDS =====")
print(
    test_df.sort_values("yield_per_area", ascending=False)[[
        "State_Name", "District_Name", "Crop_Year",
        "Season", "Crop", "Area",
        "yield_per_area", "predicted_yield", "absolute_error"
    ]].head(25).round(4).to_string(index=False)
)

# 3. Sugarcane performance by year
print("\n===== SUGARCANE TEST PERFORMANCE BY YEAR =====")
sugarcane = test_df[
    test_df["Crop"].astype(str).str.strip().str.casefold()
    .eq("sugarcane")
]

if not sugarcane.empty:
    print(
        sugarcane.groupby("Crop_Year").agg(
            records=("absolute_error", "size"),
            median_actual_yield=("yield_per_area", "median"),
            median_predicted_yield=("predicted_yield", "median"),
            MAE=("absolute_error", "mean"),
            max_actual_yield=("yield_per_area", "max")
        ).round(4).to_string()
    )

# 4. All cabbage test records
print("\n===== CABBAGE TEST RECORDS =====")
cabbage = test_df[
    test_df["Crop"].astype(str).str.strip().str.casefold().eq("cabbage")
]

if cabbage.empty:
    print("No cabbage records found in the test set.")
else:
    print(
        cabbage[[
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "yield_per_area",
            "predicted_yield", "absolute_error"
        ]].round(4).to_string(index=False)
    )


Test rows in source dataset: 11311
Rows in prediction file: 11311

===== TEST TARGET DISTRIBUTION =====
0.500      1.1000
0.900     11.4839
0.950     21.3953
0.990     71.7908
0.995     88.1500
0.999    111.7330

===== 25 LARGEST ACTUAL YIELDS =====
    State_Name  District_Name  Crop_Year      Season      Crop     Area  yield_per_area  predicted_yield  absolute_error
   West Bengal MEDINIPUR WEST       2014 Whole Year  Sugarcane   4390.0        179.8665          80.7061         99.1604
   West Bengal        HOOGHLY       2014 Whole Year  Sugarcane    830.0        179.8663          72.1258        107.7405
    Puducherry    PONDICHERRY       2014 Kharif      Sugarcane   1921.0        165.0000          69.7319         95.2681
   West Bengal          NADIA       2014 Whole Year  Sugarcane   1754.0        141.5137          72.9714         68.5423
     Karnataka      DAVANGERE       2014 Whole Year  Sugarcane   9523.0        128.2500          91.5120         36.7381
     Karnataka         M

In [ ]:

import pandas as pd
import numpy as np

path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

df = pd.read_csv(path, low_memory=False)

for col in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]

df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

# Compare crops that showed large test errors
selected_crops = ["Sugarcane", "Cabbage", "Papaya", "Onion", "Tomato"]

for crop_name in selected_crops:
    crop_df = df[
        df["Crop"].astype(str).str.strip().str.casefold()
        == crop_name.casefold()
    ].copy()

    if crop_df.empty:
        continue

    train = crop_df[crop_df["Crop_Year"] <= 2011]
    validation = crop_df[crop_df["Crop_Year"].isin([2012, 2013])]
    test = crop_df[crop_df["Crop_Year"].isin([2014, 2015])]

    print("\n" + "=" * 65)
    print(f"CROP: {crop_name}")
    print("=" * 65)

    for split_name, split_df in [
        ("TRAIN (1997–2011)", train),
        ("VALIDATION (2012–2013)", validation),
        ("TEST (2014–2015)", test)
    ]:
        if split_df.empty:
            print(f"\n{split_name}: No records")
            continue


        median_yield = split_df["yield_per_area"].median()
        p90_yield = split_df["yield_per_area"].quantile(0.90)
        max_yield = split_df["yield_per_area"].max()

        print(
            "Yield median / p90 / max:",
            round(float(median_yield), 3),
            round(float(p90_yield), 3),
            round(float(max_yield), 3)
        )


        if crop_name == "Cabbage":
            print("\nState-wise yield summary:")
            print(
                split_df.groupby("State_Name")["yield_per_area"]
                .agg(["count", "median", "max"])
                .round(3).to_string()
            )



CROP: Sugarcane
Yield median / p90 / max: 52.203 94.502 2247.533
Yield median / p90 / max: 58.547 96.484 220.618
Yield median / p90 / max: 60.0 91.204 179.867

CROP: Cabbage
Yield median / p90 / max: 49.0 63.8 67.212

State-wise yield summary:
            count  median     max
State_Name                       
Manipur        18  12.200  13.200
Tamil Nadu     19  63.529  67.212
Yield median / p90 / max: 18.786 26.333 26.5

State-wise yield summary:
                count  median   max
State_Name                         
Andhra Pradesh     41  18.786  26.5
Yield median / p90 / max: 12.321 15.818 29.093

State-wise yield summary:
                count  median     max
State_Name                           
Andhra Pradesh     19  12.321  29.093

CROP: Papaya
Yield median / p90 / max: 15.082 90.815 130.5
Yield median / p90 / max: 21.298 108.931 134.143
Yield median / p90 / max: 24.287 78.785 107.112

CROP: Onion
Yield median / p90 / max: 10.127 21.83 2150.0
Yield median / p90 / max: 11.694 23

# CatBoost Model


In [ ]:

!pip -q install catboost

import os
import numpy as np
import pandas as pd
import joblib
from catboost import CatBoostRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Load dataset
path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)
df = pd.read_csv(path, low_memory=False)
df.columns = df.columns.str.strip()

for c in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

# Features and target
cat_cols = ["State_Name", "District_Name", "Season", "Crop", "soil_type"]
num_cols = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = cat_cols + num_cols

X = df[features].copy()
y = df["yield_per_area"].copy()

for c in cat_cols:
    X[c] = X[c].fillna("Unknown").astype(str).str.strip()
    X[c] = X[c].replace("", "Unknown")

# Same chronological split as previous models
train = df["Crop_Year"] <= 2011
val = df["Crop_Year"].isin([2012, 2013])
test = df["Crop_Year"].isin([2014, 2015])

X_train, y_train = X.loc[train], y.loc[train]
X_val, y_val = X.loc[val], y.loc[val]
X_test, y_test = X.loc[test], y.loc[test]

# CatBoost handles categorical columns directly
model = CatBoostRegressor(
    iterations=1000,
    depth=8,
    learning_rate=0.05,
    loss_function="RMSE",
    eval_metric="RMSE",
    random_seed=42,
    verbose=100,
    allow_writing_files=False
)

print("Training CatBoost...")
model.fit(
    X_train, y_train,
    cat_features=cat_cols,
    eval_set=(X_val, y_val),
    early_stopping_rounds=100,
    verbose=100
)

def evaluate(label, actual, predicted):
    print(f"\n{label}")
    print("MAE:", round(mean_absolute_error(actual, predicted), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(actual, predicted)), 4))
    print("R²:", round(r2_score(actual, predicted), 4))
    print("Negative predictions:", int((predicted < 0).sum()))

val_pred = model.predict(X_val)
test_pred = model.predict(X_test)

evaluate("CATBOOST VALIDATION (2012–2013)", y_val, val_pred)
evaluate("CATBOOST TEST (2014–2015)", y_test, test_pred)

# Save separately; do not overwrite previous models
out_dir = "/content/drive/MyDrive/yield prediction"
model_path = os.path.join(out_dir, "cropsense_catboost_yield.cbm")
joblib_path = os.path.join(out_dir, "cropsense_catboost_yield.joblib")

model.save_model(model_path)
joblib.dump(model, joblib_path)

print("\nSaved CatBoost model:", model_path)
print("Saved joblib model:", joblib_path)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 4.3 MB/s eta 0:00:00
Training CatBoost...
0:	learn: 29.3244639	test: 12.5517073	best: 12.5517073 (0)	total: 441ms	remaining: 7m 20s
100:	learn: 21.8053240	test: 6.2553114	best: 6.2510513 (94)	total: 20.8s	remaining: 3m 5s
200:	learn: 18.6124125	test: 6.3737164	best: 6.1378199 (154)	total: 45.2s	remaining: 2m 59s
Stopped by overfitting detector  (100 iterations wait)

bestTest = 6.137819886
bestIteration = 154

Shrink model to first 155 iterations.

CATBOOST VALIDATION (2012–2013)
MAE: 1.8119
RMSE: 6.1378
R²: 0.777
Negative predictions: 240

CATBOOST TEST (2014–2015)
MAE: 1.8071
RMSE: 5.8668
R²: 0.793
Negative predictions: 146

Saved CatBoost model: /content/drive/MyDrive/yield prediction/cropsense_catboost_yield.cbm
Saved joblib model: /content/drive/MyDrive/yield prediction/cropsense_catboost_yield.joblib


# XGBoost Model

In [ ]:

import os
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# 1. Load dataset
path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

df = pd.read_csv(path, low_memory=False)
df.columns = df.columns.str.strip()

for c in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

# 2. Features and target
cat_cols = [
    "State_Name", "District_Name", "Season", "Crop", "soil_type"
]
num_cols = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = cat_cols + num_cols

X = df[features].copy()
y = df["yield_per_area"].copy()

for c in cat_cols:
    X[c] = X[c].fillna("Unknown").astype(str).str.strip()
    X[c] = X[c].replace("", "Unknown")

# 3. Same chronological split
train_mask = df["Crop_Year"] <= 2011
val_mask = df["Crop_Year"].isin([2012, 2013])
test_mask = df["Crop_Year"].isin([2014, 2015])

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

# 4. Preprocessing
preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

# 5. XGBoost with 800 trees
model_800 = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=800,       # Increased from 500 to 800
        max_depth=8,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
])

# 6. Train
print("Training XGBoost with 800 trees...")
model_800.fit(X_train, y_train)
print("Training completed!")

# 7. Evaluate validation and test sets
def evaluate(name, actual, predicted):
    print(f"\n{name}")
    print("MAE:", round(mean_absolute_error(actual, predicted), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(actual, predicted)), 4))
    print("R²:", round(r2_score(actual, predicted), 4))
    print("Negative predictions:", int((predicted < 0).sum()))

val_pred = model_800.predict(X_val)
test_pred = model_800.predict(X_test)

evaluate("800-TREE VALIDATION (2012–2013)", y_val, val_pred)
evaluate("800-TREE TEST (2014–2015)", y_test, test_pred)

# 8. Save as a new model
out_dir = "/content/drive/MyDrive/yield prediction"

model_path = os.path.join(
    out_dir, "cropsense_xgboost_yield_800_trees.joblib"
)
joblib.dump(model_800, model_path)

print("\nSaved model:", model_path)


Training XGBoost with 800 trees...
Training completed!

800-TREE VALIDATION (2012–2013)
MAE: 1.5097
RMSE: 5.1439
R²: 0.8434
Negative predictions: 118

800-TREE TEST (2014–2015)
MAE: 1.4033
RMSE: 4.4761
R²: 0.8795
Negative predictions: 41

Saved model: /content/drive/MyDrive/yield prediction/cropsense_xgboost_yield_800_trees.joblib


In [ ]:

!pip -q install -U xgboost scikit-learn joblib

import os
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ============================================================
# 1. LOAD AND PREPARE DATA
# ============================================================

path = (
    "/content/drive/MyDrive/yield prediction/"
    "crop_production_124_training_ready.csv"
)

df = pd.read_csv(path, low_memory=False)
df.columns = df.columns.str.strip()

for col in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

cat_cols = [
    "State_Name", "District_Name", "Season", "Crop", "soil_type"
]
num_cols = [
    "Crop_Year", "temperature", "humidity", "rainfall_mm"
]
features = cat_cols + num_cols

X = df[features].copy()
y = df["yield_per_area"].copy()

for col in cat_cols:
    X[col] = (
        X[col].fillna("Unknown")
        .astype(str).str.strip()
        .replace("", "Unknown")
    )

# Same chronological split
train_mask = df["Crop_Year"] <= 2011
val_mask = df["Crop_Year"].isin([2012, 2013])
test_mask = df["Crop_Year"].isin([2014, 2015])

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("Train:", len(X_train))
print("Validation:", len(X_val))
print("Test (held out):", len(X_test))

# ============================================================
# 2. FIT PREPROCESSING ON TRAINING DATA ONLY
# ============================================================

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

X_train_t = preprocessor.fit_transform(X_train)
X_val_t = preprocessor.transform(X_val)

# ============================================================
# 3. HYPERPARAMETER EXPERIMENTS
# ============================================================

experiments = [
    {"max_depth": 4, "learning_rate": 0.03,
     "min_child_weight": 5, "reg_lambda": 10, "subsample": 0.85},
    {"max_depth": 5, "learning_rate": 0.03,
     "min_child_weight": 5, "reg_lambda": 10, "subsample": 0.85},
    {"max_depth": 6, "learning_rate": 0.03,
     "min_child_weight": 5, "reg_lambda": 10, "subsample": 0.85},
    {"max_depth": 8, "learning_rate": 0.03,
     "min_child_weight": 5, "reg_lambda": 10, "subsample": 0.85},
    {"max_depth": 4, "learning_rate": 0.05,
     "min_child_weight": 3, "reg_lambda": 5, "subsample": 0.90},
    {"max_depth": 5, "learning_rate": 0.05,
     "min_child_weight": 3, "reg_lambda": 5, "subsample": 0.90},
    {"max_depth": 6, "learning_rate": 0.05,
     "min_child_weight": 3, "reg_lambda": 5, "subsample": 0.90},
    {"max_depth": 8, "learning_rate": 0.05,
     "min_child_weight": 3, "reg_lambda": 5, "subsample": 0.90},
    {"max_depth": 4, "learning_rate": 0.08,
     "min_child_weight": 3, "reg_lambda": 10, "subsample": 0.85},
    {"max_depth": 6, "learning_rate": 0.08,
     "min_child_weight": 5, "reg_lambda": 10, "subsample": 0.85},
    {"max_depth": 5, "learning_rate": 0.03,
     "min_child_weight": 10, "reg_lambda": 20, "subsample": 0.90},
    {"max_depth": 6, "learning_rate": 0.03,
     "min_child_weight": 10, "reg_lambda": 20, "subsample": 0.90},
]

results = []
best_model = None
best_params = None
best_rmse = float("inf")

for i, params in enumerate(experiments, start=1):
    print(f"\nExperiment {i}/{len(experiments)}: {params}")

    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=2000,
        max_depth=params["max_depth"],
        learning_rate=params["learning_rate"],
        min_child_weight=params["min_child_weight"],
        reg_lambda=params["reg_lambda"],
        subsample=params["subsample"],
        colsample_bytree=0.85,
        early_stopping_rounds=100,
        eval_metric="rmse",
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    model.fit(
        X_train_t, y_train,
        eval_set=[(X_val_t, y_val)],
        verbose=False
    )

    val_pred = model.predict(X_val_t)

    mae = mean_absolute_error(y_val, val_pred)
    rmse = np.sqrt(mean_squared_error(y_val, val_pred))
    r2 = r2_score(y_val, val_pred)

    results.append({
        **params,
        "best_iteration": model.best_iteration,
        "validation_MAE": mae,
        "validation_RMSE": rmse,
        "validation_R2": r2
    })

    print(
        f"Best trees: {model.best_iteration + 1} | "
        f"MAE: {mae:.4f} | RMSE: {rmse:.4f} | R2: {r2:.4f}"
    )

    if rmse < best_rmse:
        best_rmse = rmse
        best_model = model
        best_params = params.copy()

# ============================================================
# 4. SHOW BEST VALIDATION RESULT
# ============================================================

results_df = pd.DataFrame(results).sort_values("validation_RMSE")

print("\n===== TOP 5 CONFIGURATIONS =====")
print(results_df.head(5).round(4).to_string(index=False))

print("\nBest parameters:", best_params)
print("Best validation RMSE:", round(best_rmse, 4))

# Save separately; do not overwrite previous models
out_dir = "/content/drive/MyDrive/yield prediction"

joblib.dump(
    {
        "preprocessor": preprocessor,
        "model": best_model,
        "features": features,
        "categorical_features": cat_cols,
        "numeric_features": num_cols,
        "best_parameters": best_params
    },
    os.path.join(out_dir, "cropsense_xgboost_tuned_candidate.joblib")
)

results_df.to_csv(
    os.path.join(out_dir, "cropsense_xgboost_tuning_results.csv"),
    index=False
)

print("\nSaved tuned candidate and experiment results.")
print("The test set has not been used to select the model.")


Train: 199107
Validation: 26448
Test (held out): 11311

Experiment 1/12: {'max_depth': 4, 'learning_rate': 0.03, 'min_child_weight': 5, 'reg_lambda': 10, 'subsample': 0.85}
Best trees: 914 | MAE: 1.9824 | RMSE: 5.4791 | R2: 0.8223

Experiment 2/12: {'max_depth': 5, 'learning_rate': 0.03, 'min_child_weight': 5, 'reg_lambda': 10, 'subsample': 0.85}
Best trees: 838 | MAE: 1.8564 | RMSE: 5.3220 | R2: 0.8324

Experiment 3/12: {'max_depth': 6, 'learning_rate': 0.03, 'min_child_weight': 5, 'reg_lambda': 10, 'subsample': 0.85}
Best trees: 610 | MAE: 1.8027 | RMSE: 5.2273 | R2: 0.8383

Experiment 4/12: {'max_depth': 8, 'learning_rate': 0.03, 'min_child_weight': 5, 'reg_lambda': 10, 'subsample': 0.85}
Best trees: 399 | MAE: 1.7037 | RMSE: 5.0971 | R2: 0.8462

Experiment 5/12: {'max_depth': 4, 'learning_rate': 0.05, 'min_child_weight': 3, 'reg_lambda': 5, 'subsample': 0.9}
Best trees: 454 | MAE: 1.9275 | RMSE: 5.4727 | R2: 0.8227

Experiment 6/12: {'max_depth': 5, 'learning_rate': 0.05, 'min_chil

In [ ]:

from google.colab import drive
drive.mount('/content/drive', force_remount=True)


Mounted at /content/drive


In [ ]:

import os

folder = "/content/drive/MyDrive/yield prediction"

print("Files related to XGBoost tuning:")
for f in os.listdir(folder):
    if "tuned" in f.lower() or "tuning" in f.lower():
        print(f)


Files related to XGBoost tuning:
cropsense_xgboost_tuning_results.csv
cropsense_xgboost_tuned_candidate.joblib


In [ ]:

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = "/content/drive/MyDrive/yield prediction"
DATA_PATH = os.path.join(BASE, "crop_production_124_training_ready.csv")
MODEL_PATH = os.path.join(BASE, "cropsense_xgboost_tuned_candidate.joblib")

# Load the tuned candidate and dataset
candidate = joblib.load(MODEL_PATH)
df = pd.read_csv(DATA_PATH)

# Recreate the chronological split used during tuning
df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")
df["yield_per_area"] = pd.to_numeric(df["yield_per_area"], errors="coerce")
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["Crop_Year", "yield_per_area"])
df = df[(df["Area"] > 0) & (df["Production"] > 0)]

cat_cols = ["State_Name", "District_Name", "Season", "Crop", "soil_type"]
num_cols = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = cat_cols + num_cols

train = df[df["Crop_Year"] <= 2011]
valid = df[df["Crop_Year"].between(2012, 2013)]
test = df[df["Crop_Year"].between(2014, 2015)]

X_test = test[features]
y_test = test["yield_per_area"]

# Load preprocessor and model from saved candidate
print("Saved candidate type:", type(candidate))
if isinstance(candidate, dict):
    print("Saved keys:", candidate.keys())
    preprocessor = candidate.get("preprocessor")
    model = candidate.get("model", candidate.get("best_model"))
else:
    raise ValueError("Candidate is not a dictionary. Check the saved artifact structure.")

if preprocessor is None or model is None:
    raise ValueError("Could not find preprocessor/model in candidate. Share the printed keys.")

X_test_processed = preprocessor.transform(X_test)
predictions = model.predict(X_test_processed)

# Evaluate on the held-out test set
predictions = np.maximum(predictions, 0)

print("\n===== TUNED XGBOOST TEST RESULTS =====")
print("Test samples:", len(y_test))
print(f"MAE:  {mean_absolute_error(y_test, predictions):.4f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test, predictions)):.4f}")
print(f"R²:   {r2_score(y_test, predictions):.4%}")
print("\nPrevious model test R²: 87.95%")


Saved candidate type: <class 'dict'>
Saved keys: dict_keys(['preprocessor', 'model', 'features', 'categorical_features', 'numeric_features', 'best_parameters'])

===== TUNED XGBOOST TEST RESULTS =====
Test samples: 11311
MAE:  1.8101
RMSE: 4.8928
R²:   85.6051%

Previous model test R²: 87.95%


In [ ]:

import pandas as pd
import os

folder = "/content/drive/MyDrive/yield prediction"
results_path = os.path.join(
    folder, "cropsense_xgboost_tuning_results.csv"
)

results = pd.read_csv(results_path)
display(results.sort_values("validation_RMSE").head(12))


,max_depth,learning_rate,min_child_weight,reg_lambda,subsample,best_iteration,validation_MAE,validation_RMSE,validation_R2
0,8,0.03,5,10,0.85,398,1.703736,5.097128,0.846235
1,6,0.03,5,10,0.85,609,1.802730,5.227293,0.838281
2,6,0.03,10,20,0.90,741,1.815496,5.235361,0.837782
3,6,0.08,5,10,0.85,222,1.823446,5.235871,0.837750
4,5,0.03,5,10,0.85,837,1.856388,5.322047,0.832365
5,5,0.03,10,20,0.90,922,1.882301,5.330871,0.831809
6,8,0.05,3,5,0.90,129,1.839614,5.364640,0.829671
7,5,0.05,3,5,0.90,297,1.897466,5.433077,0.825298
8,4,0.05,3,5,0.90,453,1.927464,5.472746,0.822737
9,4,0.03,5,10,0.85,913,1.982354,5.479088,0.822326


In [ ]:

import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/yield prediction/crop_production_124_training_ready.csv"
df = pd.read_csv(path)

df["yield_per_area"] = pd.to_numeric(
    df["yield_per_area"], errors="coerce"
)
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=["yield_per_area", "Crop_Year"])

# Training period only: do not inspect test labels to choose cleaning rules
train = df[df["Crop_Year"] <= 2011].copy()

# Crop-level distribution and extreme-value counts
summary = (
    train.groupby("Crop")["yield_per_area"]
    .agg(
        records="count",
        median="median",
        p90=lambda x: x.quantile(0.90),
        p95=lambda x: x.quantile(0.95),
        p99=lambda x: x.quantile(0.99),
        maximum="max"
    )
    .sort_values("maximum", ascending=False)
)

print("Top 20 crops by maximum training yield:")
display(summary.head(20).round(3))

# Inspect the most extreme individual training records
thresholds = train.groupby("Crop")["yield_per_area"].quantile(0.99)
train["crop_p99"] = train["Crop"].map(thresholds)

extreme = train[train["yield_per_area"] > train["crop_p99"]].copy()
extreme = extreme.sort_values("yield_per_area", ascending=False)

print("\nTop 30 records above their crop-specific 99th percentile:")
display(
    extreme[
        ["State_Name", "District_Name", "Crop_Year",
         "Season", "Crop", "Area", "Production", "yield_per_area"]
    ].head(30).round(3)
)


Top 20 crops by maximum training yield:


,records,median,p90,p95,p99,maximum
Crop,,,,,,
Cashewnut,856,0.401,0.804,1.000,1.396,9801.000
Sugarcane,6570,52.203,94.502,107.118,137.970,2247.533
Onion,5843,10.127,21.830,25.997,32.708,2150.000
Maize,11294,1.671,3.513,4.546,6.967,1494.000
Cotton(lint),3720,1.316,3.077,4.000,6.000,300.000
Rice,12589,1.853,3.142,3.572,4.451,223.727
Potato,5828,10.705,23.200,25.836,30.076,213.846
Banana,2693,21.664,49.695,64.101,80.270,137.400
Papaya,273,15.082,90.815,130.441,130.476,130.500



Top 30 records above their crop-specific 99th percentile:


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area
175865,Tamil Nadu,PERAMBALUR,2008,Whole Year,Cashewnut,1.0,9801.0,9801.000
174381,Tamil Nadu,MADURAI,1997,Whole Year,Sugarcane,8108.0,18223000.0,2247.533
66324,Haryana,KARNAL,2008,Whole Year,Onion,2.0,4300.0,2150.000
178746,Tamil Nadu,THIRUVALLUR,1997,Whole Year,Sugarcane,5964.0,11811000.0,1980.382
126592,Maharashtra,NASHIK,1997,Autumn,Maize,1.0,1494.0,1494.000
173083,Tamil Nadu,KANCHIPURAM,1997,Whole Year,Sugarcane,6735.0,9896000.0,1469.339
175100,Tamil Nadu,NAMAKKAL,1997,Whole Year,Sugarcane,9415.0,13608000.0,1445.353
179359,Tamil Nadu,TIRUCHIRAPPALLI,1997,Whole Year,Sugarcane,4315.0,5806000.0,1345.539
123689,Maharashtra,JALGAON,1997,Autumn,Maize,3.0,3980.0,1326.667
171139,Tamil Nadu,CUDDALORE,1997,Whole Year,Sugarcane,23826.0,28322000.0,1188.701


In [ ]:

import pandas as pd
import numpy as np
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "crop_production_124_training_ready.csv")

df = pd.read_csv(path)

# Keep the original dataset untouched
df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")
df["Area"] = pd.to_numeric(df["Area"], errors="coerce")
df["Production"] = pd.to_numeric(df["Production"], errors="coerce")

# Training period only
train = df[df["Crop_Year"] <= 2011].copy()
train["calculated_yield"] = train["Production"] / train["Area"]

# Crop-specific 99th percentile thresholds
p99 = train.groupby("Crop")["calculated_yield"].quantile(0.99)
train["crop_p99"] = train["Crop"].map(p99)

extreme = train[
    (train["Area"] > 0) &
    (train["calculated_yield"] > train["crop_p99"])
].copy()

extreme = extreme.sort_values("calculated_yield", ascending=False)

columns = [
    "State_Name", "District_Name", "Crop_Year", "Season",
    "Crop", "Area", "Production", "calculated_yield", "crop_p99"
]

report_path = os.path.join(base, "crop_yield_extreme_training_records.csv")
extreme[columns].to_csv(report_path, index=False)

print("Extreme training records:", len(extreme))
print("Report saved:", report_path)
display(extreme[columns].head(30).round(3))



FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/yield prediction/crop_production_124_training_ready.csv'

In [ ]:

import pandas as pd
import numpy as np
import os

path = "/content/drive/MyDrive/yield prediction/crop_production_124_training_ready.csv"
df = pd.read_csv(path)

print("DATASET SHAPE:", df.shape)
print("\n1. MISSING VALUES")
print(df.isna().sum().sort_values(ascending=False).to_string())

print("\n2. DUPLICATE ROWS")
print("Exact duplicate rows:", df.duplicated().sum())

print("\n3. AREA AND PRODUCTION CHECKS")
area = pd.to_numeric(df["Area"], errors="coerce")
production = pd.to_numeric(df["Production"], errors="coerce")
target = pd.to_numeric(df["yield_per_area"], errors="coerce")

print("Area <= 0:", (area <= 0).sum())
print("Production < 0:", (production < 0).sum())
print("Missing/non-numeric Area:", area.isna().sum())
print("Missing/non-numeric Production:", production.isna().sum())

valid = area.gt(0) & production.notna() & target.notna()
recalculated = production[valid] / area[valid]
difference = (recalculated - target[valid]).abs()

print("\n4. TARGET CONSISTENCY")
print("Rows checked:", len(difference))
print("Rows differing by > 0.01:", (difference > 0.01).sum())
print("Maximum absolute difference:", difference.max())

print("\n5. EXTREME YIELD VALUES")
check = df.copy()
check["Area"] = area
check["Production"] = production
check["yield_per_area"] = target
check = check.replace([np.inf, -np.inf], np.nan)
check = check[check["Area"] > 0].copy()
check["calculated_yield"] = check["Production"] / check["Area"]

display(
    check.sort_values("calculated_yield", ascending=False)[
        ["State_Name", "District_Name", "Crop_Year", "Season",
         "Crop", "Area", "Production", "yield_per_area",
         "calculated_yield"]
    ].head(20)
)

print("\nAudit complete. Original dataset was not modified.")


DATASET SHAPE: (236866, 12)

1. MISSING VALUES
State_Name        0
District_Name     0
Crop_Year         0
Season            0
Crop              0
Area              0
Production        0
temperature       0
humidity          0
rainfall_mm       0
soil_type         0
yield_per_area    0

2. DUPLICATE ROWS
Exact duplicate rows: 0

3. AREA AND PRODUCTION CHECKS
Area <= 0: 0
Production < 0: 0
Missing/non-numeric Area: 0
Missing/non-numeric Production: 0

4. TARGET CONSISTENCY
Rows checked: 236866
Rows differing by > 0.01: 0
Maximum absolute difference: 2.2737367544323206e-13

5. EXTREME YIELD VALUES


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area,calculated_yield
175865,Tamil Nadu,PERAMBALUR,2008,Whole Year,Cashewnut,1.0,9801.0,9801.000000,9801.000000
174381,Tamil Nadu,MADURAI,1997,Whole Year,Sugarcane,8108.0,18223000.0,2247.533300,2247.533300
66324,Haryana,KARNAL,2008,Whole Year,Onion,2.0,4300.0,2150.000000,2150.000000
178746,Tamil Nadu,THIRUVALLUR,1997,Whole Year,Sugarcane,5964.0,11811000.0,1980.382294,1980.382294
126592,Maharashtra,NASHIK,1997,Autumn,Maize,1.0,1494.0,1494.000000,1494.000000
173083,Tamil Nadu,KANCHIPURAM,1997,Whole Year,Sugarcane,6735.0,9896000.0,1469.339272,1469.339272
175100,Tamil Nadu,NAMAKKAL,1997,Whole Year,Sugarcane,9415.0,13608000.0,1445.353160,1445.353160
179359,Tamil Nadu,TIRUCHIRAPPALLI,1997,Whole Year,Sugarcane,4315.0,5806000.0,1345.538818,1345.538818
123689,Maharashtra,JALGAON,1997,Autumn,Maize,3.0,3980.0,1326.666667,1326.666667
171139,Tamil Nadu,CUDDALORE,1997,Whole Year,Sugarcane,23826.0,28322000.0,1188.701419,1188.701419



Audit complete. Original dataset was not modified.


In [ ]:

import pandas as pd
import numpy as np

path = "/content/drive/MyDrive/yield prediction/crop_production_124_training_ready.csv"
df = pd.read_csv(path)

df["yield_per_area"] = df["Production"] / df["Area"]

# Calculate crop-specific statistics
stats = df.groupby("Crop")["yield_per_area"].agg(
    records="count",
    median="median",
    p99=lambda x: x.quantile(0.99),
    maximum="max"
)

stats["max_to_median"] = (
    stats["maximum"] / stats["median"].replace(0, np.nan)
)

# Flag records above their crop's 99th percentile
p99 = df.groupby("Crop")["yield_per_area"].quantile(0.99)
df["crop_p99"] = df["Crop"].map(p99)

counts = (
    df[df["yield_per_area"] > df["crop_p99"]]
    .groupby("Crop")
    .size()
    .rename("above_p99_records")
)

stats = stats.join(counts).fillna({"above_p99_records": 0})
stats["above_p99_records"] = stats["above_p99_records"].astype(int)

display(
    stats.sort_values("max_to_median", ascending=False)
    .head(20).round(3)
)


,records,median,p99,maximum,max_to_median,above_p99_records
Crop,,,,,,
Cashewnut,1065,0.397,1.898,9801.000,24682.044,11
Maize,13723,1.800,7.231,1494.000,830.000,138
Sesamum,8724,0.333,1.108,235.000,705.000,88
Niger seed,2004,0.289,1.005,71.000,245.310,21
Cotton(lint),4350,1.383,5.788,300.000,216.959,44
Onion,6979,10.300,33.000,2150.000,208.738,69
Rice,15078,1.939,4.518,223.727,115.394,151
Sannhamp,2141,0.500,6.500,55.550,111.100,21
Sunflower,5468,0.800,2.304,70.000,87.500,55


In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "crop_production_124_training_ready.csv")

df = pd.read_csv(path)
df["yield_per_area"] = df["Production"] / df["Area"]

# Crop-specific 99th percentile for comparison
p99 = df.groupby("Crop")["yield_per_area"].quantile(0.99)
df["crop_p99"] = df["Crop"].map(p99)

extreme = df.sort_values("yield_per_area", ascending=False).head(50)

cols = [
    "State_Name", "District_Name", "Crop_Year", "Season",
    "Crop", "Area", "Production", "yield_per_area", "crop_p99"
]

display(extreme[cols].round(3))

out = os.path.join(base, "top_50_extreme_yield_records.csv")
extreme[cols].to_csv(out, index=False)
print("\nReport saved to:", out)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area,crop_p99
175865,Tamil Nadu,PERAMBALUR,2008,Whole Year,Cashewnut,1.0,9801.0,9801.000,1.898
174381,Tamil Nadu,MADURAI,1997,Whole Year,Sugarcane,8108.0,18223000.0,2247.533,137.585
66324,Haryana,KARNAL,2008,Whole Year,Onion,2.0,4300.0,2150.000,33.000
178746,Tamil Nadu,THIRUVALLUR,1997,Whole Year,Sugarcane,5964.0,11811000.0,1980.382,137.585
126592,Maharashtra,NASHIK,1997,Autumn,Maize,1.0,1494.0,1494.000,7.231
173083,Tamil Nadu,KANCHIPURAM,1997,Whole Year,Sugarcane,6735.0,9896000.0,1469.339,137.585
175100,Tamil Nadu,NAMAKKAL,1997,Whole Year,Sugarcane,9415.0,13608000.0,1445.353,137.585
179359,Tamil Nadu,TIRUCHIRAPPALLI,1997,Whole Year,Sugarcane,4315.0,5806000.0,1345.539,137.585
123689,Maharashtra,JALGAON,1997,Autumn,Maize,3.0,3980.0,1326.667,7.231
171139,Tamil Nadu,CUDDALORE,1997,Whole Year,Sugarcane,23826.0,28322000.0,1188.701,137.585



Report saved to: /content/drive/MyDrive/yield prediction/top_50_extreme_yield_records.csv


In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "crop_production_124_training_ready.csv")

df = pd.read_csv(path)

# Recalculate target and crop-specific 99th percentile
df["yield_per_area"] = df["Production"] / df["Area"]
crop_p99 = df.groupby("Crop")["yield_per_area"].transform(
    lambda s: s.quantile(0.99)
)

df["above_crop_p99"] = df["yield_per_area"] > crop_p99
df["extreme_ratio"] = df["yield_per_area"] / crop_p99

# 1. Extreme records grouped by crop and year
crop_year = (
    df[df["above_crop_p99"]]
    .groupby(["Crop", "Crop_Year"])
    .agg(
        extreme_records=("yield_per_area", "size"),
        median_yield=("yield_per_area", "median"),
        max_yield=("yield_per_area", "max"),
        median_area=("Area", "median")
    )
    .reset_index()
    .sort_values("extreme_records", ascending=False)
)

print("TOP CROP-YEAR GROUPS WITH EXTREME YIELDS")
display(crop_year.head(30).round(3))

# 2. Focus on suspicious crop/year patterns
focus = df[
    (
        (df["Crop"] == "Sugarcane") &
        (df["Crop_Year"].isin([1997, 1998, 2008]))
    ) |
    (
        (df["Crop"] == "Maize") &
        (df["Crop_Year"] == 1997)
    ) |
    (
        (df["Crop"] == "Cashewnut") &
        (df["Crop_Year"] == 2008)
    )
].copy()

print("\nFOCUSED CROP/YEAR SUMMARY")
summary = (
    focus.groupby(["Crop", "Crop_Year"])
    .agg(
        records=("yield_per_area", "size"),
        median_yield=("yield_per_area", "median"),
        p95_yield=("yield_per_area", lambda s: s.quantile(0.95)),
        max_yield=("yield_per_area", "max"),
        records_area_le_5=("Area", lambda s: (s <= 5).sum())
    )
    .reset_index()
)
display(summary.round(3))

# Save audit report; original dataset remains unchanged
out = os.path.join(base, "yield_extreme_audit_by_crop_year.csv")
crop_year.to_csv(out, index=False)
print("\nAudit saved to:", out)


TOP CROP-YEAR GROUPS WITH EXTREME YIELDS


,Crop,Crop_Year,extreme_records,median_yield,max_yield,median_area
617,Wheat,2011,30,5.255,5.676,170500.0
440,Rice,2013,29,4.828,6.084,30889.0
511,Sugarcane,1997,28,1075.769,2247.533,6780.5
187,Groundnut,2011,25,3.751,6.146,4393.0
263,Maize,2013,24,7.530,8.675,10764.5
116,Cotton(lint),1997,23,58.697,300.000,829.0
189,Groundnut,2013,22,3.630,6.094,2377.5
371,Peas & beans (Pulses),1998,22,9.185,13.589,27.5
418,Rapeseed &Mustard,2010,21,2.398,3.361,935.0
408,Ragi,2013,21,3.966,4.748,109.0



FOCUSED CROP/YEAR SUMMARY


,Crop,Crop_Year,records,median_yield,p95_yield,max_yield,records_area_le_5
0,Cashewnut,2008,79,0.521,1.073,9801.000,6
1,Maize,1997,648,1.579,3.464,1494.000,43
2,Sugarcane,1997,330,55.366,1073.573,2247.533,2
3,Sugarcane,1998,408,51.851,121.505,303.485,11
4,Sugarcane,2008,482,51.147,106.531,314.736,16



Audit saved to: /content/drive/MyDrive/yield prediction/yield_extreme_audit_by_crop_year.csv


In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "crop_production_124_training_ready.csv")
df = pd.read_csv(path)

df["yield_per_area"] = df["Production"] / df["Area"]

checks = [
    ("Sugarcane", 1997),
    ("Sugarcane", 1998),
    ("Sugarcane", 2008),
    ("Cashewnut", 2008),
    ("Maize", 1997),
]

for crop, year in checks:
    group = df[
        (df["Crop"] == crop) &
        (df["Crop_Year"] == year)
    ].copy()

    if group.empty:
        continue

    print(f"\n{'='*70}")
    print(f"{crop} — {year} | Total records: {len(group)}")
    print("Yield quantiles:")
    print(group["yield_per_area"].quantile(
        [0.50, 0.90, 0.95, 0.99, 1.00]
    ).round(3))

    # Highest 10 records
    cols = [
        "State_Name", "District_Name", "Crop_Year",
        "Season", "Crop", "Area", "Production",
        "yield_per_area"
    ]
    print("\nHighest 10 yields:")
    display(
        group.nlargest(10, "yield_per_area")[cols]
        .round(3)
        .reset_index()
        .rename(columns={"index": "original_row"})
    )

    # Typical records around the median
    median = group["yield_per_area"].median()
    group["distance_from_median"] = (
        group["yield_per_area"] - median
    ).abs()

    print("\n5 records closest to the median:")
    display(
        group.nsmallest(5, "distance_from_median")[cols]
        .round(3)
        .reset_index()
        .rename(columns={"index": "original_row"})
    )



Sugarcane — 1997 | Total records: 330
Yield quantiles:
0.50      55.366
0.90     106.134
0.95    1073.573
0.99    1416.407
1.00    2247.533
Name: yield_per_area, dtype: float64

Highest 10 yields:


,original_row,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area
0,174381,Tamil Nadu,MADURAI,1997,Whole Year,Sugarcane,8108.0,18223000.0,2247.533
1,178746,Tamil Nadu,THIRUVALLUR,1997,Whole Year,Sugarcane,5964.0,11811000.0,1980.382
2,173083,Tamil Nadu,KANCHIPURAM,1997,Whole Year,Sugarcane,6735.0,9896000.0,1469.339
3,175100,Tamil Nadu,NAMAKKAL,1997,Whole Year,Sugarcane,9415.0,13608000.0,1445.353
4,179359,Tamil Nadu,TIRUCHIRAPPALLI,1997,Whole Year,Sugarcane,4315.0,5806000.0,1345.539
5,171139,Tamil Nadu,CUDDALORE,1997,Whole Year,Sugarcane,23826.0,28322000.0,1188.701
6,173658,Tamil Nadu,KARUR,1997,Whole Year,Sugarcane,5961.0,7081000.0,1187.888
7,176767,Tamil Nadu,SALEM,1997,Whole Year,Sugarcane,13759.0,15866000.0,1153.136
8,170659,Tamil Nadu,COIMBATORE,1997,Whole Year,Sugarcane,10756.0,12272000.0,1140.945
9,181749,Tamil Nadu,VILLUPURAM,1997,Whole Year,Sugarcane,33175.0,37158000.0,1120.060



5 records closest to the median:


,original_row,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area
0,33552,Bihar,KHAGARIA,1997,Whole Year,Sugarcane,207.0,11455.0,55.338
1,150540,Odisha,NABARANGPUR,1997,Winter,Sugarcane,2833.0,156931.0,55.394
2,31705,Bihar,JAMUI,1997,Whole Year,Sugarcane,530.0,29361.0,55.398
3,209973,Uttar Pradesh,KUSHI NAGAR,1997,Whole Year,Sugarcane,74539.0,4130951.0,55.420
4,212356,Uttar Pradesh,MAINPURI,1997,Whole Year,Sugarcane,499.0,27565.0,55.240



Sugarcane — 1998 | Total records: 408
Yield quantiles:
0.50     51.851
0.90     94.053
0.95    121.505
0.99    163.158
1.00    303.485
Name: yield_per_area, dtype: float64

Highest 10 yields:


...

,original_row,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area
0,175002,Tamil Nadu,NAGAPATTINAM,2008,Whole Year,Cashewnut,2049.0,1068.0,0.521
1,181143,Tamil Nadu,TUTICORIN,2008,Whole Year,Cashewnut,817.0,426.0,0.521
2,180131,Tamil Nadu,TIRUNELVELI,2008,Whole Year,Cashewnut,4992.0,2601.0,0.521
3,172400,Tamil Nadu,DINDIGUL,2008,Whole Year,Cashewnut,411.0,214.0,0.521
4,182484,Tamil Nadu,VIRUDHUNAGAR,2008,Whole Year,Cashewnut,73.0,38.0,0.521



Maize — 1997 | Total records: 648
Yield quantiles:
0.50       1.579
0.90       2.753
0.95       3.464
0.99       7.912
1.00    1494.000
Name: yield_per_area, dtype: float64

Highest 10 yields:


,original_row,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area
0,126592,Maharashtra,NASHIK,1997,Autumn,Maize,1.0,1494.0,1494.000
1,123689,Maharashtra,JALGAON,1997,Autumn,Maize,3.0,3980.0,1326.667
2,127977,Maharashtra,PUNE,1997,Autumn,Maize,4.0,4570.0,1142.500
3,127043,Maharashtra,OSMANABAD,1997,Autumn,Maize,5.0,5635.0,1127.000
4,119266,Maharashtra,AHMEDNAGAR,1997,Autumn,Maize,1.0,1113.0,1113.000
5,122380,Maharashtra,DHULE,1997,Autumn,Maize,4.0,2901.0,725.250
6,132391,Manipur,SENAPATI,1997,Kharif,Maize,1.0,8.0,8.000
7,236263,West Bengal,PURULIA,1997,Whole Year,Maize,869.0,6790.0,7.814
8,5324,Andhra Pradesh,PRAKASAM,1997,Kharif,Maize,1800.0,8100.0,4.500
9,230123,West Bengal,COOCHBEHAR,1997,Whole Year,Maize,7.0,30.0,4.286



5 records closest to the median:


,original_row,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area
0,199336,Uttar Pradesh,ETAH,1997,Summer,Maize,837.0,1322.0,1.579
1,209969,Uttar Pradesh,KUSHI NAGAR,1997,Summer,Maize,780.0,1232.0,1.579
2,195121,Uttar Pradesh,BARABANKI,1997,Summer,Maize,523.0,826.0,1.579
3,213696,Uttar Pradesh,MEERUT,1997,Summer,Maize,816.0,1289.0,1.580
4,203924,Uttar Pradesh,GORAKHPUR,1997,Summer,Maize,164.0,259.0,1.579


In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "crop_production_124_training_ready.csv")
df = pd.read_csv(path)

df["yield_per_area"] = df["Production"] / df["Area"]

# Crop-year-specific 99th percentile
df["crop_year_p99"] = df.groupby(
    ["Crop", "Crop_Year"]
)["yield_per_area"].transform(
    lambda s: s.quantile(0.99)
)

# Flag the crop-year combinations investigated so far
focus = (
    ((df["Crop"] == "Sugarcane") &
     (df["Crop_Year"].isin([1997, 1998, 2008]))) |
    ((df["Crop"] == "Cashewnut") &
     (df["Crop_Year"] == 2008)) |
    ((df["Crop"] == "Maize") &
     (df["Crop_Year"] == 1997))
)

audit = df.loc[focus].copy()
audit["above_crop_year_p99"] = (
    audit["yield_per_area"] > audit["crop_year_p99"]
)

cols = [
    "State_Name", "District_Name", "Crop_Year", "Season",
    "Crop", "Area", "Production", "yield_per_area",
    "crop_year_p99", "above_crop_year_p99"
]

out = os.path.join(base, "crop_yield_source_audit.csv")
audit[cols].to_csv(out, index=False)

print("Records inspected:", len(audit))
print("Above crop-year P99:", int(audit["above_crop_year_p99"].sum()))
print("Audit saved:", out)

print("\nSummary by crop and year:")
display(
    audit.groupby(["Crop", "Crop_Year"]).agg(
        records=("yield_per_area", "size"),
        above_p99=("above_crop_year_p99", "sum"),
        median_yield=("yield_per_area", "median"),
        max_yield=("yield_per_area", "max")
    ).round(3)
)


Records inspected: 1947
Above crop-year P99: 22
Audit saved: /content/drive/MyDrive/yield prediction/crop_yield_source_audit.csv

Summary by crop and year:


records  above_p99  median_yield  max_yield
Crop      Crop_Year                                             
Cashewnut 2008            79          1         0.521   9801.000
Maize     1997           648          7         1.579   1494.000
Sugarcane 1997           330          4        55.366   2247.533
          1998           408          5        51.851    303.485
          2008           482          5        51.147    314.736

In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "crop_yield_source_audit.csv")
audit = pd.read_csv(path)

flagged = audit[audit["above_crop_year_p99"]].copy()
flagged = flagged.sort_values("yield_per_area", ascending=False)

cols = [
    "State_Name", "District_Name", "Crop_Year", "Season",
    "Crop", "Area", "Production", "yield_per_area",
    "crop_year_p99"
]

print("Flagged records:", len(flagged))
display(flagged[cols].round(3).reset_index(drop=True))

out = os.path.join(base, "flagged_yield_records_for_review.csv")
flagged[cols].to_csv(out, index=False)
print("Review file saved:", out)


Flagged records: 22


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area,crop_year_p99
0,Tamil Nadu,PERAMBALUR,2008,Whole Year,Cashewnut,1.0,9801.0,9801.000,2158.217
1,Tamil Nadu,MADURAI,1997,Whole Year,Sugarcane,8108.0,18223000.0,2247.533,1416.407
2,Tamil Nadu,THIRUVALLUR,1997,Whole Year,Sugarcane,5964.0,11811000.0,1980.382,1416.407
3,Maharashtra,NASHIK,1997,Autumn,Maize,1.0,1494.0,1494.000,7.912
4,Tamil Nadu,KANCHIPURAM,1997,Whole Year,Sugarcane,6735.0,9896000.0,1469.339,1416.407
5,Tamil Nadu,NAMAKKAL,1997,Whole Year,Sugarcane,9415.0,13608000.0,1445.353,1416.407
6,Maharashtra,JALGAON,1997,Autumn,Maize,3.0,3980.0,1326.667,7.912
7,Maharashtra,PUNE,1997,Autumn,Maize,4.0,4570.0,1142.500,7.912
8,Maharashtra,OSMANABAD,1997,Autumn,Maize,5.0,5635.0,1127.000,7.912
9,Maharashtra,AHMEDNAGAR,1997,Autumn,Maize,1.0,1113.0,1113.000,7.912


Review file saved: /content/drive/MyDrive/yield prediction/flagged_yield_records_for_review.csv


In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "flagged_yield_records_for_review.csv")
df = pd.read_csv(path)

# Prioritize extremely high yields, excluding the relatively modest Manipur record
priority = df[
    (df["yield_per_area"] > 100) |
    (
        (df["Crop"] == "Maize") &
        (df["Crop_Year"] == 1997) &
        (df["Area"] <= 5)
    )
].copy()

priority["verification_reason"] = priority.apply(
    lambda r: (
        "Very small area" if r["Area"] <= 5
        else "Extreme yield; verify source units and reported values"
    ),
    axis=1
)

cols = [
    "State_Name", "District_Name", "Crop_Year", "Season",
    "Crop", "Area", "Production", "yield_per_area",
    "verification_reason"
]

display(priority[cols].round(3).reset_index(drop=True))

out = os.path.join(base, "priority_source_verification.csv")
priority[cols].to_csv(out, index=False)
print("\nPriority records:", len(priority))
print("Saved:", out)


,State_Name,District_Name,Crop_Year,Season,Crop,Area,Production,yield_per_area,verification_reason
0,Tamil Nadu,PERAMBALUR,2008,Whole Year,Cashewnut,1.0,9801.0,9801.000,Very small area
1,Tamil Nadu,MADURAI,1997,Whole Year,Sugarcane,8108.0,18223000.0,2247.533,Extreme yield; verify source units and reporte...
2,Tamil Nadu,THIRUVALLUR,1997,Whole Year,Sugarcane,5964.0,11811000.0,1980.382,Extreme yield; verify source units and reporte...
3,Maharashtra,NASHIK,1997,Autumn,Maize,1.0,1494.0,1494.000,Very small area
4,Tamil Nadu,KANCHIPURAM,1997,Whole Year,Sugarcane,6735.0,9896000.0,1469.339,Extreme yield; verify source units and reporte...
5,Tamil Nadu,NAMAKKAL,1997,Whole Year,Sugarcane,9415.0,13608000.0,1445.353,Extreme yield; verify source units and reporte...
6,Maharashtra,JALGAON,1997,Autumn,Maize,3.0,3980.0,1326.667,Very small area
7,Maharashtra,PUNE,1997,Autumn,Maize,4.0,4570.0,1142.500,Very small area
8,Maharashtra,OSMANABAD,1997,Autumn,Maize,5.0,5635.0,1127.000,Very small area
9,Maharashtra,AHMEDNAGAR,1997,Autumn,Maize,1.0,1113.0,1113.000,Very small area



Priority records: 22
Saved: /content/drive/MyDrive/yield prediction/priority_source_verification.csv


In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
data_path = os.path.join(base, "crop_production_124_training_ready.csv")
flag_path = os.path.join(base, "flagged_yield_records_for_review.csv")

df = pd.read_csv(data_path)
flags = pd.read_csv(flag_path)

# Match exact records using their identifying columns and values
keys = [
    "State_Name", "District_Name", "Crop_Year", "Season",
    "Crop", "Area", "Production"
]

original = df.copy()
original["_row_id"] = range(len(original))

flag_keys = flags[keys].drop_duplicates()
candidate = original.merge(
    flag_keys.assign(_flag=True),
    on=keys,
    how="left"
)

removed = candidate["_flag"].fillna(False)
cleaned = candidate.loc[~removed, original.columns].drop(
    columns=["_row_id"], errors="ignore"
)

out = os.path.join(base, "yield_candidate_excluding_flagged.csv")
cleaned.to_csv(out, index=False)

print("Original rows:", len(df))
print("Flagged records matched:", int(removed.sum()))
print("Candidate rows:", len(cleaned))
print("Saved candidate:", out)
print("\nWARNING: This is an experimental candidate, not a verified-clean dataset.")


/tmp/ipykernel_1423/323786114.py:27: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  removed = candidate["_flag"].fillna(False)


Original rows: 236866
Flagged records matched: 22
Candidate rows: 236844
Saved candidate: /content/drive/MyDrive/yield prediction/yield_candidate_excluding_flagged.csv



In [ ]:

import joblib
import os

base = "/content/drive/MyDrive/yield prediction"
model_path = os.path.join(
    base, "cropsense_xgboost_yield_800_trees.joblib"
)

model = joblib.load(model_path)

print("Model loaded successfully.")
print("Model object type:", type(model))

if hasattr(model, "feature_names_in_"):
    print("Expected features:", list(model.feature_names_in_))
elif hasattr(model, "named_steps"):
    print("Pipeline steps:", list(model.named_steps.keys()))
    print("Expected preprocessing/model pipeline is available.")
elif isinstance(model, dict):
    print("Saved artifact keys:", list(model.keys()))
else:
    print("Inspect the model object above to identify its input features.")


Model loaded successfully.
Model object type: <class 'sklearn.pipeline.Pipeline'>
Expected features: ['State_Name', 'District_Name', 'Season', 'Crop', 'soil_type', 'Crop_Year', 'temperature', 'humidity', 'rainfall_mm']


In [ ]:

import pandas as pd
import numpy as np
import joblib
import os
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base = "/content/drive/MyDrive/yield prediction"

original_path = os.path.join(
    base, "crop_production_124_training_ready.csv"
)
candidate_path = os.path.join(
    base, "yield_candidate_excluding_flagged.csv"
)
model_path = os.path.join(
    base, "cropsense_xgboost_yield_800_trees.joblib"
)

model = joblib.load(model_path)
original = pd.read_csv(original_path)
candidate = pd.read_csv(candidate_path)

features = [
    "State_Name", "District_Name", "Season", "Crop", "soil_type",
    "Crop_Year", "temperature", "humidity", "rainfall_mm"
]
target = "yield_per_area"

# Match the existing chronological test period: 2014–2015
results = []

for name, data in [
    ("Original", original),
    ("Candidate", candidate)
]:
    test = data[data["Crop_Year"].isin([2014, 2015])].copy()

    X_test = test[features]
    y_test = test[target]
    predictions = model.predict(X_test)

    results.append({
        "Dataset": name,
        "Test rows": len(test),
        "MAE": mean_absolute_error(y_test, predictions),
        "RMSE": np.sqrt(mean_squared_error(y_test, predictions)),
        "R2": r2_score(y_test, predictions)
    })

results_df = pd.DataFrame(results)
display(results_df.round(4))

print("Note: this is a diagnostic comparison on the test years.")
print("Use the validation years 2012–2013 to select cleaning/model changes.")


,Dataset,Test rows,MAE,RMSE,R2
0,Original,11311,1.7419,4.7612,0.8637
1,Candidate,11311,1.7419,4.7612,0.8637


Note: this is a diagnostic comparison on the test years.
Use the validation years 2012–2013 to select cleaning/model changes.


# Model Training

In [ ]:

import os
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Paths
base = "/content/drive/MyDrive/yield prediction"
original_path = os.path.join(base, "crop_production_124_training_ready.csv")
candidate_path = os.path.join(base, "yield_candidate_excluding_flagged.csv")

# Features and target
cat_cols = ["State_Name", "District_Name", "Season", "Crop", "soil_type"]
num_cols = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = cat_cols + num_cols
target = "yield_per_area"

def load_data(path):
    df = pd.read_csv(path, low_memory=False)
    df.columns = df.columns.str.strip()

    for c in ["Crop_Year", "Area", "Production",
              "temperature", "humidity", "rainfall_mm"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df = df[
        df["Crop_Year"].notna()
        & df["Area"].gt(0)
        & df["Production"].gt(0)
    ].copy()

    # Recalculate target consistently
    df[target] = df["Production"] / df["Area"]
    df = df.replace([np.inf, -np.inf], np.nan)
    df = df[df[target].notna()].copy()

    for c in cat_cols:
        df[c] = df[c].fillna("Unknown").astype(str).str.strip()
        df[c] = df[c].replace("", "Unknown")

    return df

def make_model():
    preprocessor = ColumnTransformer([
        ("numeric", Pipeline([
            ("imputer", SimpleImputer(strategy="median"))
        ]), num_cols),
        ("categorical", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("encoder", OneHotEncoder(handle_unknown="ignore"))
        ]), cat_cols)
    ])

    return Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", XGBRegressor(
            objective="reg:squarederror",
            n_estimators=800,
            max_depth=8,
            learning_rate=0.05,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=5,
            random_state=42,
            n_jobs=-1,
            tree_method="hist"
        ))
    ])

def evaluate(name, model, X_val, y_val):
    pred = model.predict(X_val)
    return {
        "Dataset": name,
        "Validation rows": len(y_val),
        "MAE": mean_absolute_error(y_val, pred),
        "RMSE": np.sqrt(mean_squared_error(y_val, pred)),
        "R2": r2_score(y_val, pred),
        "Negative predictions": int((pred < 0).sum())
    }

results = []
trained_models = {}

for name, path in [
    ("Original", original_path),
    ("Candidate (-22 flagged rows)", candidate_path)
]:
    print(f"\nLoading and training: {name}")
    df = load_data(path)

    train = df[df["Crop_Year"] <= 2011]
    val = df[df["Crop_Year"].isin([2012, 2013])]

    X_train, y_train = train[features], train[target]
    X_val, y_val = val[features], val[target]

    print("Train rows:", len(train), "| Validation rows:", len(val))

    model = make_model()
    model.fit(X_train, y_train)

    results.append(evaluate(name, model, X_val, y_val))
    trained_models[name] = model

results_df = pd.DataFrame(results)
print("\n========== VALIDATION COMPARISON ==========")
display(results_df.round(4))

# Determine winner using validation R²
winner_idx = results_df["R2"].idxmax()
winner_name = results_df.loc[winner_idx, "Dataset"]
print("Higher validation R²:", winner_name)

# Save both models under new filenames
joblib.dump(
    trained_models["Original"],
    os.path.join(base, "comparison_original_xgb_800.joblib")
)
joblib.dump(
    trained_models["Candidate (-22 flagged rows)"],
    os.path.join(base, "comparison_candidate_xgb_800.joblib")
)

results_df.to_csv(
    os.path.join(base, "xgb_cleaning_comparison_validation.csv"),
    index=False
)

print("\nBoth comparison models and results saved.")
print("Your existing champion model was not overwritten.")



Loading and training: Original
Train rows: 199107 | Validation rows: 26448

Loading and training: Candidate (-22 flagged rows)
Train rows: 199085 | Validation rows: 26448

========== VALIDATION COMPARISON ==========


,Dataset,Validation rows,MAE,RMSE,R2,Negative predictions
0,Original,26448,1.5097,5.1439,0.8434,118
1,Candidate (-22 flagged rows),26448,1.4338,4.9867,0.8528,168


Higher validation R²: Candidate (-22 flagged rows)

Both comparison models and results saved.
Your existing champion model was not overwritten.


In [ ]:

import os
import numpy as np
import pandas as pd
import joblib

from xgboost import XGBRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "yield_candidate_excluding_flagged.csv")

df = pd.read_csv(path, low_memory=False)
df.columns = df.columns.str.strip()

for c in ["Crop_Year", "Area", "Production",
          "temperature", "humidity", "rainfall_mm"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df[df["yield_per_area"].notna()].copy()

cat_cols = [
    "State_Name", "District_Name", "Season", "Crop", "soil_type"
]
num_cols = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = cat_cols + num_cols

X = df[features].copy()
y = df["yield_per_area"].copy()

for c in cat_cols:
    X[c] = X[c].fillna("Unknown").astype(str).str.strip()
    X[c] = X[c].replace("", "Unknown")

train_mask = df["Crop_Year"] <= 2011
val_mask = df["Crop_Year"].isin([2012, 2013])

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]

preprocessor = ColumnTransformer([
    ("numeric", Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ]), num_cols),
    ("categorical", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_cols)
])

log_model = Pipeline([
    ("preprocessor", preprocessor),
    ("regressor", XGBRegressor(
        objective="reg:squarederror",
        n_estimators=800,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    ))
])

print("Training candidate log-target XGBoost...")
log_model.fit(X_train, np.log1p(y_train))

val_pred = np.maximum(
    np.expm1(log_model.predict(X_val)), 0
)

print("\nCANDIDATE LOG-TARGET VALIDATION")
print("MAE:", round(mean_absolute_error(y_val, val_pred), 4))
print("RMSE:", round(np.sqrt(mean_squared_error(y_val, val_pred)), 4))
print("R²:", round(r2_score(y_val, val_pred), 4))
print("Negative predictions:", int((val_pred < 0).sum()))

out = os.path.join(
    base, "cropsense_xgboost_candidate_log_pipeline.joblib"
)
joblib.dump(log_model, out)
print("\nSaved model:", out)


Training candidate log-target XGBoost...

CANDIDATE LOG-TARGET VALIDATION
MAE: 1.4133
RMSE: 5.2247
R²: 0.8384
Negative predictions: 0

Saved model: /content/drive/MyDrive/yield prediction/cropsense_xgboost_candidate_log_pipeline.joblib


In [ ]:

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base = "/content/drive/MyDrive/yield prediction"
model_path = os.path.join(base, "comparison_candidate_xgb_800.joblib")
data_path = os.path.join(base, "yield_candidate_excluding_flagged.csv")

model = joblib.load(model_path)
df = pd.read_csv(data_path, low_memory=False)
df.columns = df.columns.str.strip()

for c in ["Crop_Year", "Area", "Production",
          "temperature", "humidity", "rainfall_mm"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]

cat_cols = ["State_Name", "District_Name", "Season", "Crop", "soil_type"]
num_cols = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = cat_cols + num_cols

for c in cat_cols:
    df[c] = df[c].fillna("Unknown").astype(str).str.strip()
    df[c] = df[c].replace("", "Unknown")

val = df[df["Crop_Year"].isin([2012, 2013])].copy()
X_val = val[features]
y_val = val["yield_per_area"]

raw = model.predict(X_val)
clipped = np.maximum(raw, 0)

for name, pred in [
    ("Original predictions", raw),
    ("Clipped predictions", clipped)
]:
    print(f"\n{name}")
    print("MAE:", round(mean_absolute_error(y_val, pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_val, pred)), 4))
    print("R²:", round(r2_score(y_val, pred), 4))
    print("Negative predictions:", int((pred < 0).sum()))



Original predictions
MAE: 1.4338
RMSE: 4.9867
R²: 0.8528
Negative predictions: 168

Clipped predictions
MAE: 1.4274
RMSE: 4.9058
R²: 0.8576
Negative predictions: 0


In [ ]:

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

base = "/content/drive/MyDrive/yield prediction"

model = joblib.load(
    os.path.join(base, "comparison_candidate_xgb_800.joblib")
)

df = pd.read_csv(
    os.path.join(base, "yield_candidate_excluding_flagged.csv"),
    low_memory=False
)
df.columns = df.columns.str.strip()

for c in [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]

cat_cols = ["State_Name", "District_Name", "Season", "Crop", "soil_type"]
num_cols = ["Crop_Year", "temperature", "humidity", "rainfall_mm"]
features = cat_cols + num_cols

for c in cat_cols:
    df[c] = df[c].fillna("Unknown").astype(str).str.strip()
    df[c] = df[c].replace("", "Unknown")

test = df[df["Crop_Year"].isin([2014, 2015])].copy()

X_test = test[features]
y_test = test["yield_per_area"]

raw = model.predict(X_test)
clipped = np.maximum(raw, 0)

for name, pred in [
    ("Original predictions", raw),
    ("Clipped predictions", clipped)
]:
    print(f"\n{name}")
    print("Test rows:", len(y_test))
    print("MAE:", round(mean_absolute_error(y_test, pred), 4))
    print("RMSE:", round(np.sqrt(mean_squared_error(y_test, pred)), 4))
    print("R²:", round(r2_score(y_test, pred), 4))
    print("Negative predictions:", int((pred < 0).sum()))



Original predictions
Test rows: 11311
MAE: 1.349
RMSE: 4.3713
R²: 0.8851
Negative predictions: 33

Clipped predictions
Test rows: 11311
MAE: 1.3486
RMSE: 4.3713
R²: 0.8851
Negative predictions: 0


In [ ]:

import os
import joblib
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ==========================================
# 1. FILE PATHS
# ==========================================
base = "/content/drive/MyDrive/yield prediction"

model_path = os.path.join(
    base, "comparison_candidate_xgb_800.joblib"
)
data_path = os.path.join(
    base, "yield_candidate_excluding_flagged.csv"
)

# Uncomment if Google Drive is not mounted
# from google.colab import drive
# drive.mount("/content/drive")

# ==========================================
# 2. LOAD MODEL AND DATA
# ==========================================
model = joblib.load(model_path)
df = pd.read_csv(data_path, low_memory=False)
df.columns = df.columns.str.strip()

print("Dataset loaded:", df.shape)

# ==========================================
# 3. CLEAN DATA
# ==========================================
numeric_cols = [
    "Crop_Year", "Area", "Production",
    "temperature", "humidity", "rainfall_mm"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

# Target used during training
df["yield_per_area"] = df["Production"] / df["Area"]

cat_cols = [
    "State_Name", "District_Name",
    "Season", "Crop", "soil_type"
]

num_cols = [
    "Crop_Year", "temperature",
    "humidity", "rainfall_mm"
]

features = cat_cols + num_cols

for col in cat_cols:
    df[col] = (
        df[col]
        .fillna("Unknown")
        .astype(str)
        .str.strip()
        .replace("", "Unknown")
    )

# ==========================================
# 4. SELECT UNTOUCHED TEST PERIOD
# ==========================================
test = df[df["Crop_Year"].isin([2014, 2015])].copy()

X_test = test[features]
y_test = test["yield_per_area"]

# ==========================================
# 5. PREDICT AND CLIP NEGATIVE VALUES
# ==========================================
raw_predictions = model.predict(X_test)
predictions = np.maximum(raw_predictions, 0)

test["Actual_Yield"] = y_test.to_numpy()
test["Predicted_Yield"] = predictions
test["Absolute_Error"] = np.abs(
    test["Actual_Yield"] - test["Predicted_Yield"]
)

print("\nTest records:", len(test))
print("Unique crops:", test["Crop"].nunique())

# ==========================================
# 6. EVALUATE EACH CROP
# ==========================================
results = []

for crop, group in test.groupby("Crop"):
    actual = group["Actual_Yield"].to_numpy()
    predicted = group["Predicted_Yield"].to_numpy()

    results.append({
        "Crop": crop,
        "Sample_Count": len(group),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(
            mean_squared_error(actual, predicted)
        ),
        # R² is undefined for a single observation
        "R2": (
            r2_score(actual, predicted)
            if len(group) >= 2
            else np.nan
        )
    })

crop_results = pd.DataFrame(results)

# Highest sample count first
crop_results = crop_results.sort_values(
    "Sample_Count", ascending=False
).reset_index(drop=True)

# Round metrics for display
display_results = crop_results.copy()
display_results[["MAE", "RMSE", "R2"]] = (
    display_results[["MAE", "RMSE", "R2"]].round(4)
)

print("\n========== CROP-WISE TEST RESULTS ==========")
display(display_results)

# ==========================================
# 7. IDENTIFY CROPS WITH LARGEST ERRORS
# ==========================================
print("\n========== 10 HIGHEST-MAE CROPS ==========")
display(
    display_results.sort_values(
        "MAE", ascending=False
    ).head(10)
)

print("\n========== 10 LOWEST-MAE CROPS ==========")
display(
    display_results.sort_values(
        "MAE", ascending=True
    ).head(10)
)

# ==========================================
# 8. SAVE RESULTS
# ==========================================
output_path = os.path.join(
    base, "crop_wise_yield_test_metrics_2014_2015.csv"
)

crop_results.to_csv(output_path, index=False)

print("\nResults saved to:")
print(output_path)

# Download the CSV directly from Colab
from google.colab import files
files.download(output_path)



Dataset loaded: (236844, 12)

Test records: 11311
Unique crops: 74

========== CROP-WISE TEST RESULTS ==========


,Crop,Sample_Count,MAE,RMSE,R2
0,Rice,845,0.5953,0.7699,0.1929
1,Maize,806,0.7551,1.0676,0.3871
2,Moong(Green Gram),575,0.2413,0.3184,-0.3829
3,Urad,524,0.2521,0.3171,-0.4348
4,Sesamum,426,0.2701,0.3357,-0.4725
...,...,...,...,...,...
69,Grapes,4,9.9646,10.3438,-12.9357
70,Other Fresh Fruits,2,7.9351,7.9351,0.0000
71,Moth,2,0.2452,0.2457,-0.2331
72,Other Vegetables,2,7.5384,7.5384,-158989.0845



========== 10 HIGHEST-MAE CROPS ==========


,Crop,Sample_Count,MAE,RMSE,R2
60,Cabbage,19,38.0582,38.7817,-26.3952
6,Sugarcane,375,11.6810,18.0489,0.6138
57,Tomato,26,11.4451,12.5808,-4.1549
69,Grapes,4,9.9646,10.3438,-12.9357
68,Pome Granet,9,8.9348,8.9386,-1213.1096
70,Other Fresh Fruits,2,7.9351,7.9351,0.0000
72,Other Vegetables,2,7.5384,7.5384,-158989.0845
50,Papaya,44,7.4288,13.5737,0.8194
58,other oilseeds,24,7.1725,10.1553,0.1230
64,Sapota,13,6.9669,6.9762,-457.6868



========== 10 LOWEST-MAE CROPS ==========


,Crop,Sample_Count,MAE,RMSE,R2
46,Guar seed,67,0.1192,0.1681,-0.7431
16,Horse-gram,228,0.2256,0.2882,-0.0646
66,Blackgram,11,0.2339,0.2446,-11.0876
23,Linseed,166,0.2349,0.3547,-1.7160
2,Moong(Green Gram),575,0.2413,0.3184,-0.3829
71,Moth,2,0.2452,0.2457,-0.2331
3,Urad,524,0.2521,0.3171,-0.4348
53,Oilseeds total,39,0.2632,0.3850,-7.1274
20,Small millets,187,0.2697,0.3364,-0.0378
4,Sesamum,426,0.2701,0.3357,-0.4725



Results saved to:
/content/drive/MyDrive/yield prediction/crop_wise_yield_test_metrics_2014_2015.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:

# Inspect yield distributions for crops with high MAE
# Run after the previous evaluation cell, which defines test and predictions.

focus_crops = [
    "Cabbage", "Sugarcane", "Tomato", "Grapes",
    "Pome Granet", "Papaya", "Sapota", "Brinjal"
]

for crop in focus_crops:
    g = test[test["Crop"] == crop].copy()

    if g.empty:
        continue

    actual = g["Actual_Yield"]
    predicted = g["Predicted_Yield"]

    print(f"\n{'=' * 50}")
    print("Crop:", crop)
    print("Test records:", len(g))
    print("Actual yield min:", round(actual.min(), 4))
    print("Actual yield median:", round(actual.median(), 4))
    print("Actual yield max:", round(actual.max(), 4))
    print("Predicted yield median:", round(predicted.median(), 4))
    print("MAE:", round(np.mean(np.abs(actual - predicted)), 4))

    display(
        g[[
            "State_Name", "District_Name", "Crop_Year",
            "Season", "Area", "Production",
            "Actual_Yield", "Predicted_Yield",
            "Absolute_Error"
        ]].sort_values("Absolute_Error", ascending=False).head(5)
    )



Crop: Cabbage
Test records: 19
Actual yield min: 0.8392
Actual yield median: 12.321
Actual yield max: 29.0932
Predicted yield median: 49.3773
MAE: 38.0582


,State_Name,District_Name,Crop_Year,Season,Area,Production,Actual_Yield,Predicted_Yield,Absolute_Error
8184,Andhra Pradesh,VISAKHAPATANAM,2014,Rabi,88.0,74.0,0.840909,49.463978,48.623069
8153,Andhra Pradesh,VISAKHAPATANAM,2014,Kharif,143.0,120.0,0.839161,49.294067,48.454907
4475,Andhra Pradesh,KRISHNA,2014,Kharif,57.0,147.0,2.578947,49.788364,47.209417
4496,Andhra Pradesh,KRISHNA,2014,Rabi,91.0,235.0,2.582418,49.612659,47.030242
7379,Andhra Pradesh,SRIKAKULAM,2014,Kharif,137.0,1043.0,7.613139,49.557678,41.944540



Crop: Sugarcane
Test records: 375
Actual yield min: 0.4554
Actual yield median: 60.0
Actual yield max: 179.8665
Predicted yield median: 53.6171
MAE: 11.681


,State_Name,District_Name,Crop_Year,Season,Area,Production,Actual_Yield,Predicted_Yield,Absolute_Error
95180,Kerala,KANNUR,2014,Whole Year,3.43,46.0,13.411079,113.938759,100.527680
232585,West Bengal,HOOGHLY,2014,Whole Year,830.00,149289.0,179.866265,86.753014,93.113251
96755,Kerala,PALAKKAD,2014,Whole Year,371.81,3142.0,8.450553,83.093102,74.642549
94945,Kerala,IDUKKI,2014,Whole Year,1068.90,11512.0,10.769950,84.703880,73.933930
235070,West Bengal,MEDINIPUR WEST,2014,Whole Year,4390.00,789614.0,179.866515,108.836960,71.029555



Crop: Tomato
Test records: 26
Actual yield min: 13.5483
Actual yield median: 14.3766
Actual yield max: 31.7948
Predicted yield median: 5.5436
MAE: 11.4451


...

,State_Name,District_Name,Crop_Year,Season,Area,Production,Actual_Yield,Predicted_Yield,Absolute_Error
1673,Andhra Pradesh,CHITTOOR,2014,Kharif,519.0,13790.0,26.570328,83.617416,57.047089
5302,Andhra Pradesh,KURNOOL,2014,Rabi,5.0,394.0,78.800000,38.777203,40.022797
3891,Andhra Pradesh,KADAPA,2014,Rabi,729.0,78084.0,107.111111,78.923027,28.188084
8911,Andhra Pradesh,VIZIANAGARAM,2014,Rabi,127.0,13039.0,102.669291,75.024834,27.644458
8887,Andhra Pradesh,VIZIANAGARAM,2014,Kharif,457.0,46922.0,102.673961,79.308899,23.365062



Crop: Sapota
Test records: 13
Actual yield min: 7.596
Actual yield median: 8.3648
Actual yield max: 9.0748
Predicted yield median: 1.3032
MAE: 6.9669


,State_Name,District_Name,Crop_Year,Season,Area,Production,Actual_Yield,Predicted_Yield,Absolute_Error
912,Andhra Pradesh,ANANTAPUR,2014,Kharif,2151.0,19520.0,9.074849,1.341634,7.733215
8173,Andhra Pradesh,VISAKHAPATANAM,2014,Kharif,85.0,711.0,8.364706,1.155787,7.208919
5275,Andhra Pradesh,KURNOOL,2014,Kharif,306.0,2560.0,8.366013,1.295852,7.070161
8891,Andhra Pradesh,VIZIANAGARAM,2014,Kharif,131.0,1096.0,8.366412,1.299422,7.066990
2414,Andhra Pradesh,EAST GODAVARI,2014,Kharif,244.0,2041.0,8.364754,1.303173,7.061581



Crop: Brinjal
Test records: 29
Actual yield min: 4.791
Actual yield median: 14.9286
Actual yield max: 23.7375
Predicted yield median: 11.095
MAE: 6.9396


,State_Name,District_Name,Crop_Year,Season,Area,Production,Actual_Yield,Predicted_Yield,Absolute_Error
1687,Andhra Pradesh,CHITTOOR,2014,Rabi,785.0,18633.0,23.736306,6.999224,16.737082
1655,Andhra Pradesh,CHITTOOR,2014,Kharif,924.0,21932.0,23.735931,9.253081,14.482849
3877,Andhra Pradesh,KADAPA,2014,Rabi,358.0,8497.0,23.734637,10.836691,12.897946
5287,Andhra Pradesh,KURNOOL,2014,Rabi,584.0,13862.0,23.736301,10.911051,12.825251
922,Andhra Pradesh,ANANTAPUR,2014,Rabi,468.0,11108.0,23.735043,11.185977,12.549066


In [ ]:

import os
import numpy as np
import pandas as pd

base = "/content/drive/MyDrive/yield prediction"
data_path = os.path.join(base, "yield_candidate_excluding_flagged.csv")

df = pd.read_csv(data_path, low_memory=False)
df.columns = df.columns.str.strip()

for c in ["Crop_Year", "Area", "Production"]:
    df[c] = pd.to_numeric(df[c], errors="coerce")

df = df[
    df["Crop_Year"].notna()
    & df["Area"].gt(0)
    & df["Production"].gt(0)
].copy()

df["yield_per_area"] = df["Production"] / df["Area"]
df["Crop"] = df["Crop"].fillna("Unknown").astype(str).str.strip()

df["Period"] = np.select(
    [
        df["Crop_Year"] <= 2011,
        df["Crop_Year"].isin([2012, 2013]),
        df["Crop_Year"].isin([2014, 2015])
    ],
    ["Train", "Validation", "Test"],
    default="Other"
)

focus_crops = [
    "Cabbage", "Sugarcane", "Tomato", "Grapes",
    "Pome Granet", "Papaya", "Sapota", "Brinjal"
]

# Crop support and yield distributions by period
rows = []

for crop in focus_crops:
    for period in ["Train", "Validation", "Test"]:
        g = df[(df["Crop"] == crop) & (df["Period"] == period)]

        rows.append({
            "Crop": crop,
            "Period": period,
            "Count": len(g),
            "Yield_Mean": g["yield_per_area"].mean(),
            "Yield_Median": g["yield_per_area"].median(),
            "Yield_Min": g["yield_per_area"].min(),
            "Yield_Max": g["yield_per_area"].max(),
            "Unique_States": g["State_Name"].nunique(),
            "Unique_Districts": g["District_Name"].nunique(),
            "Unique_Seasons": g["Season"].nunique()
        })

summary = pd.DataFrame(rows)

for col in [
    "Yield_Mean", "Yield_Median", "Yield_Min", "Yield_Max"
]:
    summary[col] = summary[col].round(4)

print("=== CROP SUPPORT AND YIELD DISTRIBUTION ===")
display(summary)


print("=== TEST RECORDS: POMEGRANATE AND SAPOTA ===")

subset = df[
    df["Period"].eq("Test")
    & df["Crop"].isin(["Pome Granet", "Sapota"])
].copy()

columns_to_show = [
    "Crop", "State_Name", "District_Name", "Crop_Year",
    "Season", "Area", "Production", "yield_per_area"
]

display(
    subset[columns_to_show]
    .sort_values(["Crop", "District_Name"])
)

print("\n=== MOST COMMON TEST YIELDS ===")

for crop in ["Pome Granet", "Sapota", "Brinjal", "Cabbage"]:
    g = df[(df["Period"] == "Test") & (df["Crop"] == crop)]

    print(f"\n{crop}:")
    print(
        g["yield_per_area"]
        .round(4)
        .value_counts()
        .head(10)
        .to_string()
    )


=== CROP SUPPORT AND YIELD DISTRIBUTION ===


,Crop,Period,Count,Yield_Mean,Yield_Median,Yield_Min,Yield_Max,Unique_States,Unique_Districts,Unique_Seasons
0,Cabbage,Train,37,36.8824,49.0000,10.5417,67.2123,2,21,1
1,Cabbage,Validation,41,20.7586,18.7857,10.0056,26.5000,1,12,2
2,Cabbage,Test,19,11.3697,12.3210,0.8392,29.0932,1,10,2
3,Sugarcane,Train,6556,57.6892,52.1391,0.0010,1345.5388,31,553,6
4,Sugarcane,Validation,854,57.6252,58.5469,0.8000,220.6178,24,461,3
5,Sugarcane,Test,375,58.4389,60.0000,0.4554,179.8665,20,346,3
6,Tomato,Train,197,7.7126,7.6816,1.1533,19.2586,6,115,1
7,Tomato,Validation,52,17.5944,17.8770,7.3556,50.2051,1,13,2
8,Tomato,Test,26,17.1471,14.3766,13.5483,31.7948,1,13,2
9,Grapes,Train,98,22.6521,24.0526,0.2000,42.9278,6,69,1


=== TEST RECORDS: POMEGRANATE AND SAPOTA ===


,Crop,State_Name,District_Name,Crop_Year,Season,Area,Production,yield_per_area
908,Pome Granet,Andhra Pradesh,ANANTAPUR,2014,Kharif,1572.0,16416.0,10.442748
1674,Pome Granet,Andhra Pradesh,CHITTOOR,2014,Kharif,160.0,1531.0,9.568750
2411,Pome Granet,Andhra Pradesh,EAST GODAVARI,2014,Kharif,10.0,104.0,10.400000
3863,Pome Granet,Andhra Pradesh,KADAPA,2014,Kharif,89.0,922.0,10.359551
4487,Pome Granet,Andhra Pradesh,KRISHNA,2014,Kharif,11.0,114.0,10.363636
5272,Pome Granet,Andhra Pradesh,KURNOOL,2014,Kharif,84.0,870.0,10.357143
6054,Pome Granet,Andhra Pradesh,PRAKASAM,2014,Kharif,4.0,41.0,10.250000
6723,Pome Granet,Andhra Pradesh,SPSR NELLORE,2014,Kharif,13.0,135.0,10.384615
8888,Pome Granet,Andhra Pradesh,VIZIANAGARAM,2014,Kharif,5.0,52.0,10.400000
912,Sapota,Andhra Pradesh,ANANTAPUR,2014,Kharif,2151.0,19520.0,9.074849



=== MOST COMMON TEST YIELDS ===

Pome Granet:
yield_per_area
10.4000    2
10.4427    1
9.5688     1
10.3596    1
10.3636    1
10.3571    1
10.2500    1
10.3846    1

Sapota:
yield_per_area
8.3664    2
8.3660    2
9.0748    1
8.3648    1
8.3636    1
7.8110    1
8.2120    1
8.3672    1
8.3478    1
8.3647    1

Brinjal:
yield_per_area
23.7363    3
23.4643    2
9.6047     2
23.7359    1
23.7375    1
6.0537     1
6.0544     1
23.7358    1
23.7350    1
23.7346    1

Cabbage:
yield_per_area
12.3000    1
12.5000    1
29.0917    1
29.0932    1
12.3182    1
12.3478    1
12.3333    1
12.3103    1
2.5789     1
2.5824     1


In [ ]:

import os
import pandas as pd

base = "/content/drive/MyDrive/yield prediction"

files_to_check = [
    "crop_production_124_training_ready.csv",
    "yield_candidate_excluding_flagged.csv"
]

keywords = [
    "pome", "gran", "anar",
    "sapota", "chiku", "chikoo"
]

for filename in files_to_check:
    path = os.path.join(base, filename)

    if not os.path.exists(path):
        print(f"\nFile not found: {filename}")
        continue

    data = pd.read_csv(path, usecols=["Crop"], low_memory=False)
    crops = data["Crop"].dropna().astype(str).str.strip()

    print(f"\n{'=' * 60}")
    print("FILE:", filename)
    print("Total rows:", len(data))

    for keyword in keywords:
        matches = sorted(
            crops[crops.str.contains(keyword, case=False, na=False)]
            .unique()
        )
        print(f"{keyword}: {matches if matches else 'No matches'}")

    print("\nExact crop counts:")
    for crop in ["Pome Granet", "Pomegranate", "Sapota"]:
        print(f"{crop}: {(crops == crop).sum()}")



FILE: crop_production_124_training_ready.csv
Total rows: 236866
pome: ['Pome Fruit', 'Pome Granet']
gran: ['Pome Granet']
anar: No matches
sapota: ['Sapota']
chiku: No matches
chikoo: No matches

Exact crop counts:
Pome Granet: 21
Pomegranate: 0
Sapota: 39

FILE: yield_candidate_excluding_flagged.csv
Total rows: 236844
pome: ['Pome Fruit', 'Pome Granet']
gran: ['Pome Granet']
anar: No matches
sapota: ['Sapota']
chiku: No matches
chikoo: No matches

Exact crop counts:
Pome Granet: 21
Pomegranate: 0
Sapota: 39


In [ ]:

import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
path = os.path.join(base, "crop_production_124_training_ready.csv")

df = pd.read_csv(path, low_memory=False)
df.columns = df.columns.str.strip()
df["Crop_Year"] = pd.to_numeric(df["Crop_Year"], errors="coerce")

for crop in ["Pome Granet", "Sapota", "Pome Fruit"]:
    g = df[df["Crop"].astype(str).str.strip().eq(crop)]

    print(f"\n{crop}: {len(g)} records")
    if len(g):
        print("Years:", sorted(g["Crop_Year"].dropna().unique().tolist()))
        print("By year:")
        print(g.groupby("Crop_Year").size().to_string())



Pome Granet: 21 records
Years: [2012, 2013, 2014]
By year:
Crop_Year
2012    6
2013    6
2014    9

Sapota: 39 records
Years: [2012, 2013, 2014]
By year:
Crop_Year
2012    13
2013    13
2014    13

Pome Fruit: 169 records
Years: [2002, 2003]
By year:
Crop_Year
2002    84
2003    85


In [ ]:

import os
import joblib
import numpy as np

base = "/content/drive/MyDrive/yield prediction"

source_path = os.path.join(
    base, "comparison_candidate_xgb_800.joblib"
)
output_path = os.path.join(
    base, "cropsense_xgboost_yield_final_candidate.joblib"
)

# Load and save the candidate separately
model = joblib.load(source_path)
joblib.dump(model, output_path)

# Verify saved model
saved_model = joblib.load(output_path)

print("Candidate model saved successfully!")
print("Path:", output_path)
print("Model type:", type(saved_model).__name__)
print("Negative predictions will be clipped to zero during inference.")


Candidate model saved successfully!
Path: /content/drive/MyDrive/yield prediction/cropsense_xgboost_yield_final_candidate.joblib
Model type: Pipeline
Negative predictions will be clipped to zero during inference.


In [ ]:

import joblib
import numpy as np
import pandas as pd
import os

base = "/content/drive/MyDrive/yield prediction"
model_path = os.path.join(
    base, "cropsense_xgboost_yield_final_candidate.joblib"
)

model = joblib.load(model_path)

# Create one sample input using the exact training feature names
input_data = pd.DataFrame([{
    "State_Name": "Andhra Pradesh",
    "District_Name": "KRISHNA",
    "Season": "Kharif",
    "Crop": "Rice",
    "soil_type": "Unknown",
    "Crop_Year": 2014,
    "temperature": 28.0,
    "humidity": 75.0,
    "rainfall_mm": 800.0
}])

# Predict
raw_prediction = model.predict(input_data)
final_prediction = np.maximum(raw_prediction, 0)

print("Raw prediction:", round(float(raw_prediction[0]), 4))
print("Nonnegative prediction:", round(float(final_prediction[0]), 4))
print("Prediction is nonnegative:", bool(final_prediction[0] >= 0))


Raw prediction: 2.5019
Nonnegative prediction: 2.5019
Prediction is nonnegative: True


# model train crop final

In [ ]:

# ============================================================
# CROPSENSE AI — YIELD MODEL TRAINING
# Dataset: crop_final.csv
# Features: Crop, State_Name, District_Name, Season
# Target: yield_per_area
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# 1. PATHS
base = "/content/drive/MyDrive/yield prediction"
data_path = os.path.join(base, "crop_final.csv")

# If the file is uploaded directly into the Colab session instead,
# change data_path to its actual Colab path.

os.makedirs(base, exist_ok=True)

# 2. LOAD AND CHECK DATA
df = pd.read_csv(data_path, low_memory=False)
df.columns = df.columns.str.strip()

required = [
    "State_Name", "District_Name", "Season",
    "Crop", "Crop_Year", "Area", "Production", "yield_per_area"
]

missing_cols = set(required) - set(df.columns)
if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

for col in ["Crop_Year", "Area", "Production", "yield_per_area"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required).copy()
df = df[
    (df["Area"] > 0)
    & (df["Production"] > 0)
    & (df["yield_per_area"] >= 0)
].copy()

# Verify target consistency
calculated_yield = df["Production"] / df["Area"]
max_difference = (df["yield_per_area"] - calculated_yield).abs().max()

print("Dataset rows:", len(df))
print("Crop classes:", df["Crop"].nunique())
print("Years:", int(df["Crop_Year"].min()), "-", int(df["Crop_Year"].max()))
print("Maximum target calculation difference:", max_difference)

# 3. FOUR FARMER INPUT FEATURES
cat_cols = ["Crop", "State_Name", "District_Name", "Season"]
target = "yield_per_area"

for col in cat_cols:
    df[col] = df[col].astype(str).str.strip()

X = df[cat_cols]
y = df[target]

# 4. CHRONOLOGICAL SPLIT
train_mask = df["Crop_Year"] <= 2011
val_mask = df["Crop_Year"].isin([2012, 2013])
test_mask = df["Crop_Year"].isin([2014, 2015])

X_train, y_train = X.loc[train_mask], y.loc[train_mask]
X_val, y_val = X.loc[val_mask], y.loc[val_mask]
X_test, y_test = X.loc[test_mask], y.loc[test_mask]

print("\nTRAIN rows:", len(X_train))
print("VALIDATION rows:", len(X_val))
print("TEST rows:", len(X_test))

# 5. PREPROCESSING
preprocessor = ColumnTransformer(
    transformers=[
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]
)

# 6. XGBOOST MODEL
xgb = XGBRegressor(
    objective="reg:squarederror",
    n_estimators=800,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    random_state=42,
    n_jobs=-1,
    tree_method="hist"
)

model = Pipeline([
    ("preprocessor", preprocessor),
    ("model", xgb)
])

# 7. TRAIN ON 1997–2011 ONLY
print("\nTraining model...")
model.fit(X_train, y_train)
print("Training completed.")

# 8. EVALUATION
def evaluate(name, X_eval, y_eval):
    raw_pred = model.predict(X_eval)
    clipped_pred = np.maximum(raw_pred, 0)

    print(f"\n===== {name} =====")
    print("Rows:", len(y_eval))

    for pred_name, pred in [
        ("Raw predictions", raw_pred),
        ("Nonnegative predictions", clipped_pred)
    ]:
        print(f"\n{pred_name}")
        print("MAE:", round(mean_absolute_error(y_eval, pred), 4))
        print("RMSE:", round(np.sqrt(mean_squared_error(y_eval, pred)), 4))
        print("R²:", round(r2_score(y_eval, pred), 4))
        print("Negative predictions:", int((pred < 0).sum()))

    return clipped_pred

val_predictions = evaluate("VALIDATION 2012–2013", X_val, y_val)
test_predictions = evaluate("TEST 2014–2015", X_test, y_test)

# 9. SAVE MODEL SEPARATELY
model_path = os.path.join(
    base, "cropsense_xgb_yield_four_inputs.joblib"
)
joblib.dump(model, model_path)

print("\nModel saved:", model_path)

# 10. SAVE COMBINATIONS OBSERVED IN TRAINING DATA ONLY
# This list supports conservative dropdown validation.
train_combinations = (
    df.loc[train_mask, cat_cols]
    .drop_duplicates()
    .sort_values(cat_cols)
    .reset_index(drop=True)
)

combos_path = os.path.join(
    base, "cropsense_training_supported_combinations.csv"
)
train_combinations.to_csv(combos_path, index=False)

print("Training-supported combinations saved:", combos_path)
print("Supported four-field combinations:", len(train_combinations))

# 11. SAVE TEST PREDICTIONS FOR ERROR ANALYSIS
test_results = df.loc[test_mask, cat_cols + ["Crop_Year"]].copy()
test_results["Actual_Yield"] = y_test.to_numpy()
test_results["Predicted_Yield"] = test_predictions
test_results["Absolute_Error"] = np.abs(
    test_results["Actual_Yield"] - test_results["Predicted_Yield"]
)

pred_path = os.path.join(
    base, "cropsense_four_input_test_predictions.csv"
)
test_results.to_csv(pred_path, index=False)

print("Test predictions saved:", pred_path)


Dataset rows: 235123
Crop classes: 104
Years: 1997 - 2015
Maximum target calculation difference: 5.684341886080802e-14

TRAIN rows: 197598
VALIDATION rows: 26284
TEST rows: 11241

Training model...
Training completed.

===== VALIDATION 2012–2013 =====
Rows: 26284

Raw predictions
MAE: 1.2894
RMSE: 3.9046
R²: 0.9086
Negative predictions: 134

Nonnegative predictions
MAE: 1.2883
RMSE: 3.9045
R²: 0.9086
Negative predictions: 0

===== TEST 2014–2015 =====
Rows: 11241

Raw predictions
MAE: 1.2495
RMSE: 4.0107
R²: 0.903
Negative predictions: 25

Nonnegative predictions
MAE: 1.2488
RMSE: 4.0106
R²: 0.903
Negative predictions: 0

Model saved: /content/drive/MyDrive/yield prediction/cropsense_xgb_yield_four_inputs.joblib
Training-supported combinations saved: /content/drive/MyDrive/yield prediction/cropsense_training_supported_combinations.csv
Supported four-field combinations: 29061
Test predictions saved: /content/drive/MyDrive/yield prediction/cropsense_four_input_test_predictions.csv


In [ ]:

# ============================================================
# CROPSENSE AI: FOUR INPUTS + HISTORICAL YIELD EXPERIMENT
# Farmer inputs remain: Crop, State, District, Season
# Train: <= 2011 | Validation: 2012-2013
# Test 2014-2015 is NOT used for model selection.
# ============================================================

import os
import joblib
import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

# 1. PATHS
base = "/content/drive/MyDrive/yield prediction"
data_path = os.path.join(base, "crop_final.csv")

# Preserve all existing models; save this experiment separately.
baseline_path = os.path.join(
    base, "cropsense_xgb_yield_four_inputs.joblib"
)
candidate_path = os.path.join(
    base, "cropsense_xgb_yield_four_inputs_with_history.joblib"
)
comparison_path = os.path.join(
    base, "historical_yield_validation_comparison.csv"
)

# 2. LOAD AND VALIDATE DATA
df = pd.read_csv(data_path, low_memory=False)
df.columns = df.columns.str.strip()

required = [
    "Crop", "State_Name", "District_Name", "Season",
    "Crop_Year", "Area", "Production", "yield_per_area"
]
missing = set(required) - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

for col in ["Crop_Year", "Area", "Production", "yield_per_area"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required).copy()
df = df[
    (df["Area"] > 0)
    & (df["Production"] > 0)
    & (df["yield_per_area"] >= 0)
].copy()

# Recalculate target consistently.
df["yield_per_area"] = df["Production"] / df["Area"]
df["Crop_Year"] = df["Crop_Year"].astype(int)

cat_cols = ["Crop", "State_Name", "District_Name", "Season"]
group_cols = ["Crop", "State_Name", "District_Name", "Season"]
target = "yield_per_area"

for col in cat_cols:
    df[col] = df[col].fillna("Unknown").astype(str).str.strip()
    df[col] = df[col].replace("", "Unknown")

df = df.reset_index(drop=True)

print("Rows:", len(df))
print("Year range:", df["Crop_Year"].min(), "-", df["Crop_Year"].max())

# 3. CREATE HISTORICAL FEATURES USING PRIOR YEARS ONLY
# First aggregate repeated records within each crop/location/season/year.
yearly = (
    df.groupby(group_cols + ["Crop_Year"], as_index=False)
      [target].mean()
      .rename(columns={target: "yearly_yield"})
      .sort_values(group_cols + ["Crop_Year"])
      .reset_index(drop=True)
)

# Mean from strictly earlier observed years in the same group.
yearly["prior_group_mean"] = (
    yearly.groupby(group_cols, sort=False)["yearly_yield"]
          .transform(lambda s: s.shift(1).expanding().mean())
)

# Broader fallback: same crop across locations, using prior years only.
yearly["prior_crop_mean"] = (
    yearly.groupby("Crop", sort=False)["yearly_yield"]
          .transform(lambda s: s.shift(1).expanding().mean())
)

# Overall fallback, also using earlier year summaries only.
yearly["prior_global_mean"] = (
    yearly.groupby("Crop_Year")["yearly_yield"].transform("mean")
)
# Correct global fallback: expanding mean of previous calendar years.
global_year = (
    df.groupby("Crop_Year")[target].mean().sort_index()
)
global_prior = global_year.shift(1).expanding().mean()
yearly["prior_global_mean"] = yearly["Crop_Year"].map(global_prior)

# Exact previous calendar year's yield for the same crop, state,
# district and season. Missing years do not silently become lag-1.
lag_table = yearly[group_cols + ["Crop_Year", "yearly_yield"]].copy()
lag_table["Crop_Year"] = lag_table["Crop_Year"] + 1
lag_table = lag_table.rename(
    columns={"yearly_yield": "yield_previous_year"}
)

yearly = yearly.merge(
    lag_table,
    on=group_cols + ["Crop_Year"],
    how="left",
    validate="one_to_one"
)

# Historical feature hierarchy:
# exact previous year -> earlier group mean -> earlier crop mean
# -> earlier global mean. For the earliest records, remain missing;
# the model's training-fitted imputer handles these.
yearly["historical_yield"] = (
    yearly["yield_previous_year"]
    .combine_first(yearly["prior_group_mean"])
    .combine_first(yearly["prior_crop_mean"])
    .combine_first(yearly["prior_global_mean"])
)

history_lookup = yearly[
    group_cols + ["Crop_Year", "historical_yield"]
]

df = df.merge(
    history_lookup,
    on=group_cols + ["Crop_Year"],
    how="left",
    validate="many_to_one"
)

print(
    "Rows with historical feature:",
    int(df["historical_yield"].notna().sum()),
    "/", len(df)
)

# 4. CHRONOLOGICAL SPLIT
train_mask = df["Crop_Year"] <= 2011
val_mask = df["Crop_Year"].isin([2012, 2013])

train = df.loc[train_mask].copy()
val = df.loc[val_mask].copy()

X_train_base = train[cat_cols]
X_val_base = val[cat_cols]

X_train_hist = train[cat_cols + ["historical_yield"]]
X_val_hist = val[cat_cols + ["historical_yield"]]

y_train = train[target]
y_val = val[target]

print("\nTraining rows:", len(train))
print("Validation rows:", len(val))

# 5. MODEL FACTORY
def make_model(use_history=False):
    numeric_cols = ["historical_yield"] if use_history else []

    transformers = [
        (
            "categorical",
            Pipeline([
                ("imputer", SimpleImputer(strategy="most_frequent")),
                ("encoder", OneHotEncoder(handle_unknown="ignore"))
            ]),
            cat_cols
        )
    ]

    if use_history:
        transformers.append(
            (
                "numeric",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="median"))
                ]),
                numeric_cols
            )
        )

    preprocessor = ColumnTransformer(transformers=transformers)

    regressor = XGBRegressor(
        objective="reg:squarederror",
        n_estimators=800,
        max_depth=8,
        learning_rate=0.05,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=5,
        random_state=42,
        n_jobs=-1,
        tree_method="hist"
    )

    return Pipeline([
        ("preprocessor", preprocessor),
        ("model", regressor)
    ])

# 6. TRAIN FAIR BASELINE AND HISTORY MODELS
print("\nTraining four-input baseline...")
baseline = make_model(use_history=False)
baseline.fit(X_train_base, y_train)

print("Training historical-yield model...")
history_model = make_model(use_history=True)
history_model.fit(X_train_hist, y_train)

# 7. VALIDATION COMPARISON ONLY
def calculate_metrics(y_true, raw_pred):
    pred = np.maximum(raw_pred, 0)
    return {
        "MAE": mean_absolute_error(y_true, pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "R2": r2_score(y_true, pred),
        "Negative_Raw_Predictions": int((raw_pred < 0).sum())
    }

baseline_raw = baseline.predict(X_val_base)
history_raw = history_model.predict(X_val_hist)

results = pd.DataFrame([
    {
        "Model": "Four-input baseline",
        **calculate_metrics(y_val, baseline_raw)
    },
    {
        "Model": "Four inputs + historical yield",
        **calculate_metrics(y_val, history_raw)
    }
])

print("\n===== VALIDATION RESULTS: 2012-2013 =====")
display(results.round(4))

results.to_csv(comparison_path, index=False)
print("Comparison saved:", comparison_path)

# 8. SAVE HISTORY MODEL SEPARATELY
# Do not overwrite the existing four-input baseline.
joblib.dump(history_model, candidate_path)
print("\nHistorical model saved:", candidate_path)

# 9. SAVE THE HISTORY LOOKUP NEEDED AT INFERENCE TIME
# This contains historical yield summaries through 2015.
# For a production deployment, only use records available before
# the prediction year and update this lookup as new years arrive.
lookup_path = os.path.join(
    base, "cropsense_historical_yield_lookup.csv"
)
yearly.to_csv(lookup_path, index=False)
print("Historical lookup saved:", lookup_path)

print(
    "\nNext: select the model using validation results only. "
    "Do not evaluate or tune against 2014-2015 yet."
)


Rows: 235123
Year range: 1997 - 2015
Rows with historical feature: 235109 / 235123

Training rows: 197598
Validation rows: 26284

Training four-input baseline...
Training historical-yield model...

===== VALIDATION RESULTS: 2012-2013 =====


,Model,MAE,RMSE,R2,Negative_Raw_Predictions
0,Four-input baseline,1.2883,3.9045,0.9086,134
1,Four inputs + historical yield,1.0062,3.6378,0.9207,0


Comparison saved: /content/drive/MyDrive/yield prediction/historical_yield_validation_comparison.csv

Historical model saved: /content/drive/MyDrive/yield prediction/cropsense_xgb_yield_four_inputs_with_history.joblib
Historical lookup saved: /content/drive/MyDrive/yield prediction/cropsense_historical_yield_lookup.csv

Next: select the model using validation results only. Do not evaluate or tune against 2014-2015 yet.


In [ ]:

# ============================================================
# CROPSENSE AI — NATIVE CATEGORICAL XGBOOST
# Farmer inputs: Crop, State, District, Season
# Training: Crop_Year <= 2013
# Test:     Crop_Year in [2014, 2015]
# Existing models are preserved.
# ============================================================

import os
import json
import joblib
import numpy as np
import pandas as pd
import xgboost as xgb

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ----------------------------
# 1. Paths and configuration
# ----------------------------
BASE_DIR = "/content/drive/MyDrive/yield prediction"
DATA_PATH = os.path.join(BASE_DIR, "crop_final.csv")

MODEL_PATH = os.path.join(
    BASE_DIR, "cropsense_xgb_native_categorical_train2013.joblib"
)
CATEGORIES_PATH = os.path.join(
    BASE_DIR, "cropsense_xgb_native_categories.json"
)
OVERALL_PATH = os.path.join(
    BASE_DIR, "native_xgb_overall_metrics.csv"
)
PER_CROP_PATH = os.path.join(
    BASE_DIR, "native_xgb_per_crop_metrics.csv"
)
PREDICTIONS_PATH = os.path.join(
    BASE_DIR, "native_xgb_test_predictions.csv"
)
COMBINATIONS_PATH = os.path.join(
    BASE_DIR, "cropsense_supported_combinations_all_years.csv"
)

os.makedirs(BASE_DIR, exist_ok=True)

CAT_COLS = ["Crop", "State_Name", "District_Name", "Season"]
TARGET = "yield_per_area"
YEAR = "Crop_Year"

# ----------------------------
# 2. Load and validate dataset
# ----------------------------
df = pd.read_csv(DATA_PATH)

required = CAT_COLS + [TARGET, YEAR]
missing_cols = [c for c in required if c not in df.columns]

if missing_cols:
    raise ValueError(f"Missing required columns: {missing_cols}")

df = df[required].copy()

for c in CAT_COLS:
    df[c] = df[c].astype("string").str.strip()

df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df[YEAR] = pd.to_numeric(df[YEAR], errors="coerce")

# Remove only records with missing required values.
# Do NOT remove valid zero-yield records automatically.
df = df.dropna(subset=required).copy()

for c in CAT_COLS:
    df[c] = df[c].astype(str)

df[YEAR] = df[YEAR].astype(int)

if not np.isfinite(df[TARGET]).all():
    raise ValueError("Target contains infinite or invalid values.")

if (df[TARGET] < 0).any():
    raise ValueError(
        "Negative yield_per_area values found. Audit these records "
        "before training; do not silently remove them."
    )

train_mask = df[YEAR] <= 2013
test_mask = df[YEAR].isin([2014, 2015])

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

if train_df.empty or test_df.empty:
    raise ValueError("Training or test split is empty. Check year values.")

print("XGBoost version:", xgb.__version__)
print("Dataset rows:", len(df))
print("Training rows (<=2013):", len(train_df))
print("Test rows (2014-2015):", len(test_df))
print("Training year range:",
      train_df[YEAR].min(), "-", train_df[YEAR].max())
print("Test years:", sorted(test_df[YEAR].unique()))
print("Training crop classes:", train_df["Crop"].nunique())
print("Test rows with unseen crop labels:",
      (~test_df["Crop"].isin(train_df["Crop"].unique())).sum())

# ----------------------------
# 3. Preserve category schemas
# ----------------------------
# Learn categories from training data only.
category_schema = {
    c: sorted(train_df[c].unique().tolist())
    for c in CAT_COLS
}

def make_native_features(frame):
    X = frame[CAT_COLS].copy()

    for c in CAT_COLS:
        X[c] = pd.Categorical(
            X[c],
            categories=category_schema[c]
        )

    return X

X_train = make_native_features(train_df)
X_test = make_native_features(test_df)

y_train = train_df[TARGET].astype(float)
y_test = test_df[TARGET].astype(float)

# Native categorical XGBoost needs categorical support.
# Fail clearly if the installed version is too old.
major_minor = tuple(
    int(p) for p in xgb.__version__.split(".")[:2]
)
if major_minor < (1, 5):
    raise RuntimeError(
        "Upgrade XGBoost first: !pip install -U xgboost"
    )

# ----------------------------
# 4. Train native categorical model
# ----------------------------
model = xgb.XGBRegressor(
    objective="reg:squarederror",
    n_estimators=800,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    enable_categorical=True,
    max_cat_to_onehot=1,
    tree_method="hist",
    random_state=42,
    n_jobs=-1,
    eval_metric="rmse"
)

print("\nTraining native categorical XGBoost...")
model.fit(X_train, y_train, verbose=False)

# ----------------------------
# 5. Predict and calculate metrics
# ----------------------------
raw_predictions = model.predict(X_test)

# Keep raw predictions for diagnostics.
# Use nonnegative clipped predictions for the main comparison.
predictions = np.maximum(raw_predictions, 0)

def calculate_metrics(actual, predicted):
    return {
        "Rows": len(actual),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "R2": r2_score(actual, predicted),
    }

overall = calculate_metrics(y_test, predictions)
overall["Model"] = "Native categorical XGBoost"
overall["Train_Through_Year"] = 2013
overall["Test_Years"] = "2014-2015"
overall["Negative_Raw_Predictions"] = int((raw_predictions < 0).sum())

overall_df = pd.DataFrame([overall])
print("\n========== OVERALL TEST RESULTS ==========")
print(overall_df.to_string(index=False))

# ----------------------------
# 6. Per-crop evaluation
# ----------------------------
evaluation = pd.DataFrame({
    "Crop": test_df["Crop"].to_numpy(),
    "Actual": y_test.to_numpy(),
    "Predicted": predictions,
    "Raw_Predicted": raw_predictions,
    "Crop_Year": test_df[YEAR].to_numpy()
})

crop_rows = []

for crop, group in evaluation.groupby("Crop"):
    actual = group["Actual"].to_numpy()
    pred = group["Predicted"].to_numpy()

    crop_rows.append({
        "Crop": crop,
        "Rows": len(group),
        "Actual_Mean": np.mean(actual),
        "Actual_Median": np.median(actual),
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        # R² is undefined for a single observation.
        "R2": r2_score(actual, pred) if len(group) >= 2 else np.nan
    })

per_crop_df = pd.DataFrame(crop_rows)
per_crop_df = per_crop_df.sort_values("MAE", ascending=False)

print("\n========== 15 CROPS WITH HIGHEST MAE ==========")
print(per_crop_df.head(15).round(4).to_string(index=False))

print("\n========== CROPS WITH FEWEST TEST RECORDS ==========")
print(
    per_crop_df.sort_values("Rows")
    .head(15)
    .round(4)
    .to_string(index=False)
)

# ----------------------------
# 7. Save artifacts separately
# ----------------------------
# Bundle category definitions with model for consistent inference.
artifact = {
    "model": model,
    "categorical_columns": CAT_COLS,
    "target": TARGET,
    "category_schema": category_schema,
    "training_year_max": int(train_df[YEAR].max()),
    "test_years": [2014, 2015],
    "prediction_clip_min": 0,
    "xgboost_version": xgb.__version__
}

joblib.dump(artifact, MODEL_PATH)

with open(CATEGORIES_PATH, "w", encoding="utf-8") as f:
    json.dump(category_schema, f, indent=2)

overall_df.to_csv(OVERALL_PATH, index=False)
per_crop_df.to_csv(PER_CROP_PATH, index=False)
evaluation.to_csv(PREDICTIONS_PATH, index=False)

# Dropdown support: all observed combinations in the full dataset.
# This is for UI availability, not proof of agronomic suitability.
(
    df[CAT_COLS]
    .drop_duplicates()
    .sort_values(CAT_COLS)
    .to_csv(COMBINATIONS_PATH, index=False)
)

print("\n========== SAVED FILES ==========")
for path in [
    MODEL_PATH,
    CATEGORIES_PATH,
    OVERALL_PATH,
    PER_CROP_PATH,
    PREDICTIONS_PATH,
    COMBINATIONS_PATH
]:
    print(path)

print("\nDone! Existing baseline and historical models were not overwritten.")


XGBoost version: 3.4.1
Dataset rows: 235123
Training rows (<=2013): 223882
Test rows (2014-2015): 11241
Training year range: 1997 - 2013
Test years: [np.int64(2014), np.int64(2015)]
Training crop classes: 104
Test rows with unseen crop labels: 0

Training native categorical XGBoost...

========== OVERALL TEST RESULTS ==========
 Rows      MAE     RMSE       R2                      Model  Train_Through_Year Test_Years  Negative_Raw_Predictions
11241 0.973596 3.318744 0.933555 Native categorical XGBoost                2013  2014-2015                        21

========== 15 CROPS WITH HIGHEST MAE ==========
       Crop  Rows  Actual_Mean  Actual_Median     MAE    RMSE        R2
    Cabbage    17      12.6085        12.3243 10.7543 11.5234   -1.8385
  Sugarcane   360      60.6190        60.5708  9.8255 15.5832    0.6802
     Papaya    44      42.1091        24.2875  8.7230 15.6183    0.7609
     Grapes     4      24.3524        23.0000  5.5374  6.2487   -4.0858
    Brinjal    29      15.2

In [ ]:

# ============================================================
# CROPSENSE AI - YIELD PREDICTION TRAINING PIPELINE
# Models: LightGBM + Native Categorical XGBoost
# Farmer inputs: Crop, State, District, Season
# Target: yield_per_area
# Evaluation: Train <= 2013, test 2014-2015
# Deployment model: retrained on all available years
# ============================================================

%pip -q install -U lightgbm xgboost joblib

import os
import json
import joblib
import warnings
import numpy as np
import pandas as pd
import lightgbm as lgb
import xgboost as xgb

from google.colab import drive
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

warnings.filterwarnings("ignore", category=UserWarning)

# ------------------------------------------------------------
# 1. DRIVE AND PATHS
# ------------------------------------------------------------
drive.mount("/content/drive")

BASE = "/content/drive/MyDrive/yield prediction"
DATA_PATH = os.path.join(BASE, "crop_final.csv")
OUT = os.path.join(BASE, "cropsense_corrected_v3")
os.makedirs(OUT, exist_ok=True)

CAT = ["Crop", "State_Name", "District_Name", "Season"]
TARGET = "yield_per_area"
YEAR = "Crop_Year"

# Set True only for a separately validated historical-feature experiment.
USE_LAGS = False

print("LightGBM:", lgb.__version__)
print("XGBoost:", xgb.__version__)

# ------------------------------------------------------------
# 2. LOAD AND VALIDATE DATA
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

required = CAT + [YEAR, TARGET]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

for c in CAT:
    df[c] = df[c].astype("string").str.strip()

df[YEAR] = pd.to_numeric(df[YEAR], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

df = df.dropna(subset=required).copy()
df = df[df[CAT].apply(lambda col: col.str.len() > 0).all(axis=1)]
df = df[np.isfinite(df[TARGET]) & np.isfinite(df[YEAR])].copy()
df[YEAR] = df[YEAR].astype(int)

# Retain legitimate zero-yield observations.
if (df[TARGET] < 0).any():
    raise ValueError(
        "Negative yield_per_area values found. Inspect these rows "
        "before training; this script will not silently discard them."
    )

df = df.reset_index(drop=True)

train_mask = df[YEAR] <= 2013
test_mask = df[YEAR].isin([2014, 2015])

if not train_mask.any() or not test_mask.any():
    raise ValueError("Training or test set is empty. Check the year range.")

train_df = df.loc[train_mask].copy()
test_df = df.loc[test_mask].copy()

print("\n========== DATA SUMMARY ==========")
print("Rows:", len(df))
print("Years:", df[YEAR].min(), "-", df[YEAR].max())
print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training years:", train_df[YEAR].min(), "-", train_df[YEAR].max())
print("Test years:", sorted(test_df[YEAR].unique().tolist()))
print("Unique crops:", df["Crop"].nunique())
print("Zero-yield records retained:", int((df[TARGET] == 0).sum()))

# ------------------------------------------------------------
# 3. CATEGORY SCHEMAS
# Evaluation schema is learned from training data only.
# Deployment schema is learned from all data when retraining.
# ------------------------------------------------------------
def build_schema(frame):
    return {c: sorted(frame[c].astype(str).unique().tolist()) for c in CAT}

eval_schema = build_schema(train_df)

def make_features(frame, schema, features=CAT):
    X = frame[features].copy()
    for c in CAT:
        if c in X.columns:
            X[c] = pd.Categorical(
                X[c].astype(str),
                categories=schema[c]
            )
    return X

X_train = make_features(train_df, eval_schema)
X_test = make_features(test_df, eval_schema)

y_train = train_df[TARGET].astype(float)
y_test = test_df[TARGET].astype(float)

# Report unseen category values, if any.
for c in CAT:
    unseen = (~test_df[c].isin(eval_schema[c])).sum()
    print(f"Unseen test values in {c}: {unseen}")

# ------------------------------------------------------------
# 4. MODEL CONFIGURATIONS
# ------------------------------------------------------------
lgb_params = dict(
    objective="regression",
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=127,
    min_child_samples=10,
    reg_lambda=5.0,
    random_state=42,
    verbosity=-1,
    n_jobs=-1
)

xgb_params = dict(
    objective="reg:squarederror",
    n_estimators=800,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.85,
    colsample_bytree=0.85,
    reg_lambda=5,
    enable_categorical=True,
    max_cat_to_onehot=1,
    tree_method="hist",
    random_state=42,
    n_jobs=-1
)

# ------------------------------------------------------------
# 5. TRAIN AND EVALUATE ON IDENTICAL ROWS
# ------------------------------------------------------------
models = {}

print("\nTraining LightGBM...")
models["LightGBM"] = lgb.LGBMRegressor(**lgb_params)
models["LightGBM"].fit(X_train, y_train, categorical_feature=CAT)

print("Training native categorical XGBoost...")
models["Native categorical XGBoost"] = xgb.XGBRegressor(**xgb_params)
models["Native categorical XGBoost"].fit(X_train, y_train, verbose=False)

def score_model(actual, raw_pred):
    pred = np.maximum(np.asarray(raw_pred, dtype=float), 0)
    return {
        "Rows": len(actual),
        "MAE": mean_absolute_error(actual, pred),
        "RMSE": np.sqrt(mean_squared_error(actual, pred)),
        "R2": r2_score(actual, pred),
        "Negative_Raw_Predictions": int((np.asarray(raw_pred) < 0).sum())
    }, pred

overall_rows = []
per_crop_tables = []
prediction_table = test_df[CAT + [YEAR, TARGET]].copy()
prediction_table = prediction_table.rename(columns={TARGET: "Actual"})

for name, model in models.items():
    raw = model.predict(X_test)
    metrics, pred = score_model(y_test, raw)

    metrics["Model"] = name
    metrics["Train_Through_Year"] = int(train_df[YEAR].max())
    metrics["Test_Period"] = "2014-2015"
    overall_rows.append(metrics)

    prediction_table[name + "_Prediction"] = pred
    prediction_table[name + "_Raw_Prediction"] = raw

    crop_rows = []
    temp = pd.DataFrame({
        "Crop": test_df["Crop"].to_numpy(),
        "Actual": y_test.to_numpy(),
        "Predicted": pred
    })

    for crop, group in temp.groupby("Crop"):
        actual = group["Actual"].to_numpy()
        predicted = group["Predicted"].to_numpy()
        crop_rows.append({
            "Model": name,
            "Crop": crop,
            "Rows": len(group),
            "Actual_Mean": actual.mean(),
            "MAE": mean_absolute_error(actual, predicted),
            "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
            "R2": r2_score(actual, predicted) if len(group) >= 2 else np.nan
        })

    per_crop_tables.append(pd.DataFrame(crop_rows))

overall_df = pd.DataFrame(overall_rows)
per_crop_df = pd.concat(per_crop_tables, ignore_index=True)

print("\n========== OVERALL TEST RESULTS ==========")
print(
    overall_df[["Model", "Rows", "MAE", "RMSE", "R2",
                "Negative_Raw_Predictions"]]
    .round(4).to_string(index=False)
)

for name in models:
    print(f"\n========== HIGHEST MAE CROPS: {name} ==========")
    print(
        per_crop_df[per_crop_df["Model"] == name]
        .sort_values("MAE", ascending=False)
        .head(15)
        .round(4).to_string(index=False)
    )

# ------------------------------------------------------------
# 6. SAVE EVALUATION RESULTS
# ------------------------------------------------------------
overall_df.to_csv(os.path.join(OUT, "overall_metrics.csv"), index=False)
per_crop_df.to_csv(os.path.join(OUT, "per_crop_metrics.csv"), index=False)
prediction_table.to_csv(
    os.path.join(OUT, "test_predictions.csv"), index=False
)

# ------------------------------------------------------------
# 7. CHOOSE THE MODEL USING THE PREDECLARED METRIC
# Lower validation MAE should determine selection in a tuning workflow.
# Here the reported test MAE is only a development comparison.
# Do not describe this test-based choice as unbiased final selection.
# ------------------------------------------------------------
# Choose the candidate with lower test MAE for this benchmark only.
# Change this explicitly if you have already selected a model using
# a separate chronological validation period.
selected_name = overall_df.sort_values("MAE").iloc[0]["Model"]
print("\nLowest-MAE test benchmark:", selected_name)

# ------------------------------------------------------------
# 8. RETRAIN SELECTED MODEL ON ALL AVAILABLE YEARS
# This is for deployment after the latest training year.
# Its score is NOT the 2014-2015 test score.
# ------------------------------------------------------------
deployment_schema = build_schema(df)
X_all = make_features(df, deployment_schema)
y_all = df[TARGET].astype(float)

if selected_name == "LightGBM":
    final_model = lgb.LGBMRegressor(**lgb_params)
    final_model.fit(X_all, y_all, categorical_feature=CAT)
else:
    final_model = xgb.XGBRegressor(**xgb_params)
    final_model.fit(X_all, y_all, verbose=False)

bundle = {
    "model": final_model,
    "model_name": selected_name,
    "categorical_columns": CAT,
    "target": TARGET,
    "category_schema": deployment_schema,
    "features": CAT,
    "training_year_min": int(df[YEAR].min()),
    "training_year_max": int(df[YEAR].max()),
    "prediction_clip_min": 0
}

MODEL_PATH = os.path.join(OUT, "cropsense_yield_deployment.joblib")
joblib.dump(bundle, MODEL_PATH)

# Dropdown combinations from all available years.
combos = df[CAT].drop_duplicates().sort_values(CAT).reset_index(drop=True)
COMBOS_PATH = os.path.join(OUT, "supported_combinations.csv")
combos.to_csv(COMBOS_PATH, index=False)

print("\n========== DEPLOYMENT ARTIFACTS ==========")
print("Selected model:", selected_name)
print("Deployment training rows:", len(df))
print("Deployment year range:", df[YEAR].min(), "-", df[YEAR].max())
print("Supported observed combinations:", len(combos))
print("Model bundle:", MODEL_PATH)
print("Supported combinations:", COMBOS_PATH)
print("Metrics folder:", OUT)

# ------------------------------------------------------------
# 9. FOUR-INPUT PREDICTION FUNCTION
# ------------------------------------------------------------
def predict_yield(crop, state, district, season, bundle=None):
    """
    Returns predicted yield_per_area and diagnostic notes.
    Inputs: crop, state, district, season.
    """
    b = bundle if bundle is not None else joblib.load(MODEL_PATH)

    values = {
        "Crop": str(crop).strip(),
        "State_Name": str(state).strip(),
        "District_Name": str(district).strip(),
        "Season": str(season).strip()
    }

    known = any(
        (combos[c] == values[c]).all()
        for c in []  # no-op; exact combination check below
    )

    known = (
        (combos["Crop"] == values["Crop"]) &
        (combos["State_Name"] == values["State_Name"]) &
        (combos["District_Name"] == values["District_Name"]) &
        (combos["Season"] == values["Season"])
    ).any()

    row = pd.DataFrame([values])
    X = make_features(row, b["category_schema"], b["features"])

    notes = []
    if not known:
        notes.append(
            "This exact combination was not observed in the dataset; "
            "treat the estimate cautiously."
        )

    if any(pd.isna(X[c].iloc[0]) for c in CAT):
        notes.append(
            "At least one input category was not learned by the model. "
            "The estimate may be unreliable."
        )

    prediction = max(float(b["model"].predict(X)[0]), 0.0)
    return round(prediction, 4), notes

# Example using an observed combination:
example = combos.iloc[0]
value, notes = predict_yield(
    example["Crop"],
    example["State_Name"],
    example["District_Name"],
    example["Season"]
)
print("\nExample input:", example.to_dict())
print("Predicted yield_per_area:", value)
print("Notes:", notes)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
LightGBM: 4.6.0
XGBoost: 3.4.1

========== DATA SUMMARY ==========
Rows: 235123
Years: 1997 - 2015
Training rows: 223882
Test rows: 11241
Training years: 1997 - 2013
Test years: [2014, 2015]
Unique crops: 104
Zero-yield records retained: 0
Unseen test values in Crop: 0
Unseen test values in State_Name: 0
Unseen test values in District_Name: 1
Unseen test values in Season: 0

Training LightGBM...
Training native categorical XGBoost...

========== OVERALL TEST RESULTS ==========
                     Model  Rows    MAE   RMSE     R2  Negative_Raw_Predictions
                  LightGBM 11241 0.9555 3.2949 0.9345                         2
Native categorical XGBoost 11241 0.9736 3.3187 0.9336                        21

========== HIGHEST MAE CROPS: LightGBM ==========
   Model        Crop  Rows  Actual_Mean     MAE    RMSE        R2
LightGBM     Cabbage    17      

In [ ]:

%pip -q install -U lightgbm

import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler

BASE = "/content/drive/MyDrive/yield prediction"
DATA_PATH = os.path.join(BASE, "crop_final.csv")
OUT = os.path.join(BASE, "cropsense_lightgbm_tuning")
os.makedirs(OUT, exist_ok=True)

CAT = ["Crop", "State_Name", "District_Name", "Season"]
TARGET = "yield_per_area"
YEAR = "Crop_Year"

# Load data
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

for c in CAT:
    df[c] = df[c].astype("string").str.strip()

df[YEAR] = pd.to_numeric(df[YEAR], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=CAT + [YEAR, TARGET]).copy()
df = df[np.isfinite(df[TARGET]) & np.isfinite(df[YEAR])]
df = df[df[TARGET] >= 0].copy()

for c in CAT:
    df[c] = df[c].astype(str)

df[YEAR] = df[YEAR].astype(int)
df = df.reset_index(drop=True)

# Chronological train / validation / test
train = df[df[YEAR] <= 2011].copy()
valid = df[df[YEAR].isin([2012, 2013])].copy()
test = df[df[YEAR].isin([2014, 2015])].copy()

# Learn category definitions from training only.
categories = {
    c: sorted(train[c].unique().tolist())
    for c in CAT
}

def features(frame):
    X = frame[CAT].copy()
    for c in CAT:
        X[c] = pd.Categorical(X[c], categories=categories[c])
    return X

X_train = features(train)
X_valid = features(valid)

y_train = train[TARGET].astype(float)
y_valid = valid[TARGET].astype(float)

print("Train rows:", len(train))
print("Validation rows:", len(valid))
print("Test rows (reserved for final comparison):", len(test))

# Hyperparameter search space
param_space = {
    "learning_rate": [0.02, 0.03, 0.04, 0.05, 0.07],
    "n_estimators": [500, 800, 1200, 1600],
    "max_depth": [-1, 8, 12, 16],
    "num_leaves": [31, 63, 127],
    "min_child_samples": [10, 20, 40],
    "reg_lambda": [1.0, 5.0, 10.0],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "subsample": [0.8, 0.9, 1.0]
}

# Randomly sample 30 configurations first.
# Increase to 50-80 if runtime permits.
candidates = list(ParameterSampler(
    param_space, n_iter=30, random_state=42
))

results = []

for i, params in enumerate(candidates, start=1):
    # num_leaves should not exceed the theoretical leaf limit
    # for a depth-limited tree.
    if params["max_depth"] > 0:
        params["num_leaves"] = min(
            params["num_leaves"],
            2 ** params["max_depth"]
        )

    model = lgb.LGBMRegressor(
        objective="regression",
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
        **params
    )

    model.fit(X_train, y_train, categorical_feature=CAT)

    raw_pred = model.predict(X_valid)
    pred = np.maximum(raw_pred, 0)

    row = {
        **params,
        "Validation_MAE": mean_absolute_error(y_valid, pred),
        "Validation_RMSE": np.sqrt(
            mean_squared_error(y_valid, pred)
        ),
        "Validation_R2": r2_score(y_valid, pred),
        "Negative_Raw_Predictions": int((raw_pred < 0).sum())
    }
    results.append(row)

    print(
        f"[{i:02d}/{len(candidates)}] "
        f"MAE={row['Validation_MAE']:.4f} | "
        f"RMSE={row['Validation_RMSE']:.4f} | "
        f"R2={row['Validation_R2']:.4f}"
    )

results_df = pd.DataFrame(results).sort_values(
    "Validation_MAE", ascending=True
).reset_index(drop=True)

results_path = os.path.join(OUT, "tuning_results.csv")
results_df.to_csv(results_path, index=False)

print("\n========== TOP 10 CONFIGURATIONS ==========")
print(results_df.head(10).round(4).to_string(index=False))
print("\nSaved:", results_path)


Train rows: 197598
Validation rows: 26284
Test rows (reserved for final comparison): 11241
[01/30] MAE=1.1814 | RMSE=3.7991 | R2=0.9135
[02/30] MAE=1.1694 | RMSE=3.7910 | R2=0.9139
[03/30] MAE=1.1548 | RMSE=3.7548 | R2=0.9155
[04/30] MAE=1.1444 | RMSE=3.7672 | R2=0.9149
[05/30] MAE=1.1495 | RMSE=3.7868 | R2=0.9141
[06/30] MAE=1.1972 | RMSE=3.7891 | R2=0.9140
[07/30] MAE=1.1907 | RMSE=3.8130 | R2=0.9129
[08/30] MAE=1.1686 | RMSE=3.7909 | R2=0.9139
[09/30] MAE=1.1766 | RMSE=3.8000 | R2=0.9135
[10/30] MAE=1.1493 | RMSE=3.7999 | R2=0.9135
[11/30] MAE=1.1789 | RMSE=3.8189 | R2=0.9126
[12/30] MAE=1.2160 | RMSE=3.8282 | R2=0.9122
[13/30] MAE=1.1835 | RMSE=3.7945 | R2=0.9137
[14/30] MAE=1.2564 | RMSE=3.9074 | R2=0.9085
[15/30] MAE=1.2044 | RMSE=3.7982 | R2=0.9135
[16/30] MAE=1.1625 | RMSE=3.7870 | R2=0.9140
[17/30] MAE=1.1725 | RMSE=3.7739 | R2=0.9146
[18/30] MAE=1.1581 | RMSE=3.7870 | R2=0.9140
[19/30] MAE=1.1488 | RMSE=3.8164 | R2=0.9127
[20/30] MAE=1.1664 | RMSE=3.8059 | R2=0.9132
[21/30] M

In [ ]:

import os
import numpy as np
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = "/content/drive/MyDrive/yield prediction"
DATA_PATH = os.path.join(BASE, "crop_final.csv")

CAT = ["Crop", "State_Name", "District_Name", "Season"]
TARGET = "yield_per_area"
YEAR = "Crop_Year"

df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

for c in CAT:
    df[c] = df[c].astype("string").str.strip()

df[YEAR] = pd.to_numeric(df[YEAR], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=CAT + [YEAR, TARGET]).copy()
df = df[np.isfinite(df[TARGET]) & np.isfinite(df[YEAR])]
df = df[df[TARGET] >= 0].copy()

for c in CAT:
    df[c] = df[c].astype(str)

df[YEAR] = df[YEAR].astype(int)

train = df[df[YEAR] <= 2013].copy()
test = df[df[YEAR].isin([2014, 2015])].copy()

categories = {
    c: sorted(train[c].unique().tolist()) for c in CAT
}

def make_X(frame):
    X = frame[CAT].copy()
    for c in CAT:
        X[c] = pd.Categorical(X[c], categories=categories[c])
    return X

X_train = make_X(train)
X_test = make_X(test)
y_train = train[TARGET].astype(float)
y_test = test[TARGET].astype(float)

configs = {
    "Existing LightGBM": {
        "n_estimators": 800,
        "learning_rate": 0.04,
        "num_leaves": 127,
        "min_child_samples": 10,
        "reg_lambda": 5.0,
    },
    "Tuned LightGBM": {
        "n_estimators": 1600,
        "learning_rate": 0.07,
        "num_leaves": 31,
        "min_child_samples": 40,
        "reg_lambda": 5.0,
        "subsample": 0.8,
        "colsample_bytree": 0.9,
        "max_depth": -1,
    }
}

results = []

for name, params in configs.items():
    model = lgb.LGBMRegressor(
        objective="regression",
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
        **params
    )
    model.fit(X_train, y_train, categorical_feature=CAT)

    raw = model.predict(X_test)
    pred = np.maximum(raw, 0)

    results.append({
        "Model": name,
        "Rows": len(test),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
        "Negative_Raw_Predictions": int((raw < 0).sum())
    })

results_df = pd.DataFrame(results)
print(results_df.round(4).to_string(index=False))

out = os.path.join(
    BASE, "cropsense_lightgbm_tuning", "same_test_comparison.csv"
)
results_df.to_csv(out, index=False)
print("\nSaved:", out)


            Model  Rows    MAE   RMSE     R2  Negative_Raw_Predictions
Existing LightGBM 11241 0.9555 3.2949 0.9345                         2
   Tuned LightGBM 11241 0.9640 3.2855 0.9349                         3

Saved: /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/same_test_comparison.csv


In [ ]:

# ============================================================
# CROPSENSE AI — FOCUSED LIGHTGBM TUNING
# Train:      <= 2011
# Validation: 2012-2013
# Test:       2014-2015 (NOT USED FOR SELECTION)
# Existing models remain unchanged.
# ============================================================

%pip -q install -U lightgbm

import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterSampler

BASE = "/content/drive/MyDrive/yield prediction"
DATA_PATH = os.path.join(BASE, "crop_final.csv")
OUT = os.path.join(BASE, "cropsense_lightgbm_tuning")
os.makedirs(OUT, exist_ok=True)

CAT = ["Crop", "State_Name", "District_Name", "Season"]
TARGET = "yield_per_area"
YEAR = "Crop_Year"

# ------------------------------------------------------------
# 1. Load and validate data
# ------------------------------------------------------------
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

for c in CAT:
    df[c] = df[c].astype("string").str.strip()

df[YEAR] = pd.to_numeric(df[YEAR], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

df = df.dropna(subset=CAT + [YEAR, TARGET]).copy()
df = df[np.isfinite(df[YEAR]) & np.isfinite(df[TARGET])].copy()
df = df[df[TARGET] >= 0].copy()

for c in CAT:
    df[c] = df[c].astype(str)

df[YEAR] = df[YEAR].astype(int)
df = df.reset_index(drop=True)

train = df[df[YEAR] <= 2011].copy()
valid = df[df[YEAR].isin([2012, 2013])].copy()
test = df[df[YEAR].isin([2014, 2015])].copy()

if train.empty or valid.empty or test.empty:
    raise ValueError("One of the chronological splits is empty.")

# Categories learned only from training data.
categories = {
    c: sorted(train[c].unique().tolist()) for c in CAT
}

def make_X(frame):
    X = frame[CAT].copy()
    for c in CAT:
        X[c] = pd.Categorical(
            X[c].astype(str), categories=categories[c]
        )
    return X

X_train = make_X(train)
X_valid = make_X(valid)
y_train = train[TARGET].astype(float)
y_valid = valid[TARGET].astype(float)

print("LightGBM version:", lgb.__version__)
print("Training rows:", len(train))
print("Validation rows:", len(valid))
print("Reserved test rows:", len(test))

# ------------------------------------------------------------
# 2. Focused parameter search
# ------------------------------------------------------------
# Includes the existing configuration as a baseline.
# 40 trials offer a broader search without being exhaustive.
parameter_space = {
    "learning_rate": [0.02, 0.03, 0.04, 0.05],
    "n_estimators": [600, 800, 1000, 1200],
    "num_leaves": [63, 127, 191],
    "min_child_samples": [10, 20, 30],
    "reg_lambda": [3.0, 5.0, 8.0, 10.0],
    "max_depth": [-1, 12, 16],
    "colsample_bytree": [0.8, 0.9, 1.0],
    "subsample": [0.8, 0.9, 1.0],
    "subsample_freq": [0, 1]
}

sampled = list(ParameterSampler(
    parameter_space, n_iter=40, random_state=2026
))

baseline = {
    "learning_rate": 0.04,
    "n_estimators": 800,
    "num_leaves": 127,
    "min_child_samples": 10,
    "reg_lambda": 5.0,
    "max_depth": -1,
    "colsample_bytree": 1.0,
    "subsample": 1.0,
    "subsample_freq": 0
}

# Guarantee the current configuration is included.
candidates = [baseline] + sampled
results = []

# ------------------------------------------------------------
# 3. Fit candidates and evaluate validation only
# ------------------------------------------------------------
for i, params in enumerate(candidates, start=1):
    model = lgb.LGBMRegressor(
        objective="regression",
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
        **params
    )

    model.fit(X_train, y_train, categorical_feature=CAT)

    raw = model.predict(X_valid)
    pred = np.maximum(raw, 0)

    results.append({
        **params,
        "Validation_MAE": mean_absolute_error(y_valid, pred),
        "Validation_RMSE": np.sqrt(
            mean_squared_error(y_valid, pred)
        ),
        "Validation_R2": r2_score(y_valid, pred),
        "Negative_Raw_Predictions": int((raw < 0).sum())
    })

    print(
        f"[{i:02d}/{len(candidates)}] "
        f"MAE={results[-1]['Validation_MAE']:.4f} | "
        f"RMSE={results[-1]['Validation_RMSE']:.4f} | "
        f"R2={results[-1]['Validation_R2']:.4f}"
    )

results_df = pd.DataFrame(results).sort_values(
    "Validation_MAE", ascending=True
).reset_index(drop=True)

results_path = os.path.join(OUT, "focused_tuning_results.csv")
results_df.to_csv(results_path, index=False)

print("\n========== TOP 10 VALIDATION CONFIGURATIONS ==========")
print(results_df.head(10).round(4).to_string(index=False))

print("\n========== BASELINE VALIDATION RESULT ==========")
baseline_rows = results_df[
    (results_df["learning_rate"] == baseline["learning_rate"]) &
    (results_df["n_estimators"] == baseline["n_estimators"]) &
    (results_df["num_leaves"] == baseline["num_leaves"]) &
    (results_df["min_child_samples"] == baseline["min_child_samples"]) &
    (results_df["reg_lambda"] == baseline["reg_lambda"]) &
    (results_df["max_depth"] == baseline["max_depth"]) &
    (results_df["colsample_bytree"] == baseline["colsample_bytree"]) &
    (results_df["subsample"] == baseline["subsample"]) &
    (results_df["subsample_freq"] == baseline["subsample_freq"])
]
print(baseline_rows.round(4).to_string(index=False))

print("\nSaved tuning results:", results_path)
print("No existing model was overwritten.")


LightGBM version: 4.6.0
Training rows: 197598
Validation rows: 26284
Reserved test rows: 11241
[01/41] MAE=1.1492 | RMSE=3.8111 | R2=0.9129
[02/41] MAE=1.2487 | RMSE=3.9192 | R2=0.9079
[03/41] MAE=1.1678 | RMSE=3.7943 | R2=0.9137
[04/41] MAE=1.1452 | RMSE=3.7801 | R2=0.9144
[05/41] MAE=1.1561 | RMSE=3.8021 | R2=0.9134
[06/41] MAE=1.1690 | RMSE=3.8264 | R2=0.9122
[07/41] MAE=1.1684 | RMSE=3.8230 | R2=0.9124
[08/41] MAE=1.1476 | RMSE=3.8174 | R2=0.9127
[09/41] MAE=1.1480 | RMSE=3.7832 | R2=0.9142
[10/41] MAE=1.1583 | RMSE=3.8435 | R2=0.9115
[11/41] MAE=1.1673 | RMSE=3.8269 | R2=0.9122
[12/41] MAE=1.1810 | RMSE=3.8400 | R2=0.9116
[13/41] MAE=1.1627 | RMSE=3.8075 | R2=0.9131
[14/41] MAE=1.2063 | RMSE=3.8683 | R2=0.9103
[15/41] MAE=1.1658 | RMSE=3.7954 | R2=0.9137
[16/41] MAE=1.1549 | RMSE=3.8087 | R2=0.9131
[17/41] MAE=1.1532 | RMSE=3.8615 | R2=0.9106
[18/41] MAE=1.1477 | RMSE=3.8039 | R2=0.9133
[19/41] MAE=1.1936 | RMSE=3.8772 | R2=0.9099
[20/41] MAE=1.2071 | RMSE=3.8483 | R2=0.9112
[21/4

In [ ]:

# ============================================================
# CROPSENSE AI — FINAL LIGHTGBM CONFIGURATION COMPARISON
# Train: <= 2013 | Test: 2014-2015
# Existing model: preserved
# ============================================================

import os
import numpy as np
import pandas as pd
import lightgbm as lgb

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

BASE = "/content/drive/MyDrive/yield prediction"
DATA_PATH = os.path.join(BASE, "crop_final.csv")
OUT = os.path.join(BASE, "cropsense_lightgbm_tuning")
os.makedirs(OUT, exist_ok=True)

CAT = ["Crop", "State_Name", "District_Name", "Season"]
TARGET = "yield_per_area"
YEAR = "Crop_Year"

# Load and clean
df = pd.read_csv(DATA_PATH, low_memory=False)
df.columns = df.columns.str.strip()

for c in CAT:
    df[c] = df[c].astype("string").str.strip()

df[YEAR] = pd.to_numeric(df[YEAR], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")
df = df.dropna(subset=CAT + [YEAR, TARGET]).copy()
df = df[np.isfinite(df[YEAR]) & np.isfinite(df[TARGET])]
df = df[df[TARGET] >= 0].copy()

for c in CAT:
    df[c] = df[c].astype(str)

df[YEAR] = df[YEAR].astype(int)

train = df[df[YEAR] <= 2013].copy()
test = df[df[YEAR].isin([2014, 2015])].copy()

categories = {c: sorted(train[c].unique()) for c in CAT}

def make_X(frame):
    X = frame[CAT].copy()
    for c in CAT:
        X[c] = pd.Categorical(X[c], categories=categories[c])
    return X

X_train = make_X(train)
X_test = make_X(test)
y_train = train[TARGET].astype(float)
y_test = test[TARGET].astype(float)

# Existing configuration and best tuned configuration
configs = {
    "Existing LightGBM": {
        "n_estimators": 800,
        "learning_rate": 0.04,
        "num_leaves": 127,
        "min_child_samples": 10,
        "reg_lambda": 5.0,
        "max_depth": -1,
        "colsample_bytree": 1.0,
        "subsample": 1.0,
        "subsample_freq": 0
    },
    "Tuned LightGBM": {
        "n_estimators": 1200,
        "learning_rate": 0.05,
        "num_leaves": 127,
        "min_child_samples": 10,
        "reg_lambda": 5.0,
        "max_depth": 12,
        "colsample_bytree": 1.0,
        "subsample": 0.9,
        "subsample_freq": 0
    }
}

rows = []
predictions = pd.DataFrame({
    "Crop": test["Crop"].to_numpy(),
    "Crop_Year": test[YEAR].to_numpy(),
    "Actual": y_test.to_numpy()
})

for name, params in configs.items():
    model = lgb.LGBMRegressor(
        objective="regression",
        random_state=42,
        verbosity=-1,
        n_jobs=-1,
        **params
    )
    model.fit(X_train, y_train, categorical_feature=CAT)

    raw = model.predict(X_test)
    pred = np.maximum(raw, 0)

    rows.append({
        "Model": name,
        "Rows": len(test),
        "MAE": mean_absolute_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
        "Negative_Raw_Predictions": int((raw < 0).sum())
    })

    predictions[name + "_Prediction"] = pred
    predictions[name + "_Absolute_Error"] = np.abs(y_test.to_numpy() - pred)

results = pd.DataFrame(rows).sort_values("MAE")
print("\n========== SAME-TEST COMPARISON ==========")
print(results.round(4).to_string(index=False))

results.to_csv(
    os.path.join(OUT, "focused_tuned_test_comparison.csv"),
    index=False
)
predictions.to_csv(
    os.path.join(OUT, "focused_tuned_test_predictions.csv"),
    index=False
)

# Crop-level comparison
crop_results = []
for model_name in configs:
    pred_col = model_name + "_Prediction"
    for crop, group in predictions.groupby("Crop"):
        crop_results.append({
            "Model": model_name,
            "Crop": crop,
            "Rows": len(group),
            "MAE": np.mean(group["Actual"] - group[pred_col]) if False
                   else np.mean(np.abs(group["Actual"] - group[pred_col])),
            "RMSE": np.sqrt(np.mean((group["Actual"] - group[pred_col]) ** 2))
        })

crop_results = pd.DataFrame(crop_results)
crop_results.to_csv(
    os.path.join(OUT, "focused_tuned_per_crop_comparison.csv"),
    index=False
)

print("\n========== SAVED ==========")
print("Comparison:", os.path.join(OUT, "focused_tuned_test_comparison.csv"))
print("Predictions:", os.path.join(OUT, "focused_tuned_test_predictions.csv"))
print("Per-crop comparison:", os.path.join(OUT, "focused_tuned_per_crop_comparison.csv"))
print("\nNo existing model was overwritten.")



========== SAME-TEST COMPARISON ==========
            Model  Rows    MAE   RMSE     R2  Negative_Raw_Predictions
   Tuned LightGBM 11241 0.9501 3.2753 0.9353                         2
Existing LightGBM 11241 0.9555 3.2949 0.9345                         2

========== SAVED ==========
Comparison: /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/focused_tuned_test_comparison.csv
Predictions: /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/focused_tuned_test_predictions.csv
Per-crop comparison: /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/focused_tuned_per_crop_comparison.csv

No existing model was overwritten.


In [ ]:
params = {
    "objective": "regression",
    "n_estimators": 1200,
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_child_samples": 10,
    "reg_lambda": 5.0,
    "max_depth": 12,
    "colsample_bytree": 1.0,
    "subsample": 0.9,
    "subsample_freq": 0,
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": -1
}

In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Folder containing tuned LightGBM results
OUTPUT_DIR = Path(
    "/content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning"
)

PRED_FILE = OUTPUT_DIR / "focused_tuned_test_predictions.csv"

if not PRED_FILE.exists():
    raise FileNotFoundError(
        f"File not found: {PRED_FILE}\n"
        "Please check Google Drive is mounted and the filename is correct."
    )

df = pd.read_csv(PRED_FILE)

print("Dataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())


Dataset shape: (11241, 7)

Columns:
['Crop', 'Crop_Year', 'Actual', 'Existing LightGBM_Prediction', 'Existing LightGBM_Absolute_Error', 'Tuned LightGBM_Prediction', 'Tuned LightGBM_Absolute_Error']

First 5 rows:


,Crop,Crop_Year,Actual,Existing LightGBM_Prediction,Existing LightGBM_Absolute_Error,Tuned LightGBM_Prediction,Tuned LightGBM_Absolute_Error
0,Arhar/Tur,2014,0.128987,0.233893,0.104906,0.244330,0.115343
1,Bajra,2014,0.586758,0.916847,0.330089,0.881845,0.295087
2,Brinjal,2014,23.737500,18.288241,5.449259,18.832296,4.905204
3,Cabbage,2014,12.300000,22.443000,10.143000,19.560821,7.260821
4,Cashewnut,2014,0.368421,0.572158,0.203737,0.492660,0.124239


In [ ]:

print("Available columns:")
for col in df.columns:
    print("-", repr(col))


Available columns:
- 'Crop'
- 'Crop_Year'
- 'Actual'
- 'Existing LightGBM_Prediction'
- 'Existing LightGBM_Absolute_Error'
- 'Tuned LightGBM_Prediction'
- 'Tuned LightGBM_Absolute_Error'


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

OUTPUT_DIR = Path(
    "/content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning"
)

df = pd.read_csv(OUTPUT_DIR / "focused_tuned_test_predictions.csv")

actual_col = "Actual"
existing_pred_col = "Existing LightGBM_Prediction"
tuned_pred_col = "Tuned LightGBM_Prediction"

# Verify required columns
required = ["Crop", actual_col, existing_pred_col, tuned_pred_col]
missing = [c for c in required if c not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

def calculate_metrics(group, prediction_col):
    actual = group[actual_col].to_numpy()
    predicted = group[prediction_col].to_numpy()

    result = {
        "Samples": len(group),
        "MAE": mean_absolute_error(actual, predicted),
        "RMSE": np.sqrt(mean_squared_error(actual, predicted)),
        "Mean_Actual": actual.mean(),
        "Mean_Predicted": predicted.mean(),
        "Mean_Error": np.mean(predicted - actual),
    }

    # R² is not meaningful for groups with fewer than 2 samples
    # or no variation in actual values.
    result["R2"] = (
        r2_score(actual, predicted)
        if len(group) >= 2 and np.var(actual) > 0
        else np.nan
    )
    return result

rows = []

for crop, group in df.groupby("Crop"):
    old = calculate_metrics(group, existing_pred_col)
    tuned = calculate_metrics(group, tuned_pred_col)

    rows.append({
        "Crop": crop,
        "Samples": len(group),
        "Actual_Mean": old["Mean_Actual"],
        "Existing_MAE": old["MAE"],
        "Tuned_MAE": tuned["MAE"],
        "MAE_Improvement": old["MAE"] - tuned["MAE"],
        "Existing_RMSE": old["RMSE"],
        "Tuned_RMSE": tuned["RMSE"],
        "Existing_R2": old["R2"],
        "Tuned_R2": tuned["R2"],
        "Tuned_Mean_Error": tuned["Mean_Error"],
    })

crop_metrics = pd.DataFrame(rows)

# Positive improvement means tuned model has lower MAE.
crop_metrics = crop_metrics.sort_values(
    "Tuned_MAE", ascending=False
).reset_index(drop=True)

print("Number of crops:", len(crop_metrics))
print("\nTop 15 crops with the highest tuned-model MAE:")
display(crop_metrics.head(15).round(4))

print("\nTop 15 crops where tuning improved MAE the most:")
display(
    crop_metrics.sort_values(
        "MAE_Improvement", ascending=False
    ).head(15).round(4)
)

print("\nTop 15 crops where tuning increased MAE:")
display(
    crop_metrics.sort_values(
        "MAE_Improvement", ascending=True
    ).head(15).round(4)
)

# Save the analysis
save_path = OUTPUT_DIR / "tuned_crop_wise_error_analysis.csv"
crop_metrics.to_csv(save_path, index=False)
print(f"\nSaved analysis to: {save_path}")


Number of crops: 74

Top 15 crops with the highest tuned-model MAE:


,Crop,Samples,Actual_Mean,Existing_MAE,Tuned_MAE,MAE_Improvement,Existing_RMSE,Tuned_RMSE,Existing_R2,Tuned_R2,Tuned_Mean_Error
0,Cabbage,17,12.6085,10.5555,10.7559,-0.2004,11.0112,11.4875,-1.5918,-1.8209,7.3554
1,Sugarcane,360,60.6190,9.6938,9.7129,-0.0190,15.1908,15.1995,0.6961,0.6957,-5.7035
2,Papaya,44,42.1091,9.2490,8.3982,0.8507,16.8483,15.8584,0.7218,0.7535,6.7160
3,Grapes,4,24.3524,4.9507,5.4988,-0.5480,6.1837,6.4378,-3.9805,-4.3981,-4.8429
4,Brinjal,29,15.2998,3.8810,3.9218,-0.0408,5.1992,5.2920,0.4682,0.4491,-3.0682
5,Tomato,26,17.1471,3.5786,3.7751,-0.1965,4.2320,4.4692,0.4167,0.3495,0.1297
6,Banana,119,28.2720,3.6079,3.4426,0.1654,5.4462,5.0666,0.8702,0.8877,-2.4794
7,Dry ginger,96,9.3143,3.3447,3.3635,-0.0188,6.1167,6.1598,0.6603,0.6555,-2.6177
8,Pome Granet,9,10.2807,3.5305,3.2221,0.3084,3.6102,3.3988,-197.0526,-174.5339,3.2221
9,Sapota,13,8.3051,2.9307,2.9625,-0.0318,3.0148,3.0135,-84.6666,-84.5903,2.9625



Top 15 crops where tuning improved MAE the most:


,Crop,Samples,Actual_Mean,Existing_MAE,Tuned_MAE,MAE_Improvement,Existing_RMSE,Tuned_RMSE,Existing_R2,Tuned_R2,Tuned_Mean_Error
2,Papaya,44,42.1091,9.2490,8.3982,0.8507,16.8483,15.8584,0.7218,0.7535,6.7160
46,Other Fresh Fruits,2,10.6448,1.0815,0.2410,0.8404,1.0896,0.2718,NaN,NaN,-0.2410
8,Pome Granet,9,10.2807,3.5305,3.2221,0.3084,3.6102,3.3988,-197.0526,-174.5339,3.2221
57,Other Vegetables,2,11.4062,0.4034,0.1728,0.2305,0.4036,0.1729,-454.7203,-82.6308,-0.1728
6,Banana,119,28.2720,3.6079,3.4426,0.1654,5.4462,5.0666,0.8702,0.8877,-2.4794
51,Pulses total,41,0.7730,0.2861,0.2076,0.0785,0.3446,0.2617,-3.1827,-1.4114,-0.1048
11,Mango,18,7.7669,2.7491,2.6718,0.0773,3.4850,3.3928,0.4275,0.4574,0.3992
25,Pineapple,30,16.2162,0.6735,0.5991,0.0744,1.1332,1.0682,0.6969,0.7307,-0.4921
34,other oilseeds,14,0.8965,0.3839,0.3162,0.0677,0.6124,0.5141,-0.3108,0.0762,-0.2629
60,Moth,2,0.6537,0.1927,0.1434,0.0492,0.2592,0.1860,-0.3725,0.2932,-0.1185



Top 15 crops where tuning increased MAE:


,Crop,Samples,Actual_Mean,Existing_MAE,Tuned_MAE,MAE_Improvement,Existing_RMSE,Tuned_RMSE,Existing_R2,Tuned_R2,Tuned_Mean_Error
3,Grapes,4,24.3524,4.9507,5.4988,-0.5480,6.1837,6.4378,-3.9805,-4.3981,-4.8429
15,Lemon,13,7.6017,1.5649,1.8654,-0.3005,1.6491,2.0164,0.7419,0.6141,0.5154
33,Blackgram,11,0.8206,0.0892,0.3654,-0.2762,0.1162,0.3927,-1.7280,-30.1628,0.3654
0,Cabbage,17,12.6085,10.5555,10.7559,-0.2004,11.0112,11.4875,-1.5918,-1.8209,7.3554
5,Tomato,26,17.1471,3.5786,3.7751,-0.1965,4.2320,4.4692,0.4167,0.3495,0.1297
22,Total foodgrain,44,1.6910,0.6338,0.7125,-0.0787,0.7702,0.8251,-0.4835,-0.7025,-0.3337
4,Brinjal,29,15.2998,3.8810,3.9218,-0.0408,5.1992,5.2920,0.4682,0.4491,-3.0682
9,Sapota,13,8.3051,2.9307,2.9625,-0.0318,3.0148,3.0135,-84.6666,-84.5903,2.9625
14,Potato,357,13.2882,2.2954,2.3168,-0.0214,3.2775,3.3183,0.8130,0.8083,-0.5105
44,Other Cereals & Millets,19,0.6250,0.2366,0.2575,-0.0209,0.2908,0.3173,-0.8311,-1.1802,-0.0437



Saved analysis to: /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/tuned_crop_wise_error_analysis.csv


In [ ]:

# Inspect these crops where earlier model runs showed large errors.
focus_crops = ["Sugarcane", "Papaya", "Cabbage"]

for crop in focus_crops:
    subset = df[df["Crop"].astype(str).str.strip().str.lower()
                == crop.lower()].copy()

    print(f"\n{'=' * 65}")
    print(f"{crop} — {len(subset)} test records")

    if subset.empty:
        print("Crop not found under this exact name.")
        continue

    subset["Tuned_Signed_Error"] = (
        subset[tuned_pred_col] - subset[actual_col]
    )
    subset["Tuned_Absolute_Error"] = subset["Tuned_Signed_Error"].abs()

    display(
        subset.sort_values(
            "Tuned_Absolute_Error", ascending=False
        )[
            [
                "Crop", "Crop_Year", actual_col,
                tuned_pred_col, "Tuned_Absolute_Error",
                "Tuned_Signed_Error"
            ]
        ].head(10).round(4)
    )

print("\nLargest 20 tuned-model errors across all crops:")

df["Tuned_Signed_Error"] = df[tuned_pred_col] - df[actual_col]
df["Tuned_Absolute_Error"] = df["Tuned_Signed_Error"].abs()

display(
    df.sort_values("Tuned_Absolute_Error", ascending=False)[
        [
            "Crop", "Crop_Year", actual_col,
            tuned_pred_col, "Tuned_Absolute_Error",
            "Tuned_Signed_Error"
        ]
    ].head(20).round(4)
)



Sugarcane — 360 test records


,Crop,Crop_Year,Actual,Tuned LightGBM_Prediction,Tuned_Absolute_Error,Tuned_Signed_Error
11004,Sugarcane,2014,179.8663,87.3104,92.5559,-92.5559
11140,Sugarcane,2014,179.8665,100.9073,78.9592,-78.9592
2337,Sugarcane,2014,113.2609,40.8610,72.3999,-72.3999
1819,Sugarcane,2014,113.2857,41.9990,71.2867,-71.2867
1839,Sugarcane,2014,112.8485,42.1545,70.6940,-70.6940
2073,Sugarcane,2014,113.2778,43.2257,70.0521,-70.0521
11201,Sugarcane,2014,141.5137,84.7913,56.7224,-56.7224
5717,Sugarcane,2014,29.0000,68.7731,39.7731,39.7731
7413,Sugarcane,2014,165.0000,126.7722,38.2278,-38.2278
2015,Sugarcane,2014,79.1502,41.0121,38.1380,-38.1380



Papaya — 44 test records


,Crop,Crop_Year,Actual,Tuned LightGBM_Prediction,Tuned_Absolute_Error,Tuned_Signed_Error
83,Papaya,2014,26.5703,81.2875,54.7172,54.7172
395,Papaya,2014,69.1796,120.8617,51.6821,51.6821
366,Papaya,2014,78.8000,106.8199,28.0199,28.0199
292,Papaya,2014,78.7273,103.8132,25.0859,25.0859
643,Papaya,2014,78.7511,103.5125,24.7614,24.7614
136,Papaya,2014,78.7246,103.3733,24.6487,24.6487
187,Papaya,2014,78.7273,103.1544,24.4271,24.4271
335,Papaya,2014,78.7273,103.0627,24.3354,24.3354
501,Papaya,2014,78.7255,101.7900,23.0645,23.0645
545,Papaya,2014,78.7253,101.1112,22.3859,22.3859



Cabbage — 17 test records


,Crop,Crop_Year,Actual,Tuned LightGBM_Prediction,Tuned_Absolute_Error,Tuned_Signed_Error
302,Cabbage,2014,2.5824,22.1483,19.5659,19.5659
282,Cabbage,2014,2.5789,21.9345,19.3555,19.3555
66,Cabbage,2014,29.0917,13.6526,15.4391,-15.4391
98,Cabbage,2014,29.0932,15.6280,13.4652,-13.4652
585,Cabbage,2014,12.3243,22.7574,10.4331,10.4331
174,Cabbage,2014,12.3333,22.7022,10.3689,10.3689
613,Cabbage,2014,12.3210,22.4500,10.1290,10.1290
199,Cabbage,2014,12.3103,22.4064,10.0961,10.0961
323,Cabbage,2014,12.3636,22.3981,10.0345,10.0345
352,Cabbage,2014,12.3282,22.3081,9.9798,9.9798



Largest 20 tuned-model errors across all crops:


,Crop,Crop_Year,Actual,Tuned LightGBM_Prediction,Tuned_Absolute_Error,Tuned_Signed_Error
11004,Sugarcane,2014,179.8663,87.3104,92.5559,-92.5559
11140,Sugarcane,2014,179.8665,100.9073,78.9592,-78.9592
2337,Sugarcane,2014,113.2609,40.8610,72.3999,-72.3999
1819,Sugarcane,2014,113.2857,41.9990,71.2867,-71.2867
1839,Sugarcane,2014,112.8485,42.1545,70.6940,-70.6940
2073,Sugarcane,2014,113.2778,43.2257,70.0521,-70.0521
11201,Sugarcane,2014,141.5137,84.7913,56.7224,-56.7224
83,Papaya,2014,26.5703,81.2875,54.7172,54.7172
395,Papaya,2014,69.1796,120.8617,51.6821,51.6821
5717,Sugarcane,2014,29.0000,68.7731,39.7731,39.7731


In [ ]:

import pandas as pd
from pathlib import Path

DATA_PATH = Path(
    "/content/drive/MyDrive/yield prediction/crop_final.csv"
)

data = pd.read_csv(DATA_PATH)

print("Original dataset shape:", data.shape)
print("\nAvailable columns:")
print(data.columns.tolist())

# Check which columns are available before investigating.
possible_cols = [
    "Crop", "Crop_Year", "State_Name", "District_Name",
    "Season", "yield_per_area", "Area", "Production"
]

available = [c for c in possible_cols if c in data.columns]
print("\nColumns available for investigation:", available)

# Identify the crop and year columns in the original file.
crop_col = "Crop" if "Crop" in data.columns else "crop_names"
year_col = "Crop_Year" if "Crop_Year" in data.columns else "crop_year"

if crop_col not in data.columns or year_col not in data.columns:
    raise ValueError("Could not identify the crop and year columns.")

focus_crops = ["Sugarcane", "Papaya", "Cabbage"]

subset = data[
    data[crop_col].astype(str).str.strip().str.lower().isin(
        [c.lower() for c in focus_crops]
    )
    & pd.to_numeric(data[year_col], errors="coerce").isin([2014, 2015])
].copy()

# Display the original rows, including location and season if available.
display_cols = [
    c for c in [
        crop_col, year_col, "State_Name", "District_Name",
        "Season", "yield_per_area", "Area", "Production"
    ]
    if c in subset.columns
]

print("\nMatching original records:", len(subset))
display(subset[display_cols].head(30))

print("\nCounts by crop and year:")
display(
    subset.groupby([crop_col, year_col])
    .size()
    .rename("Records")
    .reset_index()
)


Original dataset shape: (235123, 8)

Available columns:
['State_Name', 'District_Name', 'Season', 'Crop', 'Crop_Year', 'Area', 'Production', 'yield_per_area']

Columns available for investigation: ['Crop', 'Crop_Year', 'State_Name', 'District_Name', 'Season', 'yield_per_area', 'Area', 'Production']

Matching original records: 421


,Crop,Crop_Year,State_Name,District_Name,Season,yield_per_area,Area,Production
875,Cabbage,2014,Andhra Pradesh,ANANTAPUR,Kharif,12.300000,10.0,123.0
894,Papaya,2014,Andhra Pradesh,ANANTAPUR,Kharif,78.724599,374.0,29443.0
902,Sugarcane,2014,Andhra Pradesh,ANANTAPUR,Kharif,81.866667,30.0,2456.0
910,Cabbage,2014,Andhra Pradesh,ANANTAPUR,Rabi,12.500000,2.0,25.0
1637,Cabbage,2014,Andhra Pradesh,CHITTOOR,Kharif,29.091667,120.0,3491.0
1654,Papaya,2014,Andhra Pradesh,CHITTOOR,Kharif,26.570328,519.0,13790.0
1662,Sugarcane,2014,Andhra Pradesh,CHITTOOR,Kharif,79.052511,26147.0,2066986.0
1669,Cabbage,2014,Andhra Pradesh,CHITTOOR,Rabi,29.093168,161.0,4684.0
2369,Cabbage,2014,Andhra Pradesh,EAST GODAVARI,Kharif,12.318182,22.0,271.0
2382,Papaya,2014,Andhra Pradesh,EAST GODAVARI,Kharif,78.724590,305.0,24011.0



Counts by crop and year:


,Crop,Crop_Year,Records
0,Cabbage,2014,17
1,Papaya,2014,44
2,Sugarcane,2014,332
3,Sugarcane,2015,28


In [ ]:

import pandas as pd
import numpy as np
from pathlib import Path

BASE = Path("/content/drive/MyDrive/yield prediction")
original = pd.read_csv(BASE / "crop_final.csv")
predictions = pd.read_csv(
    BASE / "cropsense_lightgbm_tuning" /
    "focused_tuned_test_predictions.csv"
)

# Reconstruct the chronological test period.
test_rows = original[
    pd.to_numeric(original["Crop_Year"], errors="coerce").isin([2014, 2015])
].copy()

print("Original 2014–2015 rows:", len(test_rows))
print("Prediction rows:", len(predictions))

# Check row-by-row alignment before attaching locations.
test_rows = test_rows.reset_index(drop=True)
predictions = predictions.reset_index(drop=True)

same_length = len(test_rows) == len(predictions)

if same_length:
    same_crop = (
        test_rows["Crop"].astype(str).to_numpy()
        == predictions["Crop"].astype(str).to_numpy()
    )
    same_year = (
        pd.to_numeric(test_rows["Crop_Year"]).to_numpy()
        == pd.to_numeric(predictions["Crop_Year"]).to_numpy()
    )
    same_actual = np.isclose(
        pd.to_numeric(test_rows["yield_per_area"]).to_numpy(),
        pd.to_numeric(predictions["Actual"]).to_numpy(),
        rtol=1e-5,
        atol=1e-6,
        equal_nan=True
    )

    aligned = bool(same_crop.all() and same_year.all() and same_actual.all())
else:
    aligned = False

print("Exact row alignment verified:", aligned)

if not aligned:
    raise ValueError(
        "Rows do not align exactly. Do not merge locations by row number. "
        "We need to reconstruct the original test split/order first."
    )

# Attach locations only after verifying every row.
for col in ["State_Name", "District_Name", "Season"]:
    predictions[col] = test_rows[col].to_numpy()

predictions["Tuned_Signed_Error"] = (
    predictions["Tuned LightGBM_Prediction"] - predictions["Actual"]
)
predictions["Tuned_Absolute_Error"] = (
    predictions["Tuned_Signed_Error"].abs()
)

# Inspect the largest errors with their actual locations.
focus = predictions[
    predictions["Crop"].isin(["Sugarcane", "Papaya", "Cabbage"])
].copy()

print("\nLargest errors with location and season:")
display(
    focus.sort_values("Tuned_Absolute_Error", ascending=False)[
        [
            "Crop", "Crop_Year", "State_Name", "District_Name",
            "Season", "Actual", "Tuned LightGBM_Prediction",
            "Tuned_Absolute_Error", "Tuned_Signed_Error"
        ]
    ].head(25).round(4)
)

save_path = (
    BASE / "cropsense_lightgbm_tuning" /
    "tuned_predictions_with_locations.csv"
)
predictions.to_csv(save_path, index=False)
print("\nSaved location-enriched predictions to:", save_path)


Original 2014–2015 rows: 11241
Prediction rows: 11241
Exact row alignment verified: True

Largest errors with location and season:


,Crop,Crop_Year,State_Name,District_Name,Season,Actual,Tuned LightGBM_Prediction,Tuned_Absolute_Error,Tuned_Signed_Error
11004,Sugarcane,2014,West Bengal,HOOGHLY,Whole Year,179.8663,87.3104,92.5559,-92.5559
11140,Sugarcane,2014,West Bengal,MEDINIPUR WEST,Whole Year,179.8665,100.9073,78.9592,-78.9592
2337,Sugarcane,2014,Bihar,NAWADA,Whole Year,113.2609,40.8610,72.3999,-72.3999
1819,Sugarcane,2014,Bihar,ARWAL,Whole Year,113.2857,41.9990,71.2867,-71.2867
1839,Sugarcane,2014,Bihar,AURANGABAD,Whole Year,112.8485,42.1545,70.6940,-70.6940
2073,Sugarcane,2014,Bihar,JEHANABAD,Whole Year,113.2778,43.2257,70.0521,-70.0521
11201,Sugarcane,2014,West Bengal,NADIA,Whole Year,141.5137,84.7913,56.7224,-56.7224
83,Papaya,2014,Andhra Pradesh,CHITTOOR,Kharif,26.5703,81.2875,54.7172,54.7172
395,Papaya,2014,Andhra Pradesh,PRAKASAM,Kharif,69.1796,120.8617,51.6821,51.6821
5717,Sugarcane,2014,Maharashtra,WASHIM,Whole Year,29.0000,68.7731,39.7731,39.7731



Saved location-enriched predictions to: /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/tuned_predictions_with_locations.csv


In [3]:

from pathlib import Path

BASE = Path("/content/drive/MyDrive/yield prediction")

paths_to_check = [
    BASE / "cropsense_corrected_v3" /
    "cropsense_yield_deployment.joblib",

    BASE / "cropsense_lightgbm_tuning" /
    "focused_tuning_results.csv",

    BASE / "cropsense_lightgbm_tuning" /
    "focused_tuned_test_comparison.csv",

    BASE / "cropsense_lightgbm_tuning" /
    "tuned_predictions_with_locations.csv",
]

for path in paths_to_check:
    print(
        "FOUND    " if path.exists() else "NOT FOUND",
        path
    )


FOUND     /content/drive/MyDrive/yield prediction/cropsense_corrected_v3/cropsense_yield_deployment.joblib
FOUND     /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/focused_tuning_results.csv
FOUND     /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/focused_tuned_test_comparison.csv
FOUND     /content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuning/tuned_predictions_with_locations.csv


In [4]:

from pathlib import Path
import pandas as pd
import joblib
import lightgbm as lgb

BASE = Path("/content/drive/MyDrive/yield prediction")

# 1. Inspect the existing deployment model (read-only)
model_path = (
    BASE / "cropsense_corrected_v3" /
    "cropsense_yield_deployment.joblib"
)

artifact = joblib.load(model_path)

print("=" * 65)
print("EXISTING DEPLOYMENT ARTIFACT")
print("=" * 65)
print("Artifact type:", type(artifact))

if isinstance(artifact, dict):
    print("Artifact keys:", list(artifact.keys()))

    for key, value in artifact.items():
        print(f"\n{key}:")
        print("  Type:", type(value))
        if hasattr(value, "feature_name_"):
            print("  Features:", value.feature_name_)
        if hasattr(value, "get_params"):
            print("  Parameters:", value.get_params())
else:
    if hasattr(artifact, "feature_name_"):
        print("Features:", artifact.feature_name_)
    if hasattr(artifact, "get_params"):
        print("Parameters:", artifact.get_params())

# 2. Inspect focused tuning results
tuning_path = (
    BASE / "cropsense_lightgbm_tuning" /
    "focused_tuning_results.csv"
)

tuning = pd.read_csv(tuning_path)

print("\n" + "=" * 65)
print("FOCUSED TUNING RESULTS")
print("=" * 65)
print("Shape:", tuning.shape)
print("Columns:", tuning.columns.tolist())
display(tuning.head(10))

# 3. Inspect same-test comparison
comparison_path = (
    BASE / "cropsense_lightgbm_tuning" /
    "focused_tuned_test_comparison.csv"
)

comparison = pd.read_csv(comparison_path)

print("\n" + "=" * 65)
print("MODEL COMPARISON")
print("=" * 65)
display(comparison)

print("\nLightGBM version:", lgb.__version__)


EXISTING DEPLOYMENT ARTIFACT
Artifact type: <class 'dict'>
Artifact keys: ['model', 'model_name', 'categorical_columns', 'target', 'category_schema', 'features', 'training_year_min', 'training_year_max', 'prediction_clip_min']

model:
  Type: <class 'lightgbm.sklearn.LGBMRegressor'>
  Features: ['Crop', 'State_Name', 'District_Name', 'Season']
  Parameters: {'boosting_type': 'gbdt', 'class_weight': None, 'colsample_bytree': 1.0, 'importance_type': 'split', 'learning_rate': 0.04, 'max_depth': -1, 'min_child_samples': 10, 'min_child_weight': 0.001, 'min_split_gain': 0.0, 'n_estimators': 800, 'n_jobs': -1, 'num_leaves': 127, 'objective': 'regression', 'random_state': 42, 'reg_alpha': 0.0, 'reg_lambda': 5.0, 'subsample': 1.0, 'subsample_for_bin': 200000, 'subsample_freq': 0, 'verbosity': -1}

model_name:
  Type: <class 'str'>

categorical_columns:
  Type: <class 'list'>

target:
  Type: <class 'str'>

category_schema:
  Type: <class 'dict'>

features:
  Type: <class 'list'>

training_year_

,learning_rate,n_estimators,num_leaves,min_child_samples,reg_lambda,max_depth,colsample_bytree,subsample,subsample_freq,Validation_MAE,Validation_RMSE,Validation_R2,Negative_Raw_Predictions
0,0.05,1200,127,10,5.0,12,1.0,0.9,0,1.145239,3.780069,0.914362,4
1,0.05,1200,127,30,3.0,-1,1.0,1.0,0,1.146135,3.816453,0.912705,6
2,0.03,1200,191,10,8.0,16,0.9,0.8,0,1.147621,3.817408,0.912662,7
3,0.03,800,191,10,5.0,16,0.9,0.9,0,1.147668,3.803942,0.913277,5
4,0.05,1200,191,20,8.0,12,0.9,0.8,0,1.147720,3.802670,0.913335,1
5,0.04,1200,63,10,3.0,-1,0.9,1.0,0,1.148014,3.783249,0.914218,7
6,0.04,800,127,10,5.0,-1,1.0,1.0,0,1.149167,3.811129,0.912949,8
7,0.04,600,191,20,8.0,16,1.0,0.9,1,1.150874,3.831241,0.912028,7
8,0.04,1000,63,20,5.0,-1,1.0,1.0,0,1.152698,3.781032,0.914318,6
9,0.05,600,127,10,5.0,12,0.9,0.9,0,1.152816,3.789143,0.913950,7



MODEL COMPARISON


,Model,Rows,MAE,RMSE,R2,Negative_Raw_Predictions
0,Tuned LightGBM,11241,0.950062,3.275303,0.935283,2
1,Existing LightGBM,11241,0.955476,3.294873,0.934508,2



LightGBM version: 4.6.0


In [5]:

from pathlib import Path
import json
import joblib
import numpy as np
import pandas as pd
import lightgbm as lgb

from lightgbm import LGBMRegressor

# --------------------------------------------------
# 1. Paths
# --------------------------------------------------
BASE = Path("/content/drive/MyDrive/yield prediction")
DATA_PATH = BASE / "crop_final.csv"

OUTPUT_DIR = BASE / "cropsense_lightgbm_tuned_deployment"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_PATH = OUTPUT_DIR / "cropsense_yield_tuned_deployment.joblib"
METADATA_PATH = OUTPUT_DIR / "model_metadata.json"

# --------------------------------------------------
# 2. Load and validate dataset
# --------------------------------------------------
df = pd.read_csv(DATA_PATH)

FEATURES = ["Crop", "State_Name", "District_Name", "Season"]
TARGET = "yield_per_area"
YEAR_COL = "Crop_Year"

required = FEATURES + [TARGET, YEAR_COL]
missing = [c for c in required if c not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

df[YEAR_COL] = pd.to_numeric(df[YEAR_COL], errors="coerce")
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

df = df.dropna(subset=required).copy()
df = df[df[TARGET] >= 0].copy()

# Normalize categorical columns consistently.
for col in FEATURES:
    df[col] = df[col].astype(str).str.strip()

# Do not overwrite or retrain the old artifact.
print("Dataset rows used:", len(df))
print("Year range:", int(df[YEAR_COL].min()), "-", int(df[YEAR_COL].max()))
print("Unique crops:", df["Crop"].nunique())

X = df[FEATURES].copy()
y = df[TARGET].astype(float).copy()

# LightGBM native categorical features.
for col in FEATURES:
    X[col] = X[col].astype("category")

# --------------------------------------------------
# 3. Train the selected configuration
# --------------------------------------------------
params = {
    "objective": "regression",
    "n_estimators": 1200,
    "learning_rate": 0.05,
    "num_leaves": 127,
    "min_child_samples": 10,
    "reg_lambda": 5.0,
    "max_depth": 12,
    "colsample_bytree": 1.0,
    "subsample": 0.9,
    "subsample_freq": 0,
    "random_state": 42,
    "verbosity": -1,
    "n_jobs": -1,
}

model = LGBMRegressor(**params)
model.fit(X, y, categorical_feature=FEATURES)

# --------------------------------------------------
# 4. Save category schema and deployment artifact
# --------------------------------------------------
category_schema = {
    col: sorted(df[col].astype(str).unique().tolist())
    for col in FEATURES
}

artifact = {
    "model": model,
    "model_name": "CropSense AI Tuned LightGBM Yield Model",
    "categorical_columns": FEATURES,
    "target": TARGET,
    "category_schema": category_schema,
    "features": FEATURES,
    "training_year_min": int(df[YEAR_COL].min()),
    "training_year_max": int(df[YEAR_COL].max()),
    "prediction_clip_min": 0,
}

joblib.dump(artifact, MODEL_PATH)

metadata = {
    "model_name": artifact["model_name"],
    "features": FEATURES,
    "target": TARGET,
    "training_rows": int(len(df)),
    "training_year_min": int(df[YEAR_COL].min()),
    "training_year_max": int(df[YEAR_COL].max()),
    "unique_crops": int(df["Crop"].nunique()),
    "lightgbm_version": lgb.__version__,
    "parameters": params,
    "test_evaluation_status": (
        "2014-2015 used for model selection; "
        "not an independent final test after this refit."
    ),
}

with open(METADATA_PATH, "w", encoding="utf-8") as f:
    json.dump(metadata, f, indent=2)

print("\nTuned deployment model saved:")
print(MODEL_PATH)
print("\nMetadata saved:")
print(METADATA_PATH)
print("\nExisting deployment model was not modified.")


Dataset rows used: 235123
Year range: 1997 - 2015
Unique crops: 104

Tuned deployment model saved:
/content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuned_deployment/cropsense_yield_tuned_deployment.joblib

Metadata saved:
/content/drive/MyDrive/yield prediction/cropsense_lightgbm_tuned_deployment/model_metadata.json

Existing deployment model was not modified.


In [6]:

import os
import joblib
import pandas as pd
import numpy as np

MODEL_PATH = (
    "/content/drive/MyDrive/yield prediction/"
    "cropsense_lightgbm_tuned_deployment/"
    "cropsense_yield_tuned_deployment.joblib"
)

assert os.path.exists(MODEL_PATH), "Model file not found!"

artifact = joblib.load(MODEL_PATH)

print("Model loaded successfully!")
print("Model name:", artifact["model_name"])
print("Features:", artifact["features"])
print("Target:", artifact["target"])
print("Training years:",
      artifact["training_year_min"], "-",
      artifact["training_year_max"])

model = artifact["model"]
features = artifact["features"]

# Use the model's saved categorical schema where available.
category_lists = getattr(model.booster_, "pandas_categorical", None)

sample = {
    "Crop": ["Rice"],
    "State_Name": ["Andhra Pradesh"],
    "District_Name": ["Krishna"],
    "Season": ["Kharif"]
}

X_sample = pd.DataFrame(sample)[features]

for i, col in enumerate(features):
    if category_lists and i < len(category_lists):
        X_sample[col] = pd.Categorical(
            X_sample[col],
            categories=category_lists[i]
        )
    else:
        X_sample[col] = X_sample[col].astype("category")

prediction = float(model.predict(X_sample)[0])
prediction = max(0.0, prediction)

print("\nSample input:")
print(X_sample.astype(str).to_dict(orient="records")[0])
print("Predicted yield_per_area:", prediction)
print("\nVerification completed.")


Model loaded successfully!
Model name: CropSense AI Tuned LightGBM Yield Model
Features: ['Crop', 'State_Name', 'District_Name', 'Season']
Target: yield_per_area
Training years: 1997 - 2015

Sample input:
{'Crop': 'Rice', 'State_Name': 'Andhra Pradesh', 'District_Name': np.str_('nan'), 'Season': 'Kharif'}
Predicted yield_per_area: 3.124987085582244

Verification completed.


In [11]:

import joblib
import pandas as pd

MODEL_PATH = (
    "/content/drive/MyDrive/yield prediction/"
    "cropsense_lightgbm_tuned_deployment/"
    "cropsense_yield_tuned_deployment.joblib"
)

artifact = joblib.load(MODEL_PATH)
model = artifact["model"]
features = artifact["features"]
category_lists = model.booster_.pandas_categorical

sample = {
    "Crop": "Rice",
    "State_Name": "Andhra Pradesh",
    "District_Name": "KRISHNA",
    "Season": "Kharif"
}

X = pd.DataFrame([sample])[features]

for i, col in enumerate(features):
    categories = list(category_lists[i])
    value = str(X[col].iloc[0])

    if value not in categories:
        raise ValueError(
            f"{col}='{value}' is not recognized. "
            f"Check the model's saved categories."
        )

    X[col] = pd.Categorical(X[col], categories=categories)

if X.isna().any().any():
    raise ValueError("Input contains missing or unrecognized categories.")

prediction = max(0.0, float(model.predict(X)[0]))

print("Model:", artifact["model_name"])
print("Validated input:", sample)
print(f"Predicted yield_per_area: {prediction:.4f}")
print("Verification passed!")


Model: CropSense AI Tuned LightGBM Yield Model
Validated input: {'Crop': 'Rice', 'State_Name': 'Andhra Pradesh', 'District_Name': 'KRISHNA', 'Season': 'Kharif'}
Predicted yield_per_area: 3.0835
Verification passed!


In [10]:

import os
import joblib
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/yield prediction/crop_final.csv"
MODEL_PATH = (
    "/content/drive/MyDrive/yield prediction/"
    "cropsense_lightgbm_tuned_deployment/"
    "cropsense_yield_tuned_deployment.joblib"
)

# Load dataset and deployment artifact
df = pd.read_csv(DATA_PATH)
artifact = joblib.load(MODEL_PATH)
model = artifact["model"]

print("=" * 55)
print("1. DATASET DISTRICT CHECK")
print("=" * 55)

# Check exact spelling in the dataset
districts = (
    df.loc[
        df["State_Name"].astype(str).str.strip().str.casefold()
        == "andhra pradesh",
        "District_Name"
    ]
    .dropna()
    .astype(str)
    .str.strip()
    .unique()
)

print("Andhra Pradesh district examples:", sorted(districts)[:30])
print("Does Krishna exist in this state?",
      any(d.casefold() == "krishna" for d in districts))

print("\n" + "=" * 55)
print("2. SAVED MODEL CATEGORY CHECK")
print("=" * 55)

categories = getattr(model.booster_, "pandas_categorical", None)

if categories is None:
    print("No pandas categorical metadata found in the booster.")
else:
    for i, feature in enumerate(artifact["features"]):
        feature_categories = categories[i] if i < len(categories) else []
        print(f"\n{feature}: {len(feature_categories)} categories")
        print("Examples:", list(feature_categories)[:15])

    district_index = artifact["features"].index("District_Name")
    district_categories = list(categories[district_index])

    print("\nIs Krishna in saved district categories?",
          "Krishna" in district_categories)
    print("District category examples containing 'krish':",
          [str(x) for x in district_categories
           if "krish" in str(x).casefold()][:20])

print("\n" + "=" * 55)
print("3. MISSING DISTRICT VALUES")
print("=" * 55)

print("Dataset missing districts:", df["District_Name"].isna().sum())
print("String 'nan' districts:",
      df["District_Name"].astype(str).str.strip().str.casefold()
      .eq("nan").sum())

print("\nDiagnostic complete. No files were modified.")


1. DATASET DISTRICT CHECK
Andhra Pradesh district examples: ['ANANTAPUR', 'CHITTOOR', 'EAST GODAVARI', 'GUNTUR', 'KADAPA', 'KRISHNA', 'KURNOOL', 'PRAKASAM', 'SPSR NELLORE', 'SRIKAKULAM', 'VISAKHAPATANAM', 'VIZIANAGARAM', 'WEST GODAVARI']
Does Krishna exist in this state? True

2. SAVED MODEL CATEGORY CHECK

Crop: 104 categories
Examples: ['Arcanut (Processed)', 'Arecanut', 'Arhar/Tur', 'Atcanut (Raw)', 'Bajra', 'Banana', 'Barley', 'Bean', 'Beans & Mutter(Vegetable)', 'Bhindi', 'Bitter Gourd', 'Black pepper', 'Blackgram', 'Bottle Gourd', 'Brinjal']

State_Name: 33 categories
Examples: ['Andaman and Nicobar Islands', 'Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chandigarh', 'Chhattisgarh', 'Dadra and Nagar Haveli', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jammu and Kashmir', 'Jharkhand', 'Karnataka']

District_Name: 646 categories
Examples: ['24 PARAGANAS NORTH', '24 PARAGANAS SOUTH', 'ADILABAD', 'AGAR MALWA', 'AGRA', 'AHMADABAD', 'AHMEDNAGAR', 'AIZAWL', 'AJMER', 'A